In [ ]:
#NOT RUNNED YET


from __future__ import annotations

import contextlib
import heapq
import io
import json
import math
import os
import shutil
import sqlite3
import time
from pathlib import Path
from typing import Dict, List, Sequence, Tuple, Optional

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import rioxarray
import xarray as xr
from onstove import OnStove
from rasterio.features import geometry_mask
from rasterio.transform import rowcol
from rasterio.warp import Resampling, reproject, transform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree

DATA_DIR = Path("dataset_big")

MODEL_PICKLE_PATH = DATA_DIR / ".." / ".." / "example" / "NGA" / "model_inputs.pkl"  # external source
SCENARIO_CSV_PATH = DATA_DIR / ".." / ".." / "example" / "NGA" / "soc_specs.csv"  # external source
INCOME_RASTER_PATH = DATA_DIR / "income_nigeria.tif"  # generated in 4.1
VEHICLE_POSSIBILITY_PATH = DATA_DIR / "vehicle_possibility.tif"  # generated in 4.1
VEHICLES_SHARE_PATH = DATA_DIR / "vehicles_allocation_share.tif"  # generated in 4.1
WALK_SHARE_PATH = DATA_DIR / "walk_allocation_share.tif"  # generated in 4.1
VEHICLES_N_PATH = DATA_DIR / "vehicles_allocation_n_effettivo.tif"  # generated in 4.1
POPULATION_RASTER_PATH = DATA_DIR / "Population.tif"  # external source
URBAN_RASTER_PATH = DATA_DIR / "Urban.tif"  # external source
COUNTRY_BOUNDARIES_PATH = DATA_DIR / "Country_boundaries.geojson"  # external source
FRICTION_WALK_PATH = DATA_DIR / "friction_walk.tif"  # external source
FRICTION_MOTO_PATH = DATA_DIR / "friction_moto.tif"  # external source
FULL_CHAIN_GPKG_PATH = DATA_DIR / "full_lpg_chain_nig_3857.gpkg"  # external source
HUFF_PIXEL_RASTER_PATH = DATA_DIR / "huff_preferred_distributor_per_pixel.tif"  # generated in 4.2
HUFF_LOOKUP_CSV_PATH = DATA_DIR / "huff_reseller_lookup.csv"  # generated in 4.2
HUFF_PROFILE_JSON_PATH = DATA_DIR / "huff_run_profile.json"  # generated in 4.2
LPG_USE_SHARE_PATH = DATA_DIR / "lpg_use_share.tif"  # generated in 4.3
SELL_POINT_GPKG_PATH = DATA_DIR / "sell_point_clients.gpkg"  # generated in 4.3
CHAIN_WITH_COST_PATH = DATA_DIR / "chain_with_cost.gpkg"  # generated in 4.4/4.5
FILLING_POINT_CLIENTS_GPKG_PATH = DATA_DIR / "filling_point_clients.gpkg"  # generated in 4.4
ASSIGNED_COST_GPKG_PATH = DATA_DIR / "filling_point_assigned.gpkg"  # external source
END_USER_PRICE_PATH = DATA_DIR / "end_user_price.tif"  # generated in 4.6

DATA_DIR.mkdir(parents=True, exist_ok=True)

essential_inputs = [
    MODEL_PICKLE_PATH,
    SCENARIO_CSV_PATH,
    POPULATION_RASTER_PATH,
    URBAN_RASTER_PATH,
    COUNTRY_BOUNDARIES_PATH,
    FRICTION_WALK_PATH,
    FRICTION_MOTO_PATH,
    FULL_CHAIN_GPKG_PATH,
    ASSIGNED_COST_GPKG_PATH,
]

missing_inputs = [p for p in essential_inputs if not p.exists()]
if missing_inputs:
    raise FileNotFoundError("Missing essential input file(s): " + ", ".join(str(p) for p in missing_inputs))

save_outputs = {
    "income_nigeria.tif": True,
    "vehicle_possibility.tif": True,
    "vehicles_allocation_share.tif": True,
    "walk_allocation_share.tif": True,
    "vehicles_allocation_n_effettivo.tif": True,
    "huff_preferred_distributor_per_pixel.tif": True,
    "huff_reseller_lookup.csv": True,
    "huff_run_profile.json": True,
    "lpg_use_share.tif": True,
    "sell_point_clients.gpkg": True,
    "chain_with_cost.gpkg": True,
    "filling_point_clients.gpkg": True,
    "end_user_price.tif": True,
}

memory_store = {
    "rasters": {},
    "rasters_da": {},
    "vectors": {},
    "tables": {},
    "json": {},
}

def _key_from_path(path: str | Path) -> str:
    return Path(path).name

def _cache_raster(path: str | Path, array: np.ndarray, profile: dict, nodata=None, descriptions=None) -> None:
    key = _key_from_path(path)
    memory_store["rasters"][key] = {
        "array": np.array(array, copy=True),
        "profile": profile.copy(),
        "nodata": nodata,
        "descriptions": list(descriptions) if descriptions is not None else None,
    }

def _cache_raster_da(path: str | Path, da) -> None:
    key = _key_from_path(path)
    memory_store["rasters_da"][key] = da

def _read_cached_raster(path: str | Path):
    return memory_store["rasters"].get(_key_from_path(path))

def _open_raster_da(path: str | Path):
    key = _key_from_path(path)
    cached = memory_store["rasters_da"].get(key)
    if cached is not None:
        return cached
    return rioxarray.open_rasterio(path)

def _cache_gpkg_layer(path: str | Path, layer: str, gdf: gpd.GeoDataFrame) -> None:
    key = _key_from_path(path)
    entry = memory_store["vectors"].setdefault(key, {})
    layers = entry.setdefault("layers", {})
    layers[layer] = gdf.copy()

def _read_gpkg_layer(path: str | Path, layer: str) -> gpd.GeoDataFrame:
    key = _key_from_path(path)
    entry = memory_store["vectors"].get(key)
    if entry is not None:
        layers = entry.get("layers", {})
        if layer in layers:
            return layers[layer].copy()
    return gpd.read_file(path, layer=layer)

def _write_gpkg_layer(path: str | Path, layer: str, gdf: gpd.GeoDataFrame) -> None:
    _cache_gpkg_layer(path, layer, gdf)
    if save_outputs.get(_key_from_path(path), True):
        gdf.to_file(path, layer=layer, driver="GPKG")

def _write_csv(path: str | Path, df: pd.DataFrame) -> None:
    key = _key_from_path(path)
    memory_store["tables"][key] = df.copy()
    if save_outputs.get(key, True):
        df.to_csv(path, index=False)

def _write_json(path: str | Path, payload: dict) -> None:
    key = _key_from_path(path)
    memory_store["json"][key] = payload
    if save_outputs.get(key, True):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)

def _read_multiband_raster(path: str | Path):
    cached = _read_cached_raster(path)
    if cached is not None:
        arr = cached["array"]
        desc = cached.get("descriptions")
        nodata = cached.get("nodata")
        return arr, desc, nodata
    with rasterio.open(path) as src:
        arr = src.read()
        desc = list(src.descriptions)
        nodata = src.nodata
    return arr, desc, nodata

"""
Create an income raster for Nigeria using OnStove income_estimation.
This script uses only method 1 (AWE estimation) in this notebook.
Method 2 (Income CSV interpolation) is intentionally commented out.
"""

# User parameters (paths relative to this notebook folder)
model_pickle_path = os.path.normpath(str(MODEL_PICKLE_PATH))
scenario_csv_path = os.path.normpath(str(SCENARIO_CSV_PATH))
output_directory = str(DATA_DIR)

# Method 2 not used in this notebook (kept as reference)
# use_income_csv = True
# income_csv_path = "dataset_big/nigeria_income_percentiles.csv"

# Required for AWE mode
gini_value = 0.339      # Example: 0.35
gdp_pc_value = 2139    # Example: 2200.0
pareto_weight = 0.32

# Output raster column and name (AWE writes absolute_wealth)
output_variable = "absolute_wealth"
output_raster_name = "income_nigeria"

# Basic input checks
if not os.path.exists(model_pickle_path):
    raise FileNotFoundError(f"Model pickle not found: {model_pickle_path}")
if not os.path.exists(scenario_csv_path):
    raise FileNotFoundError(f"Scenario csv not found: {scenario_csv_path}")
os.makedirs(output_directory, exist_ok=True)

# 1) Load prepared model and scenario
model = OnStove.read_model(model_pickle_path)
model.read_scenario_data(scenario_csv_path, delimiter=",")
model.output_directory = output_directory

# 2) Configure inputs for selected income mode
# AWE only branch
if gini_value is None or gdp_pc_value is None:
    raise ValueError("Set gini_value and gdp_pc_value for AWE mode")
model.specs["gini"] = float(gini_value)
model.specs["gdp_pc"] = float(gdp_pc_value)
model.income_estimation(awe=True, income_data=None, pareto_weight=pareto_weight)

# 3) Quick checks
if "relative_wealth" not in model.gdf.columns:
    raise KeyError("Model gdf is missing 'relative_wealth', required by income_estimation")
if output_variable not in model.gdf.columns:
    raise KeyError(f"Expected output column '{output_variable}' not found in model.gdf")

# 4) Export raster (suppresses OnStove's internal print statements)
with contextlib.redirect_stdout(io.StringIO()):
    model.to_raster(variable=output_variable)

# Move file to final destination and clean up
src_name = f"{output_variable}_mean.tif"
src_path = os.path.join(output_directory, "Rasters", src_name)
dst_path = os.path.join(output_directory, f"{output_raster_name}.tif")
if os.path.exists(src_path):
    if os.path.exists(dst_path):
        os.remove(dst_path)
    os.replace(src_path, dst_path)
    print(f"Raster moved to: {dst_path}")

if os.path.exists(dst_path):
    income_da = rioxarray.open_rasterio(dst_path)
    _cache_raster_da(dst_path, income_da)
    _cache_raster(dst_path, income_da.values, income_da.rio.profile(), income_da.rio.nodata, income_da.rio.descriptions)
    if not save_outputs.get(_key_from_path(dst_path), True):
        os.remove(dst_path)

# Delete the temporary Rasters directory
rasters_dir = os.path.join(output_directory, "Rasters")
if os.path.exists(rasters_dir):
    shutil.rmtree(rasters_dir)
    print(f"Temporary directory deleted: {rasters_dir}")

print("Income mode: AWE")
print(f"Final raster: {dst_path if os.path.exists(dst_path) else 'NOT FOUND'}")

"""
Reproject income raster to EPSG:3857 (Web Mercator), normalize, and create vehicle_possibility.
Vehicle possibility = normalized income with exponent applied.
"""

def main():
    
    # User parameters
    input_path = str(INCOME_RASTER_PATH)
    output_vehicle_path = str(VEHICLE_POSSIBILITY_PATH)
    target_crs = "EPSG:3857"
    target_resolution = 1000  # meters in EPSG:3857 (1 km)
    output_nodata = -9999.0

    # Shape parameter for vehicle possibility (applied on normalized income).
    # 1.0 keeps linear relation, >1 emphasizes high-income pixels, <1 flattens differences.
    income_exponent = 1.0
    
    # Read the income raster in its original CRS (already on Nigeria from OnStove)
    print(f"Reading income raster from {input_path}...")
    income_da = _open_raster_da(input_path)
    original_crs = income_da.rio.crs
    original_nodata = income_da.rio.nodata
    print(f"Original CRS: {original_crs}")
    print(f"Original resolution: {income_da.rio.resolution()}")
    print(f"Original nodata value: {original_nodata}")
    print(f"Array contains NaN values: {np.isnan(income_da.values).any()}")
    
    # Reproject to target CRS
    print(f"Reprojecting to {target_crs} with resolution {target_resolution} m...")
    income_3857 = income_da.rio.reproject(
        target_crs,
        resolution=target_resolution,
        resampling=Resampling.nearest
    )
    income_3857 = income_3857.fillna(output_nodata)
    income_3857.rio.write_nodata(output_nodata, inplace=True)
    
    # Normalize income to [0, 1] and apply exponent to create vehicle_possibility.
    print("Normalizing income to [0, 1] and applying exponent...")
    data = income_3857.values.astype(np.float32, copy=True)
    valid = (np.isfinite(data)) & (data != output_nodata)
    if original_nodata is not None:
        valid = valid & (data != original_nodata)
    
    if valid.any():
        vmin = data[valid].min()
        vmax = data[valid].max()
        
        if vmax > vmin:
            data[valid] = (data[valid] - vmin) / (vmax - vmin)
        else:
            data[valid] = 0.0
        
        # Apply exponent on already normalized data
        data[valid] = np.clip(data[valid], 0.0, 1.0)
        data[valid] = np.power(data[valid], income_exponent, dtype=np.float32)
        
        data[~valid] = output_nodata
        
        print(f"Data normalized and exponent applied: min={float(data[valid].min()):.4f}, max={float(data[valid].max()):.4f}")
        print(f"Nodata value={output_nodata}, nodata pixels={int((~valid).sum())}")
    else:
        raise ValueError("No valid pixels found for normalization.")
    
    # Build vehicle_possibility raster (single output)
    vehicle_possibility = xr.DataArray(
        data,
        coords=income_3857.coords,
        dims=income_3857.dims,
        attrs=income_3857.attrs,
        name="vehicle_possibility",
    )
    vehicle_possibility.rio.write_crs(target_crs, inplace=True)
    vehicle_possibility.rio.write_nodata(output_nodata, inplace=True)
    vehicle_possibility.attrs["source"] = "income_only"
    vehicle_possibility.attrs["income_exponent"] = float(income_exponent)

    print(f"Writing vehicle possibility raster to {output_vehicle_path}...")
    if save_outputs.get(_key_from_path(output_vehicle_path), True):
        vehicle_possibility.rio.to_raster(output_vehicle_path, compress="DEFLATE", nodata=output_nodata)
    _cache_raster_da(output_vehicle_path, vehicle_possibility)
    _cache_raster(output_vehicle_path, vehicle_possibility.values, vehicle_possibility.rio.profile(), output_nodata, vehicle_possibility.rio.descriptions)

    out = vehicle_possibility.values
    out_valid = np.isfinite(out) & (out != output_nodata)
    print(f"Done. Vehicle possibility raster saved as {output_vehicle_path} in {target_crs}")
    print(f"Approach: income normalized with exponent={income_exponent}")
    print(f"Output range: min={float(out[out_valid].min()):.4f}, max={float(out[out_valid].max()):.4f}")
    
if __name__ == "__main__":
    main()


"""
Vehicle-access allocation rationale

This cell distributes a fixed national target of car access across pixels
in Nigeria, combining:
1) vehicle_possibility (derived from income only)
2) population per pixel.

Method idea:
- the pixel with the highest vehicle_possibility is constrained to 100% access,
- all other pixels receive a lower access share, scaled smoothly,
- a calibration parameter (alpha) is found so that the sum of allocated people
  with car access is exactly equal to the national target,
- very small modeled access shares are floored to zero using a minimum share
  threshold during calibration and final allocation.

National target definition:
- reference vehicles = 11,600,000 [OICA, 2020]
- average users sharing one vehicle
- target people with access = shared_users_per_vehicle * reference vehicles

Final outputs are three separate rasters (names kept for compatibility):
- vehicles_allocation_share.tif: share (%) of population with car access in each pixel
- walk_allocation_share.tif: share (%) of population without car access in each pixel
- vehicles_allocation_n_effettivo.tif: number of people with car access in each pixel


VALUE OF AVERAGE USERS PER VEHICLE AND CONSIDERATIONS

For the Nigeria case study, I recommend using 5.06 persons per household as the national reference value, 
with a clear urban-rural split of 4.50 persons in urban areas and 5.42 persons in rural areas, based on the 
Nigeria Living Standards Survey 2020 by the National Bureau of Statistics. Since direct data on car-sharing 
per vehicle is rarely available, this household size is a reasonable proxy for the number of people 
potentially sharing one private car, especially when the model needs a consistent population-based assumption.
https://www.nigerianstat.gov.ng/elibrary/read/1123

For extrapolation across Nigeria, these values can be applied as the baseline assumption for 
household-based vehicle access, while noting that they represent an approximation rather than 
a measured car-occupancy statistic. If a broader regional benchmark is needed for Sub-Saharan 
Africa, a useful indicative range is about 5–6 persons in urban areas and 6–7 persons in rural 
areas, which is consistent with demographic patterns reported in the literature on African 
household structure. based on demographic analyses drawing from the United Nations Population 
Division database on household size and composition and DHS‑type surveys covering the region.
https://www.un.org/development/desa/pd/household-size-and-composition
"""

# Inputs
vehicle_possibility_path = str(VEHICLE_POSSIBILITY_PATH)
population_path = str(POPULATION_RASTER_PATH)
output_share_car_access_path = str(VEHICLES_SHARE_PATH)
output_share_walk_access_path = str(WALK_SHARE_PATH)
output_cars_path = str(VEHICLES_N_PATH)

urban_raster_path = str(URBAN_RASTER_PATH)        # values: 0‑100 (urbanisation degree)
URBAN_THRESHOLD = 20                                # cells >= 20 considered urban

reference_total_vehicles = 11_600_000
f_urban = 4.50      # users_urban
f_rural = 5.42      # users_rural
target_vehicles = reference_total_vehicles   #target


output_nodata = -9999.0
min_vehicle_share = 1e-5

def total_vehicles_for_alpha(alpha, s_norm, pop, F_arr, min_share):
    """Return total allocated vehicles for a given alpha after share flooring."""
    rates = np.power(s_norm, alpha, dtype=np.float64)
    rates = np.where(rates < min_share, 0.0, rates)
    vehicles = pop * rates / F_arr
    return float(np.sum(vehicles, dtype=np.float64))

# 1) Read rasters and align population to the vehicle-possibility grid
score_da = _open_raster_da(vehicle_possibility_path)
pop_da = _open_raster_da(population_path)
pop_da = pop_da.rio.reproject_match(score_da, resampling=Resampling.nearest)

# Read urban raster and align to the same grid
urban_da = _open_raster_da(urban_raster_path)
urban_da = urban_da.rio.reproject_match(score_da, resampling=Resampling.nearest)


score_nodata = score_da.rio.nodata
pop_nodata = pop_da.rio.nodata


score = score_da.values.astype(np.float64, copy=False)
pop = pop_da.values.astype(np.float64, copy=False)

urban = urban_da.values.astype(np.float64, copy=False)
urban_nodata = urban_da.rio.nodata



# 2) Build valid mask: finite, positive population, and non-nodata values
valid = (np.isfinite(score) & np.isfinite(pop) & np.isfinite(urban) & (pop > 0))
if score_nodata is not None:
    valid &= score != score_nodata
if pop_nodata is not None:
    valid &= pop != pop_nodata
if urban_nodata is not None:
    valid &= urban != urban_nodata

if not valid.any():
    raise ValueError("No valid pixels found after masking.")

# 3) Convert vehicle possibility to [0, 1] among valid pixels
s = score[valid]
s_min = float(np.min(s))
s_max = float(np.max(s))

if s_max <= s_min:
    raise ValueError("Vehicle possibility has no variation on valid pixels.")

s_norm = (s - s_min) / (s_max - s_min)
s_norm = np.clip(s_norm, 0.0, 1.0)

# Force one max pixel to exactly 1.0 => 100% access in that pixel
max_idx = int(np.argmax(s))
s_norm[max_idx] = 1.0

pop_valid = pop[valid]


urban_valid = urban[valid]
urban_flag = urban_valid >= URBAN_THRESHOLD
F_valid = np.where(urban_flag, f_urban, f_rural).astype(np.float64)

# 4) Calibrate alpha with bisection so “sum(vehicles) = target_vehicles”.

max_vehicles_at_alpha0 = total_vehicles_for_alpha(0.0, s_norm, pop_valid, F_valid, min_vehicle_share)
if target_vehicles > max_vehicles_at_alpha0:
    raise ValueError(
        f"Requested target_vehicles={target_vehicles:,} exceeds maximum theoretical capacity={max_vehicles_at_alpha0:,.0f}."
    )

left, right = 0.0, 40.0
for _ in range(80):
    mid = 0.5 * (left + right)
    veh_mid = total_vehicles_for_alpha(mid, s_norm, pop_valid, F_valid, min_vehicle_share)
    if veh_mid > target_vehicles:
        left = mid
    else:
        right = mid
alpha = 0.5 * (left + right)

# 5) Compute final per-pixel outputs
raw_rates_valid = np.power(s_norm, alpha, dtype=np.float64)
rates_valid = np.where(raw_rates_valid < min_vehicle_share, 0.0, raw_rates_valid)
zeroed_small_positive = int(np.sum((raw_rates_valid > 0.0) & (raw_rates_valid < min_vehicle_share)))

vehicles_valid = pop_valid * rates_valid / F_valid
total_allocated_vehicles = float(np.sum(vehicles_valid))
access_people_valid = pop_valid * rates_valid

share_car_access_percent = np.full(score.shape, output_nodata, dtype=np.float32)
share_walk_access_percent = np.full(score.shape, output_nodata, dtype=np.float32)
n_effettivo = np.full(score.shape, output_nodata, dtype=np.float32)

share_car_access_percent[valid] = (rates_valid * 100.0).astype(np.float32)
share_walk_access_percent[valid] = (100.0 - share_car_access_percent[valid]).astype(np.float32)
n_effettivo[valid] = access_people_valid.astype(np.float32)

# 6) Save to separate GeoTIFF files (same grid, same CRS)
share_car_access_da = xr.DataArray(
    share_car_access_percent,
    coords=score_da.coords,
    dims=score_da.dims,
    name="share_car_access",
)
share_car_access_da.rio.write_crs(score_da.rio.crs, inplace=True)
share_car_access_da.rio.write_nodata(output_nodata, inplace=True)
share_car_access_da.attrs["long_name"] = "vehicles_allocation_share_percent"
share_car_access_da.attrs["reference_total_vehicles"] = int(reference_total_vehicles)
share_car_access_da.attrs["alpha"] = float(alpha)
share_car_access_da.attrs["min_vehicle_share"] = float(min_vehicle_share)
share_car_access_da.attrs["f_urban"] = float(f_urban)
share_car_access_da.attrs["f_rural"] = float(f_rural)
share_car_access_da.attrs["target_vehicles"] = int(target_vehicles)
share_car_access_da.attrs["allocated_vehicles"] = float(total_allocated_vehicles)   
if save_outputs.get(_key_from_path(output_share_car_access_path), True):
    share_car_access_da.rio.to_raster(output_share_car_access_path, compress="DEFLATE", nodata=output_nodata)
_cache_raster_da(output_share_car_access_path, share_car_access_da)
_cache_raster(output_share_car_access_path, share_car_access_da.values, share_car_access_da.rio.profile(), output_nodata, share_car_access_da.rio.descriptions)



share_walk_access_da = xr.DataArray(
    share_walk_access_percent,
    coords=score_da.coords,
    dims=score_da.dims,
    name="share_walk_access",
)
share_walk_access_da.rio.write_crs(score_da.rio.crs, inplace=True)
share_walk_access_da.rio.write_nodata(output_nodata, inplace=True)
share_walk_access_da.attrs["long_name"] = "walk_allocation_share_percent"
share_walk_access_da.attrs["derived_from"] = "100 - vehicles_allocation_share"
share_walk_access_da.attrs["reference_total_vehicles"] = int(reference_total_vehicles)
share_walk_access_da.attrs["f_urban"] = float(f_urban)
share_walk_access_da.attrs["f_rural"] = float(f_rural)
share_walk_access_da.attrs["target_vehicles"] = int(target_vehicles)
share_walk_access_da.attrs["allocated_vehicles"] = float(total_allocated_vehicles)
share_walk_access_da.attrs["alpha"] = float(alpha)
share_walk_access_da.attrs["min_vehicle_share"] = float(min_vehicle_share)
if save_outputs.get(_key_from_path(output_share_walk_access_path), True):
    share_walk_access_da.rio.to_raster(output_share_walk_access_path, compress="DEFLATE", nodata=output_nodata)
_cache_raster_da(output_share_walk_access_path, share_walk_access_da)
_cache_raster(output_share_walk_access_path, share_walk_access_da.values, share_walk_access_da.rio.profile(), output_nodata, share_walk_access_da.rio.descriptions)

cars_da = xr.DataArray(
    n_effettivo,
    coords=score_da.coords,
    dims=score_da.dims,
    name="n_effettivo",
)
cars_da.rio.write_crs(score_da.rio.crs, inplace=True)
cars_da.rio.write_nodata(output_nodata, inplace=True)
cars_da.attrs["long_name"] = "vehicle_access_allocated_people"
cars_da.attrs["reference_total_vehicles"] = int(reference_total_vehicles)
cars_da.attrs["f_urban"] = float(f_urban)
cars_da.attrs["f_rural"] = float(f_rural)
cars_da.attrs["target_vehicles"] = int(target_vehicles)
cars_da.attrs["allocated_vehicles"] = float(total_allocated_vehicles)
cars_da.attrs["alpha"] = float(alpha)
cars_da.attrs["min_vehicle_share"] = float(min_vehicle_share)
if save_outputs.get(_key_from_path(output_cars_path), True):
    cars_da.rio.to_raster(output_cars_path, compress="DEFLATE", nodata=output_nodata)
_cache_raster_da(output_cars_path, cars_da)
_cache_raster(output_cars_path, cars_da.values, cars_da.rio.profile(), output_nodata, cars_da.rio.descriptions)

# 7) Minimal summary of allocation result
total_allocated_people = float(np.sum(n_effettivo[n_effettivo != output_nodata], dtype=np.float64))



# Gap veicoli (dovrebbe essere molto piccolo per via della calibrazione)
vehicle_gap = total_allocated_vehicles - target_vehicles
vehicle_gap_pct = (vehicle_gap / target_vehicles) * 100.0

effective_f = total_allocated_people / total_allocated_vehicles if total_allocated_vehicles > 0 else 0.0
max_share_car_access = float(np.max(share_car_access_percent[share_car_access_percent != output_nodata]))
min_share_walk_access = float(np.min(share_walk_access_percent[share_walk_access_percent != output_nodata]))


print(f"Estimated alpha: {alpha:.6f}")
print(f"Reference total vehicles: {reference_total_vehicles:,.0f}")
print(f"F urban: {f_urban}, F rural: {f_rural}")
print(f"Target vehicles: {target_vehicles:,.0f}")
print(f"Allocated vehicles: {total_allocated_vehicles:,.0f}")
print(f"Vehicle gap (allocated - target): {vehicle_gap:+,.3f} ({vehicle_gap_pct:+.6f}%)")
print(f"Total people with car access: {total_allocated_people:,.0f}")
print(f"Effective national average users/vehicle: {effective_f:.2f}")
print(f"Min vehicle share threshold applied: {min_vehicle_share:.1e}")
print(f"Pixels floored to zero by threshold: {zeroed_small_positive:,}")
print(f"Max car access share (%): {max_share_car_access:.2f}")
print(f"Min walk access share (%): {min_share_walk_access:.2f}")
print(f"Car share raster written: {output_share_car_access_path}")
print(f"Walk share raster written: {output_share_walk_access_path}")
print(f"Access-people raster written: {output_cars_path}")

# Quick stats on car-ownership share distribution per pixel

share_path = str(VEHICLES_SHARE_PATH)
share_da = _open_raster_da(share_path)

nodata = share_da.rio.nodata
data = share_da.values.astype(np.float64, copy=False)

valid = np.isfinite(data)
if nodata is not None:
    valid &= data != nodata

vals = data[valid]

if vals.size == 0:
    raise ValueError("No valid values found in share raster.")

print("=== SHARE DISTRIBUTION (% auto per pixel) ===")
print(f"Valid cells: {vals.size:,}")
print(f"Min: {vals.min():.2f}%")
print(f"P10: {np.percentile(vals, 10):.2f}%")
print(f"P25: {np.percentile(vals, 25):.2f}%")
print(f"Median: {np.percentile(vals, 50):.2f}%")
print(f"Mean: {vals.mean():.2f}%")
print(f"P75: {np.percentile(vals, 75):.2f}%")
print(f"P90: {np.percentile(vals, 90):.2f}%")
print(f"P95: {np.percentile(vals, 95):.2f}%")
print(f"Max: {vals.max():.2f}%")

# Bins in percentage points
bins = np.array([0, 1, 5, 10, 20, 40, 60, 80, 95, 100.0001], dtype=np.float64)
labels = [
    "0-1%", "1-5%", "5-10%", "10-20%", "20-40%",
    "40-60%", "60-80%", "80-95%", "95-100%"
]

counts, _ = np.histogram(vals, bins=bins)
shares = counts / vals.size * 100.0

print("\n=== CELLS BY SHARE CLASS ===")
for lbl, c, s in zip(labels, counts, shares):
    print(f"{lbl:>8}: {c:>8,} cells ({s:>6.2f}%)")

# Extra: how many cells are near full adoption
above_90 = np.sum(vals >= 90)
above_50 = np.sum(vals >= 50)
print("\n=== QUICK THRESHOLDS ===")
print(f"Cells >= 50%: {above_50:,} ({above_50/vals.size*100:.2f}%)")
print(f"Cells >= 90%: {above_90:,} ({above_90/vals.size*100:.2f}%)")

"""
Huff‑based LPG Distributor Assignment (Multi‑source Dijkstra)

This script implements a fast, production‑ready Huff model assignment of LPG resellers
to inhabited 1 km pixels in Nigeria. It uses a two‑phase strategy to combine speed and accuracy:

1. Phase 1 – Approximate assignment (multi‑source Dijkstra with cell finalisation)
   A single multi‑source Dijkstra per travel mode rapidly determines the most attractive
   reseller for each pixel. Because cells are finalised once assigned, this phase introduces
   a small detour approximation but runs in ~30 seconds.

2. Phase 2 – Exact metric computation (single‑source Dijkstra per reseller)
   For each reseller, a standard Dijkstra (without finalisation) is run starting from its
   seed cell. The algorithm recomputes exact travel times and distances **only** for the
   pixels that Phase 1 assigned to that reseller. Early termination stops the search once
   all assigned pixels have been reached or a generous time limit is exceeded.

Key features:
- 8‑neighbour connectivity with symmetric edge costs (average friction × distance).
- Independent walk and motorised assignments.
- Configurable Huff parameters (β=2.0, ε=1e-6).

Outputs:
- Multi‑band GeoTIFF with car/walk shares, best reseller IDs (from Phase 1), and exact
  travel times/distances (from Phase 2).
- Lookup CSV for resellers placed on the grid.
- Optional JSON run profile.

NOTE ON THE APPROXIMATION INTRODUCED BY THE MULTI‑SOURCE DIJKSTRA
------------------------------------------------------------------
Unlike a per‑pixel Dijkstra that computes independent shortest paths to all resellers,
the multi‑source algorithm finalises a cell once it is assigned to a particular reseller.
This means other resellers cannot later “pass through” that cell, which in theory can
introduce a small detour in highly competitive areas.

We quantified this effect on a random sample of 60,000 inhabited pixels:
- Median difference in assigned travel time: 1.98 min
- 95th percentile difference: 23.30 min
- Maximum observed difference: 272.43 min
- Pixels with deviation > 1 min: 57.2%
- Pixels with deviation > 5 min: 36.3%
- Pixels with deviation > 10 min: 19.4%

The two‑phase design eliminates this approximation from the final travel metrics.
Phase 1 still provides the fast assignment, while Phase 2 supplies exact times and distances.
The overall runtime increases moderately (typically under 2 minutes) while retaining the
massive speedup over a full per‑pixel Huff model.


**Sensitivity of Huff Assignment to Attractiveness Transformation**

A sensitivity analysis was conducted to assess how the Huff‑based distributor assignment responds 
to different transformations of the original attractiveness scores \(A \in [0,1]\). 
Six transformations were tested: the baseline \(A\), multiplicative scalings \(A \times 5\) and \(A \times 100\), 
and additive shifts \((A+1)\), \((A+2)\), together with their scaled variants \(5 \times (A+1)\) and \(5 \times (A+2)\). 
For each case, the multi‑source Dijkstra algorithm was re‑run for both walk and motorised modes, and the resulting 
assignments were compared.

**Key findings:**  
- **Multiplicative scaling leaves the results unchanged.** All pure scaling transformations produced identical 
assigned reseller IDs and identical median travel times/distances. This confirms that the Huff weight \(\sqrt{A}\) 
is insensitive to a uniform rescaling of attractiveness; only relative differences matter.  
- **Adding a constant alters the spatial allocation.** Introducing an additive term (e.g., \(A+1\) or \(A+2\)) raises 
the minimum attractiveness and compresses the relative variation among resellers. This leads to a systematic, though 
modest, reduction in median travel times (e.g., walk time from 368 to ~353 minutes) and distances, and changes the 
assigned reseller for about 14–17% of inhabited pixels compared to the baseline.  
- **The effect is governed by the additive constant, not by an outer scaling factor.** The results for 
\((A+1)\) and \(5 \times (A+1)\) are identical, as are those for \((A+2)\) and \(5 \times (A+2)\). The magnitude 
of the shift (i.e., the value of the constant) controls the degree of departure from the baseline allocation.

**Implications for model specification:**  
Given the invariance to multiplicative scaling, the original 0–1 range of attractiveness is retained for simplicity 
and interpretability. If, for policy or behavioural reasons, one wishes to **increase the relative importance of 
distance over attractiveness**, adding a constant to \(A\) (e.g., using \(\sqrt{A+c}\) instead of \(\sqrt{A}\)) 
provides a practical and computationally efficient solution. This approach preserves the additive cost structure 
of the Dijkstra algorithm and avoids the complications that would arise from raising travel time to a 
power \(\beta \neq 1\), which would require non‑additive edge weights and potentially more complex path‑finding 
routines. In essence, adding a constant limits the relative differences in attractiveness and thus gives greater 
weight to proximity, achieving an effect analogous to increasing the distance exponent in a classical Huff model, 
but without altering the core algorithmic framework.


Attractiveness expansion: how does it works?

The algorithm runs a multi‑source Dijkstra search where each reseller expands simultaneously. It uses a priority 
queue that always contains the border pixels that have been reached but not yet assigned. Every queue entry stores 
a key (a cost) equal to travel_time / sqrt(attractiveness). At each step, the algorithm extracts the pixel with 
the smallest key (low cost), finalizes it (assigns it to that reseller), and then expands from it to its 8 
neighbors. Any newly reached neighbor is inserted into the queue with its own key. If a neighbor was already 
in the queue with a larger key, the smaller key takes precedence. 
This process continues until all inhabited pixels are assigned. Crucially, the physical expansion speed 
(pixels per real minute) depends only on the friction surface and distance, not on attractiveness. 
Attractiveness does not make a reseller’s wave move faster in real time; it only reorders the queue: a 
higher sqrt(attractiveness) lowers the key, giving that reseller priority over others at equal travel time. 
Therefore the order of assignment is driven by minimizing travel_time / sqrt(attractiveness), 
not by any “expansion velocity” proportional to that ratio. In competition, a more attractive reseller will 
“jump the queue” and claim pixels earlier, but its actual arrival time at each pixel is exactly the same 
as it would be without attractiveness.





"""

# -----------------------------------------------------------------------------
# USER CONFIGURATION – adjust these paths and parameters as needed
# -----------------------------------------------------------------------------
DATASET_DIR = DATA_DIR

# Input files (relative to DATASET_DIR)
POPULATION_TIF = POPULATION_RASTER_PATH.name
CAR_SHARE_TIF = VEHICLES_SHARE_PATH.name
FRICTION_WALK_TIF = FRICTION_WALK_PATH.name
FRICTION_MOTO_TIF = FRICTION_MOTO_PATH.name
RESELLER_GPKG = FULL_CHAIN_GPKG_PATH.name
RESELLER_LAYER = "resell_and_filling"

# Column names in the reseller layer
RESELLER_ID_COL = "id_res&fil"
ATTRACTIVENESS_COL = "attractiveness"

# Output files (saved inside DATASET_DIR)
OUTPUT_RASTER = HUFF_PIXEL_RASTER_PATH.name
OUTPUT_LOOKUP = HUFF_LOOKUP_CSV_PATH.name
OUTPUT_PROFILE = HUFF_PROFILE_JSON_PATH.name          # set to None to disable

# Huff model parameters
BETA = 2.0
EPS = 1e-6
ATTR_MIN = 1e-6

# Raster nodata value
NODATA = -9999.0

# Progress reporting frequency (pixels)
LOG_EVERY = 100000

# Pixel size in metres (grid is EPSG:3857, 1 km cells)
PIXEL_SIZE_M = 1000.0

# Factor for early termination in exact recomputation (multiply max observed time from Phase 1)
TIME_LIMIT_FACTOR = 1.5

if not save_outputs.get(OUTPUT_PROFILE, True):
    OUTPUT_PROFILE = None

# -----------------------------------------------------------------------------
# 8‑neighbour moves with Euclidean distance factors (in pixel units)
# -----------------------------------------------------------------------------
NEIGHBORS = (
    (-1,  0, 1.0),
    ( 1,  0, 1.0),
    ( 0, -1, 1.0),
    ( 0,  1, 1.0),
    (-1, -1, math.sqrt(2.0)),
    (-1,  1, math.sqrt(2.0)),
    ( 1, -1, math.sqrt(2.0)),
    ( 1,  1, math.sqrt(2.0)),
)

# -----------------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------------
def _safe_ratio(num: float, den: float) -> float:
    """Return num/den, or 0.0 if den == 0."""
    return float(num / den) if den else 0.0


def _print_progress(
    mode: str,
    assigned: int,
    total: int,
    elapsed: float,
) -> None:
    """Print progress with speed and ETA."""
    if total <= 0:
        return
    pct = 100.0 * assigned / total
    rate = _safe_ratio(assigned, elapsed)
    remaining = max(total - assigned, 0)
    eta_sec = _safe_ratio(remaining, max(rate, 1e-12))
    print(
        f"[{mode}] {assigned:,}/{total:,} ({pct:.2f}%) | "
        f"{rate:.1f} px/s | ETA {eta_sec/60:.1f} min"
    )


def read_reference_population(path: Path) -> Tuple[np.ndarray, Dict]:
    """Read population raster and return array + profile."""
    cached = _read_cached_raster(path)
    if cached is not None:
        arr = cached["array"]
        if arr.ndim == 3:
            arr = arr[0]
        pop = arr.astype(np.float32)
        profile = cached["profile"].copy()
        return pop, profile
    with rasterio.open(path) as src:
        pop = src.read(1).astype(np.float32)
        profile = src.profile.copy()
    return pop, profile


def reproject_to_reference(
    src_path: Path,
    ref_profile: Dict,
    resampling: Resampling,
) -> np.ndarray:
    """Reproject (or resample) a raster to match the reference grid."""
    cached = _read_cached_raster(src_path)
    dst = np.full(
        (ref_profile["height"], ref_profile["width"]),
        np.nan,
        dtype=np.float32,
    )
    if cached is not None:
        src_arr = cached["array"]
        if src_arr.ndim == 3:
            src_arr = src_arr[0]
        reproject(
            source=src_arr,
            destination=dst,
            src_transform=cached["profile"]["transform"],
            src_crs=cached["profile"]["crs"],
            src_nodata=cached.get("nodata"),
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
        return dst
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def normalize_car_share(arr: np.ndarray) -> np.ndarray:
    """Convert car share to fraction (0–1). Assumes input may be 0–100."""
    out = arr.astype(np.float32, copy=True)
    finite = np.isfinite(out)
    if finite.any():
        vmax = float(np.nanmax(out[finite]))
        if vmax > 1.0 + 1e-4 and vmax <= 100.0 + 1e-4:
            out[finite] = out[finite] / 100.0
    out[~finite] = 0.0
    np.clip(out, 0.0, 1.0, out=out)
    return out


def sanitize_friction(arr: np.ndarray) -> np.ndarray:
    """Set non‑positive or non‑finite friction to NaN."""
    out = arr.astype(np.float32, copy=True)
    invalid = ~np.isfinite(out) | (out <= 0.0)
    out[invalid] = np.nan
    return out


def connected_components_8(mask: np.ndarray) -> np.ndarray:
    """
    Label 8‑connected components of True cells.
    Returns an integer array (0 = background).
    Falls back to pure Python if scipy is not available.
    """
    # Try to use scipy first (much faster)
    try:
        from scipy import ndimage
        structure = np.ones((3, 3), dtype=np.uint8)
        labels, _ = ndimage.label(mask, structure=structure)
        return labels.astype(np.int32)
    except ImportError:
        # Fallback pure Python implementation
        h, w = mask.shape
        labels = np.zeros((h, w), dtype=np.int32)
        current = 0
        stack = []
        for r in range(h):
            for c in range(w):
                if not mask[r, c] or labels[r, c] != 0:
                    continue
                current += 1
                labels[r, c] = current
                stack.append((r, c))
                while stack:
                    rr, cc = stack.pop()
                    for dr, dc, _ in NEIGHBORS:
                        nr, nc = rr + dr, cc + dc
                        if nr < 0 or nr >= h or nc < 0 or nc >= w:
                            continue
                        if not mask[nr, nc] or labels[nr, nc] != 0:
                            continue
                        labels[nr, nc] = current
                        stack.append((nr, nc))
        return labels


def load_reseller_seeds(
    gpkg_path: Path,
    layer: str,
    id_col: str,
    attr_col: str,
    ref_profile: Dict,
) -> List[Dict]:
    """
    Load reseller points, reproject to reference CRS, and keep only those
    falling inside the raster extent. If multiple points map to the same pixel,
    keep the most attractive one.
    Returns a list of dicts with keys: reseller_id, attractiveness, row, col.
    """
    gdf = gpd.read_file(gpkg_path, layer=layer)
    if gdf.empty:
        raise ValueError("Reseller layer is empty.")

    ref_crs = ref_profile["crs"]
    if gdf.crs is None:
        raise ValueError("Reseller GeoPackage has no CRS.")
    if gdf.crs != ref_crs:
        gdf = gdf.to_crs(ref_crs)

    # Clean IDs and attractiveness
    ids = pd.to_numeric(gdf[id_col], errors="coerce")
    attrs = pd.to_numeric(gdf[attr_col], errors="coerce")
    attrs = attrs.fillna(ATTR_MIN).clip(lower=ATTR_MIN)

    xs = gdf.geometry.x.to_numpy()
    ys = gdf.geometry.y.to_numpy()
    rows, cols = rowcol(ref_profile["transform"], xs, ys)

    h, w = ref_profile["height"], ref_profile["width"]
    seeds_dict = {}
    for rid, attr, r, c in zip(ids, attrs, rows, cols):
        if not np.isfinite(rid):
            continue
        rr = int(r)
        cc = int(c)
        if rr < 0 or rr >= h or cc < 0 or cc >= w:
            continue

        key = (rr, cc)
        candidate = {
            "reseller_id": float(rid),
            "attractiveness": float(attr),
            "row": rr,
            "col": cc,
        }
        prev = seeds_dict.get(key)
        if prev is None or candidate["attractiveness"] > prev["attractiveness"]:
            seeds_dict[key] = candidate

    seeds = list(seeds_dict.values())
    if not seeds:
        raise ValueError("No reseller seed mapped onto the reference grid.")
    return seeds


def run_multisource_dijkstra(
    mode_name: str,
    friction: np.ndarray,
    traversable: np.ndarray,
    target_mask: np.ndarray,
    labels: np.ndarray,
    seeds: List[Dict],
    pixel_size_m: float,
    log_every: int = LOG_EVERY,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, int, float]:
    """
    Multi‑source Dijkstra con Huff‑style scoring (β=2).

    Returns:
        best_seed_idx : 2D array of seed indices (or -1)
        best_time_min : 2D array of travel times in minutes (or inf)
        best_dist_km  : 2D array of path distance in km (or inf)
        assigned      : number of target pixels assigned
        elapsed_sec   : solver wall time
    """
    h, w = friction.shape
    n_seeds = len(seeds)

    # Precompute source data
    seed_rows = np.array([s["row"] for s in seeds], dtype=np.int32)
    seed_cols = np.array([s["col"] for s in seeds], dtype=np.int32)
    sqrt_attr = np.array(
        [math.sqrt(max(s["attractiveness"], ATTR_MIN)) for s in seeds],
        dtype=np.float64,
    )

    # Determine which connected components contain at least one seed
    seed_labels = labels[seed_rows, seed_cols]
    max_label = labels.max()
    has_seed = np.zeros(max_label + 1, dtype=bool)
    has_seed[seed_labels[seed_labels > 0]] = True

    # State arrays
    best_adj = np.full((h, w), np.inf, dtype=np.float64)   # adjusted time
    best_time = np.full((h, w), np.inf, dtype=np.float64)  # raw time (minutes)
    best_dist = np.full((h, w), np.inf, dtype=np.float64)  # distance (km)
    best_seed = np.full((h, w), -1, dtype=np.int32)
    finalized = np.zeros((h, w), dtype=bool)

    # Priority queue: (adj_time, row, col, seed_idx)
    heap = []
    for si in range(n_seeds):
        r = seed_rows[si]
        c = seed_cols[si]
        if not traversable[r, c]:
            continue
        # At source pixel, tie‑break in favour of higher attractiveness
        if (0.0 < best_adj[r, c]) or (
            best_adj[r, c] == 0.0
            and best_seed[r, c] >= 0
            and sqrt_attr[si] > sqrt_attr[best_seed[r, c]]
        ):
            best_adj[r, c] = 0.0
            best_time[r, c] = 0.0
            best_dist[r, c] = 0.0
            best_seed[r, c] = si
            heapq.heappush(heap, (0.0, r, c, si))

    total_targets = int(np.count_nonzero(target_mask & traversable))
    assigned = 0
    t_start = time.perf_counter()
    last_log = 0

    while heap and assigned < total_targets:
        adj, r, c, si = heapq.heappop(heap)
        if finalized[r, c]:
            continue
        if adj > best_adj[r, c]:
            continue

        finalized[r, c] = True
        if target_mask[r, c]:
            assigned += 1
            if assigned - last_log >= log_every:
                _print_progress(
                    mode_name,
                    assigned,
                    total_targets,
                    time.perf_counter() - t_start,
                )
                last_log = assigned

        inv_scale = 1.0 / sqrt_attr[si]
        t_curr = best_time[r, c]
        d_curr = best_dist[r, c]

        for dr, dc, dist_factor in NEIGHBORS:
            nr = r + dr
            nc = c + dc
            if nr < 0 or nr >= h or nc < 0 or nc >= w:
                continue
            if finalized[nr, nc] or not traversable[nr, nc]:
                continue

            # Edge cost: average friction × distance (metres) → minutes
            edge_time = float(
                0.5 * (friction[r, c] + friction[nr, nc])
                * (pixel_size_m * dist_factor)
            )
            t_new = t_curr + edge_time
            adj_new = t_new * inv_scale

            # Distance in km (geometric, independent of friction)
            edge_km = (pixel_size_m * dist_factor) / 1000.0
            d_new = d_curr + edge_km

            # Lexicographic tie‑break: lower adjusted time, then lower raw time
            if (adj_new + 1e-12) < best_adj[nr, nc] or (
                abs(adj_new - best_adj[nr, nc]) <= 1e-12
                and t_new < best_time[nr, nc]
            ):
                best_adj[nr, nc] = adj_new
                best_time[nr, nc] = t_new
                best_dist[nr, nc] = d_new
                best_seed[nr, nc] = si
                heapq.heappush(heap, (adj_new, nr, nc, si))

    # -------------------------------------------------------------------------
    # Component‑aware fallback
    # -------------------------------------------------------------------------
    unassigned = target_mask & traversable & (best_seed < 0)
    eligible = unassigned & has_seed[labels]
    fallback_hits = 0
    if np.any(eligible):
        unique_lbls = np.unique(labels[eligible])
        for lbl in unique_lbls:
            if lbl <= 0:
                continue
            comp_mask = labels == lbl
            comp_targets = eligible & comp_mask
            if not np.any(comp_targets):
                continue

            comp_seed_idx = np.where(seed_labels == lbl)[0]
            if comp_seed_idx.size == 0:
                continue

            fallback_hits += int(np.count_nonzero(comp_targets))

            # Reset state inside the component
            best_adj_comp = np.full((h, w), np.inf, dtype=np.float64)
            best_time_comp = np.full((h, w), np.inf, dtype=np.float64)
            best_dist_comp = np.full((h, w), np.inf, dtype=np.float64)
            best_seed_comp = np.full((h, w), -1, dtype=np.int32)
            finalized_comp = np.zeros((h, w), dtype=bool)

            heap_comp = []
            for si in comp_seed_idx:
                r0 = seed_rows[si]
                c0 = seed_cols[si]
                if not comp_mask[r0, c0] or not traversable[r0, c0]:
                    continue
                best_adj_comp[r0, c0] = 0.0
                best_time_comp[r0, c0] = 0.0
                best_dist_comp[r0, c0] = 0.0
                best_seed_comp[r0, c0] = si
                heapq.heappush(heap_comp, (0.0, r0, c0, si))

            while heap_comp:
                adj, r, c, si = heapq.heappop(heap_comp)
                if finalized_comp[r, c]:
                    continue
                if adj > best_adj_comp[r, c]:
                    continue
                if not comp_mask[r, c]:
                    continue

                finalized_comp[r, c] = True

                inv_scale = 1.0 / sqrt_attr[si]
                t_curr = best_time_comp[r, c]
                d_curr = best_dist_comp[r, c]

                for dr, dc, dist_factor in NEIGHBORS:
                    nr = r + dr
                    nc = c + dc
                    if nr < 0 or nr >= h or nc < 0 or nc >= w:
                        continue
                    if (
                        finalized_comp[nr, nc]
                        or not traversable[nr, nc]
                        or not comp_mask[nr, nc]
                    ):
                        continue

                    edge_time = float(
                        0.5 * (friction[r, c] + friction[nr, nc])
                        * (pixel_size_m * dist_factor)
                    )
                    t_new = t_curr + edge_time
                    adj_new = t_new * inv_scale

                    edge_km = (pixel_size_m * dist_factor) / 1000.0
                    d_new = d_curr + edge_km

                    if (adj_new + 1e-12) < best_adj_comp[nr, nc] or (
                        abs(adj_new - best_adj_comp[nr, nc]) <= 1e-12
                        and t_new < best_time_comp[nr, nc]
                    ):
                        best_adj_comp[nr, nc] = adj_new
                        best_time_comp[nr, nc] = t_new
                        best_dist_comp[nr, nc] = d_new
                        best_seed_comp[nr, nc] = si
                        heapq.heappush(heap_comp, (adj_new, nr, nc, si))

            # Merge fallback results into global arrays
            update = comp_targets & (best_seed_comp >= 0)
            best_seed[update] = best_seed_comp[update]
            best_time[update] = best_time_comp[update]
            best_dist[update] = best_dist_comp[update]

    elapsed = time.perf_counter() - t_start
    assigned_final = int(np.count_nonzero(target_mask & traversable & (best_seed >= 0)))
    _print_progress(mode_name, assigned_final, total_targets, elapsed)

    return best_seed, best_time, best_dist, assigned_final, elapsed


def seeds_to_raster(
    best_seed_idx: np.ndarray,
    best_time_min: np.ndarray,
    best_dist_km: np.ndarray,
    seeds: List[Dict],
    inhabited_mask: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Convert seed indices, times and distances to final rasters."""
    h, w = best_seed_idx.shape
    id_raster = np.full((h, w), NODATA, dtype=np.float32)
    time_raster = np.full((h, w), NODATA, dtype=np.float32)
    dist_raster = np.full((h, w), NODATA, dtype=np.float32)

    valid = inhabited_mask & (best_seed_idx >= 0) & np.isfinite(best_time_min)
    if np.any(valid):
        seed_ids = np.array([s["reseller_id"] for s in seeds], dtype=np.float64)
        idx = best_seed_idx[valid]
        id_raster[valid] = seed_ids[idx].astype(np.float32)
        time_raster[valid] = best_time_min[valid].astype(np.float32)
        dist_raster[valid] = best_dist_km[valid].astype(np.float32)

    return id_raster, time_raster, dist_raster


def write_multiband_raster(
    out_path: Path,
    ref_profile: Dict,
    bands: Sequence[np.ndarray],
    band_names: Sequence[str],
) -> None:
    """Write a multi‑band GeoTIFF with LZW compression and band descriptions."""
    profile = ref_profile.copy()
    profile.update(
        driver="GTiff",
        dtype="float32",
        count=len(bands),
        compress="lzw",
        nodata=NODATA,
        tiled=True,
        bigtiff="IF_SAFER",
    )

    # Ensure tile sizes are multiples of 16
    h, w = profile["height"], profile["width"]
    tile_x = min(256, w)
    tile_y = min(256, h)
    tile_x = max(16, (tile_x // 16) * 16)
    tile_y = max(16, (tile_y // 16) * 16)
    profile["blockxsize"] = tile_x
    profile["blockysize"] = tile_y

    if save_outputs.get(_key_from_path(out_path), True):
        with rasterio.open(out_path, "w", **profile) as dst:
            for i, (arr, name) in enumerate(zip(bands, band_names), start=1):
                dst.write(arr.astype(np.float32), i)
                dst.set_band_description(i, name)

    _cache_raster(out_path, np.stack([b.astype(np.float32, copy=False) for b in bands], axis=0), profile, NODATA, band_names)


# -----------------------------------------------------------------------------
# Exact metric recomputation (Phase 2)
# -----------------------------------------------------------------------------
def recompute_exact_metrics(
    mode_name: str,
    friction: np.ndarray,
    traversable: np.ndarray,
    id_raster: np.ndarray,                # Phase‑1 assignment (IDs, NODATA elsewhere)
    phase1_time_raster: np.ndarray,       # approximate times from Phase 1 (for limit)
    seeds: List[Dict],
    pixel_size_m: float,
    time_limit_factor: float = TIME_LIMIT_FACTOR,
    log_every: int = 500,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Recompute exact travel times and distances for pixels assigned to each reseller.
    For every seed, run a single‑source Dijkstra (no finalisation) and record
    the true metrics only for pixels whose Phase‑1 assignment matches the seed.

    Early termination:
        - Stop when all assigned pixels of the current seed have been reached.
        - Alternatively, stop when the time exceeds `max_phase1_time * time_limit_factor`.
          If this second condition triggers, a warning is printed.

    Returns:
        exact_time_raster : 2D array with exact travel times (minutes, NODATA for unassigned)
        exact_dist_raster : 2D array with exact distances (km, NODATA for unassigned)
    """
    h, w = friction.shape
    exact_time = np.full((h, w), NODATA, dtype=np.float32)
    exact_dist = np.full((h, w), NODATA, dtype=np.float32)

    # Build mapping from seed ID to its index
    seed_id_to_idx = {}
    for si, s in enumerate(seeds):
        sid = s["reseller_id"]
        if sid not in seed_id_to_idx:
            seed_id_to_idx[sid] = si

    assigned_mask = (id_raster != NODATA) & traversable
    unique_ids = np.unique(id_raster[assigned_mask])

    n_seeds_total = len(unique_ids)
    n_processed = 0
    t_start = time.perf_counter()
    last_log = 0
    warnings_issued = set()

    print(f"[{mode_name}-exact] Processing {n_seeds_total:,} resellers...")

    for sid in unique_ids:
        si = seed_id_to_idx.get(sid)
        if si is None:
            continue
        seed = seeds[si]
        sr, sc = seed["row"], seed["col"]
        if not traversable[sr, sc]:
            continue

        target_mask = assigned_mask & (id_raster == sid)
        target_coords = np.argwhere(target_mask)  # list of (r,c)
        target_count = len(target_coords)
        if target_count == 0:
            continue

        # Determine maximum allowed time for early termination
        phase1_times = phase1_time_raster[target_mask]
        max_phase1_time = np.max(phase1_times) if len(phase1_times) > 0 else 0.0
        time_limit = max_phase1_time * time_limit_factor if max_phase1_time > 0 else np.inf

        # Dijkstra state
        dist_time = np.full((h, w), np.inf, dtype=np.float64)
        dist_dist = np.full((h, w), np.inf, dtype=np.float64)
        visited = np.zeros((h, w), dtype=bool)

        dist_time[sr, sc] = 0.0
        dist_dist[sr, sc] = 0.0
        heap = [(0.0, sr, sc)]

        reached_targets = 0
        target_set = set((r, c) for r, c in target_coords)

        while heap:
            t, r, c = heapq.heappop(heap)
            if visited[r, c]:
                continue
            visited[r, c] = True

            # If time limit exceeded (and we haven't reached all targets), stop early
            if t > time_limit and reached_targets < target_count:
                if sid not in warnings_issued:
                    print(f"  Warning: Seed {sid} exceeded time limit ({t:.1f} > {time_limit:.1f} min) "
                          f"before reaching all targets ({reached_targets}/{target_count}).")
                    warnings_issued.add(sid)
                break

            # If this cell is a target for this seed, record exact metrics
            if (r, c) in target_set:
                exact_time[r, c] = t
                exact_dist[r, c] = dist_dist[r, c]
                reached_targets += 1
                if reached_targets == target_count:
                    break  # all done

            t_curr = t
            d_curr = dist_dist[r, c]

            for dr, dc, dist_factor in NEIGHBORS:
                nr = r + dr
                nc = c + dc
                if nr < 0 or nr >= h or nc < 0 or nc >= w:
                    continue
                if visited[nr, nc] or not traversable[nr, nc]:
                    continue

                edge_time = float(
                    0.5 * (friction[r, c] + friction[nr, nc])
                    * (pixel_size_m * dist_factor)
                )
                t_new = t_curr + edge_time
                edge_km = (pixel_size_m * dist_factor) / 1000.0
                d_new = d_curr + edge_km

                if t_new < dist_time[nr, nc] - 1e-12:
                    dist_time[nr, nc] = t_new
                    dist_dist[nr, nc] = d_new
                    heapq.heappush(heap, (t_new, nr, nc))

        # End while heap

        n_processed += 1
        if n_processed - last_log >= log_every:
            elapsed = time.perf_counter() - t_start
            pct = 100.0 * n_processed / n_seeds_total
            rate = _safe_ratio(n_processed, elapsed)
            remaining = n_seeds_total - n_processed
            eta = _safe_ratio(remaining, max(rate, 1e-12))
            print(
                f"[{mode_name}-exact] {n_processed:,}/{n_seeds_total:,} resellers done "
                f"({pct:.1f}%) | {rate:.1f} res/s | ETA {eta/60:.1f} min"
            )
            last_log = n_processed

    # End loop over seeds
    elapsed = time.perf_counter() - t_start
    print(f"[{mode_name}-exact] All resellers processed. Assigned pixels recomputed.")

    return exact_time, exact_dist


# -----------------------------------------------------------------------------
# Main pipeline
# -----------------------------------------------------------------------------
def run_huff_pipeline(dataset_dir: Path = DATASET_DIR) -> Dict:
    """
    Execute the full Huff assignment and write outputs.

    Returns a dictionary with run statistics.
    """
    ds = Path(dataset_dir)
    t_total_start = time.perf_counter()

    # -------------------------------------------------------------------------
    # Load and prepare rasters
    # -------------------------------------------------------------------------
    t0 = time.perf_counter()
    pop, ref_profile = read_reference_population(ds / POPULATION_TIF)
    inhabited = np.isfinite(pop) & (pop > 0.0)
    t_read_pop = time.perf_counter() - t0

    t0 = time.perf_counter()
    car_share_raw = reproject_to_reference(
        ds / CAR_SHARE_TIF, ref_profile, Resampling.bilinear
    )
    walk_friction = reproject_to_reference(
        ds / FRICTION_WALK_TIF, ref_profile, Resampling.bilinear
    )
    moto_friction = reproject_to_reference(
        ds / FRICTION_MOTO_TIF, ref_profile, Resampling.bilinear
    )

    car_share = normalize_car_share(car_share_raw)
    walk_share = 1.0 - car_share

    walk_friction = sanitize_friction(walk_friction)
    moto_friction = sanitize_friction(moto_friction)
    t_reproject = time.perf_counter() - t0

    # -------------------------------------------------------------------------
    # Load reseller seeds
    # -------------------------------------------------------------------------
    t0 = time.perf_counter()
    seeds_all = load_reseller_seeds(
        gpkg_path=ds / RESELLER_GPKG,
        layer=RESELLER_LAYER,
        id_col=RESELLER_ID_COL,
        attr_col=ATTRACTIVENESS_COL,
        ref_profile=ref_profile,
    )
    t_load_seeds = time.perf_counter() - t0

    # Build traversable masks
    walk_traversable = np.isfinite(walk_friction) & (walk_friction > 0.0)
    moto_traversable = np.isfinite(moto_friction) & (moto_friction > 0.0)

    seeds_walk = [s for s in seeds_all if walk_traversable[s["row"], s["col"]]]
    seeds_moto = [s for s in seeds_all if moto_traversable[s["row"], s["col"]]]

    if not seeds_walk:
        raise RuntimeError("No walkable reseller seeds.")
    if not seeds_moto:
        raise RuntimeError("No motorised‑accessible reseller seeds.")

    # Save lookup CSV (unique seeds per pixel, merging both modes)
    lookup_dict = {}
    for s in seeds_walk + seeds_moto:
        key = (s["row"], s["col"])
        lookup_dict[key] = s
    lookup_seeds = list(lookup_dict.values())
    _write_csv(ds / OUTPUT_LOOKUP, pd.DataFrame(lookup_seeds))

    # -------------------------------------------------------------------------
    # Connected components
    # -------------------------------------------------------------------------
    t0 = time.perf_counter()
    walk_labels = connected_components_8(walk_traversable)
    moto_labels = connected_components_8(moto_traversable)
    t_components = time.perf_counter() - t0

    print(
        f"Inhabited pixels: {inhabited.sum():,} | "
        f"Walk seeds: {len(seeds_walk):,} | Moto seeds: {len(seeds_moto):,}"
    )

    # -------------------------------------------------------------------------
    # Phase 1: Approximate assignment (multi‑source Dijkstra)
    # -------------------------------------------------------------------------
    walk_seed_idx, walk_time_approx, walk_dist_approx, walk_assigned, t_walk_solver = run_multisource_dijkstra(
        mode_name="walk",
        friction=walk_friction,
        traversable=walk_traversable,
        target_mask=inhabited,
        labels=walk_labels,
        seeds=seeds_walk,
        pixel_size_m=PIXEL_SIZE_M,
    )

    moto_seed_idx, moto_time_approx, moto_dist_approx, moto_assigned, t_moto_solver = run_multisource_dijkstra(
        mode_name="moto",
        friction=moto_friction,
        traversable=moto_traversable,
        target_mask=inhabited,
        labels=moto_labels,
        seeds=seeds_moto,
        pixel_size_m=PIXEL_SIZE_M,
    )

    # Convert to ID rasters (using approximate times/distances for now)
    walk_id, walk_time_approx_out, walk_dist_approx_out = seeds_to_raster(
        walk_seed_idx, walk_time_approx, walk_dist_approx, seeds_walk, inhabited
    )
    moto_id, moto_time_approx_out, moto_dist_approx_out = seeds_to_raster(
        moto_seed_idx, moto_time_approx, moto_dist_approx, seeds_moto, inhabited
    )

    # -------------------------------------------------------------------------
    # Phase 2: Exact metric recomputation
    # -------------------------------------------------------------------------
    t0_exact = time.perf_counter()
    walk_time_exact, walk_dist_exact = recompute_exact_metrics(
        mode_name="walk",
        friction=walk_friction,
        traversable=walk_traversable,
        id_raster=walk_id,
        phase1_time_raster=walk_time_approx_out,
        seeds=seeds_walk,
        pixel_size_m=PIXEL_SIZE_M,
    )
    moto_time_exact, moto_dist_exact = recompute_exact_metrics(
        mode_name="moto",
        friction=moto_friction,
        traversable=moto_traversable,
        id_raster=moto_id,
        phase1_time_raster=moto_time_approx_out,
        seeds=seeds_moto,
        pixel_size_m=PIXEL_SIZE_M,
    )
    t_exact = time.perf_counter() - t0_exact







    # ---------------------------------------------------------------------
    # Optional diagnostic: compare approximate vs exact metrics
    # ---------------------------------------------------------------------
    def _print_comparison(mode, approx_time, exact_time, approx_dist, exact_dist, mask):
        valid = mask & (approx_time != NODATA) & (exact_time != NODATA)
        if np.any(valid):
            time_diff = np.abs(approx_time[valid] - exact_time[valid])
            dist_diff = np.abs(approx_dist[valid] - exact_dist[valid])
            print(f"[{mode}] Approx vs exact time diff (min): median={np.median(time_diff):.2f}, max={np.max(time_diff):.2f}")
            print(f"[{mode}] Approx vs exact dist diff (km):  median={np.median(dist_diff):.2f}, max={np.max(dist_diff):.2f}")

    _print_comparison("walk", walk_time_approx_out, walk_time_exact, walk_dist_approx_out, walk_dist_exact, inhabited)
    _print_comparison("moto", moto_time_approx_out, moto_time_exact, moto_dist_approx_out, moto_dist_exact, inhabited)









    # Replace approximate time/distance rasters with exact ones
    walk_time_out = walk_time_exact
    walk_dist_out = walk_dist_exact
    moto_time_out = moto_time_exact
    moto_dist_out = moto_dist_exact

    # -------------------------------------------------------------------------
    # Prepare share bands with nodata outside inhabited areas
    # -------------------------------------------------------------------------
    car_band = np.full_like(car_share, NODATA, dtype=np.float32)
    walk_band = np.full_like(walk_share, NODATA, dtype=np.float32)
    car_band[inhabited] = car_share[inhabited]
    walk_band[inhabited] = walk_share[inhabited]

    # -------------------------------------------------------------------------
    # Write multi‑band GeoTIFF
    # -------------------------------------------------------------------------
    write_multiband_raster(
        ds / OUTPUT_RASTER,
        ref_profile,
        bands=(
            car_band,
            walk_band,
            walk_id,
            walk_time_out,
            walk_dist_out,
            moto_id,
            moto_time_out,
            moto_dist_out,
        ),
        band_names=[
            "car_share",
            "walk_share",
            "best_reseller_id_walk",
            "best_time_walk_min",
            "best_distance_walk_km",
            "best_reseller_id_car",
            "best_time_car_min",
            "best_distance_car_km",
        ],
    )

    # -------------------------------------------------------------------------
    # Final statistics and JSON profile
    # -------------------------------------------------------------------------
    inhabited_n = int(inhabited.sum())
    walk_assigned_final = int(np.count_nonzero(inhabited & (walk_id != NODATA)))
    moto_assigned_final = int(np.count_nonzero(inhabited & (moto_id != NODATA)))

    # Compute summary statistics from exact values
    walk_valid_times = walk_time_out[(inhabited) & (walk_id != NODATA)]
    moto_valid_times = moto_time_out[(inhabited) & (moto_id != NODATA)]
    walk_valid_dist = walk_dist_out[(inhabited) & (walk_id != NODATA)]
    moto_valid_dist = moto_dist_out[(inhabited) & (moto_id != NODATA)]

    total_elapsed = time.perf_counter() - t_total_start

    stats = {
        "run_date": time.strftime("%Y-%m-%d %H:%M:%S"),
        "inputs": {
            "dataset_dir": str(ds),
            "population": POPULATION_TIF,
            "car_share": CAR_SHARE_TIF,
            "friction_walk": FRICTION_WALK_TIF,
            "friction_moto": FRICTION_MOTO_TIF,
            "resellers": f"{RESELLER_GPKG}:{RESELLER_LAYER}",
        },
        "parameters": {
            "beta": BETA,
            "epsilon": EPS,
            "attr_min": ATTR_MIN,
            "pixel_size_m": PIXEL_SIZE_M,
        },
        "counts": {
            "inhabited_pixels": inhabited_n,
            "seeds_walk": len(seeds_walk),
            "seeds_moto": len(seeds_moto),
            "assigned_walk": walk_assigned_final,
            "assigned_moto": moto_assigned_final,
        },
        "percentages": {
            "walk_assignment_pct": 100.0 * _safe_ratio(walk_assigned_final, inhabited_n),
            "moto_assignment_pct": 100.0 * _safe_ratio(moto_assigned_final, inhabited_n),
        },
        "timings_seconds": {
            "read_population": t_read_pop,
            "reproject": t_reproject,
            "load_seeds": t_load_seeds,
            "connected_components": t_components,
            "walk_solver_phase1": t_walk_solver,
            "moto_solver_phase1": t_moto_solver,
            "exact_recomputation": t_exact,
            "total": total_elapsed,
        },
        "outputs": {
            "raster": OUTPUT_RASTER,
            "lookup_csv": OUTPUT_LOOKUP,
        },
    }

    if OUTPUT_PROFILE:
        _write_json(ds / OUTPUT_PROFILE, stats)

    # Print summary
    print("\n=== SUMMARY ===")
    print(f"Pixels processed      : {inhabited_n:,}")
    print(f"Walk assigned         : {walk_assigned_final:,} ({stats['percentages']['walk_assignment_pct']:.2f}%)")
    print(f"Moto assigned         : {moto_assigned_final:,} ({stats['percentages']['moto_assignment_pct']:.2f}%)")
    if len(walk_valid_times) > 0:
        print(f"Walk time (min)       : min={np.min(walk_valid_times):.2f} median={np.median(walk_valid_times):.2f} max={np.max(walk_valid_times):.2f}")
    if len(moto_valid_times) > 0:
        print(f"Moto time (min)       : min={np.min(moto_valid_times):.2f} median={np.median(moto_valid_times):.2f} max={np.max(moto_valid_times):.2f}")
    print(f"Total elapsed time    : {total_elapsed:.2f} seconds")
    print(f"Output raster         : {ds / OUTPUT_RASTER}")
    print(f"Lookup CSV            : {ds / OUTPUT_LOOKUP}")
    if OUTPUT_PROFILE:
        print(f"Run profile           : {ds / OUTPUT_PROFILE}")
    if len(walk_valid_dist) > 0:
        print(f"Walk distance (km)    : min={np.min(walk_valid_dist):.2f} median={np.median(walk_valid_dist):.2f} max={np.max(walk_valid_dist):.2f}")
    if len(moto_valid_dist) > 0:
        print(f"Moto distance (km)    : min={np.min(moto_valid_dist):.2f} median={np.median(moto_valid_dist):.2f} max={np.max(moto_valid_dist):.2f}")

    return stats


# -----------------------------------------------------------------------------
# Execute the pipeline
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    run_stats = run_huff_pipeline()

"""
LPG demand calibration and reseller client allocation (Nigeria, 1 km, EPSG:3857)

This version reads pixel preference from a multilayer raster produced in step 4.2
"""

DATA_DIR = str(DATA_DIR)

# Inputs
PIXEL_PREFERENCE_RASTER = str(HUFF_PIXEL_RASTER_PATH)
RESELLER_GPKG = str(FULL_CHAIN_GPKG_PATH)
RESELLER_LAYER = "resell_and_filling"
POPULATION_RASTER = str(POPULATION_RASTER_PATH)
URBAN_RASTER = str(URBAN_RASTER_PATH)
BOUNDARY_FILE = str(COUNTRY_BOUNDARIES_PATH)

# Outputs
OUTPUT_LPG_USE_RASTER = str(LPG_USE_SHARE_PATH)
OUTPUT_RESELLER_GPKG = str(SELL_POINT_GPKG_PATH)
OUTPUT_RESELLER_LAYER = "sell_point_clients"

RESELLER_ID_COLUMN = "id_res&fil"

# Calibration parameters
WALK_THRESHOLD_MIN = 30
CAR_THRESHOLD_MIN = 45
urban_share = 0.25468  #  source ONSTOVE
rural_share = 0.027905  #  source ONSTOVE
URBAN_THRESHOLD = 20

# Nodata conventions
NODATA_FLOAT = -9999.0
NODATA_INT = -1


def _read_raster(path: str):
    cached = _read_cached_raster(path)
    if cached is not None:
        arr = cached["array"]
        if arr.ndim == 3:
            arr = arr[0]
        arr = arr.astype(np.float32, copy=False)
        profile = cached["profile"].copy()
        nodata = cached.get("nodata")
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
        return arr, profile, nodata
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    return arr, profile, nodata


def _align_to_reference(path: str, ref_profile: dict, resampling: Resampling) -> np.ndarray:
    cached = _read_cached_raster(path)
    dst = np.full((ref_profile["height"], ref_profile["width"]), np.nan, dtype=np.float32)
    if cached is not None:
        src_arr = cached["array"]
        if src_arr.ndim == 3:
            src_arr = src_arr[0]
        reproject(
            source=src_arr,
            destination=dst,
            src_transform=cached["profile"]["transform"],
            src_crs=cached["profile"]["crs"],
            src_nodata=cached.get("nodata"),
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
        return dst
    with rasterio.open(path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def _read_pixel_preference_raster(path: str) -> dict[str, np.ndarray]:
    cached = _read_cached_raster(path)
    if cached is not None:
        arr = cached["array"]
        if arr.ndim == 2:
            raise RuntimeError(f"Expected at least 7 bands in {path}, found 1.")
        if arr.shape[0] < 7:
            raise RuntimeError(f"Expected at least 7 bands in {path}, found {arr.shape[0]}.")
        nodata = cached.get("nodata")
        arr = arr.astype(np.float32, copy=False)
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
        return {
            "car_share": arr[0],
            "walk_share": arr[1],
            "best_reseller_id_walk": arr[2],
            "best_time_walk_min": arr[3],
            "best_reseller_id_car": arr[5],
            "best_time_car_min": arr[6],
        }
    with rasterio.open(path) as src:
        if src.count < 7:   # almeno fino a banda 7 (best_time_car_min)
            raise RuntimeError(f"Expected at least 7 bands in {path}, found {src.count}.")
        # Leggi le bande nell'ordine corretto:
        # 1: car_share, 2: walk_share, 3: best_reseller_id_walk, 4: best_time_walk_min,
        # 6: best_reseller_id_car, 7: best_time_car_min
        # (saltiamo banda 5: distanza walk, banda 8: distanza car)
        arr = src.read(indexes=[1, 2, 3, 4, 6, 7]).astype(np.float32, copy=False)
        nodata = src.nodata

    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)

    return {
        "car_share": arr[0],
        "walk_share": arr[1],
        "best_reseller_id_walk": arr[2],
        "best_time_walk_min": arr[3],
        "best_reseller_id_car": arr[4],
        "best_time_car_min": arr[5],
    }


def _read_reseller_ids(gdf: gpd.GeoDataFrame) -> np.ndarray:
    if RESELLER_ID_COLUMN not in gdf.columns:
        raise KeyError(f"Missing column '{RESELLER_ID_COLUMN}' in reseller layer.")

    rid = pd.to_numeric(gdf[RESELLER_ID_COLUMN], errors="coerce")
    if rid.isna().any():
        raise ValueError(f"Column '{RESELLER_ID_COLUMN}' contains non-numeric values.")
    if (rid <= 0).any():
        raise ValueError(f"Column '{RESELLER_ID_COLUMN}' must contain positive values.")
    if not rid.is_unique:
        raise ValueError(f"Column '{RESELLER_ID_COLUMN}' must be unique in layer '{RESELLER_LAYER}'.")

    return rid.astype(np.int64).to_numpy()


def _write_float_raster(path: str, array: np.ndarray, ref_profile: dict):
    profile = ref_profile.copy()
    profile.update(dtype="float32", count=1, nodata=NODATA_FLOAT, compress="lzw")
    out = np.where(np.isfinite(array), array, NODATA_FLOAT).astype(np.float32)
    if save_outputs.get(_key_from_path(path), True):
        with rasterio.open(path, "w", **profile) as dst:
            dst.write(out, 1)
    _cache_raster(path, out, profile, NODATA_FLOAT, None)


print("[1/7] Loading pixel preference raster...")
pixel_pref = _read_pixel_preference_raster(PIXEL_PREFERENCE_RASTER)

print("[2/7] Loading population and urban rasters...")
pop, pop_profile, _ = _read_raster(POPULATION_RASTER)
transform = pop_profile["transform"]
crs = pop_profile["crs"]
height, width = pop.shape

urban = _align_to_reference(URBAN_RASTER, pop_profile, Resampling.nearest)
urban = np.where(np.isfinite(urban), urban, np.nan).astype(np.float32)

print(f"Grid: {width} x {height}, CRS={crs}")

print("[3/7] Building Nigeria mask...")
boundary = gpd.read_file(BOUNDARY_FILE)
if boundary.crs != crs:
    boundary = boundary.to_crs(crs)
nigeria_geom = boundary.geometry.union_all()
mask_nigeria = geometry_mask(
    [nigeria_geom],
    transform=transform,
    invert=True,
    out_shape=(height, width),
)

pop = np.where(np.isfinite(pop), np.maximum(pop, 0.0), np.nan).astype(np.float32)

print("[4/7] Preparing access eligibility and calibration factors...")
valid_pref = np.isfinite(pixel_pref["walk_share"]) & np.isfinite(pixel_pref["car_share"])
valid_pref = valid_pref & ((pixel_pref["best_reseller_id_walk"] >= 0) | (pixel_pref["best_reseller_id_car"] >= 0))
rows, cols = np.where(valid_pref)
rows = rows.astype(np.int64)
cols = cols.astype(np.int64)

if rows.size == 0:
    raise RuntimeError("No valid preference pixels found in multilayer raster.")

pixel_pop = pop[rows, cols].astype(np.float64)
valid_pop = np.isfinite(pixel_pop) & (pixel_pop > 0)
rows = rows[valid_pop]
cols = cols[valid_pop]
pixel_pop = pixel_pop[valid_pop]

walk_share = np.clip(pixel_pref["walk_share"][rows, cols].astype(np.float64), 0.0, 1.0)
car_share = np.clip(pixel_pref["car_share"][rows, cols].astype(np.float64), 0.0, 1.0)
share_sum = walk_share + car_share
if np.any(share_sum <= 0):
    raise RuntimeError("Invalid walk/car shares: at least one pixel has zero total share.")
walk_share = walk_share / share_sum
car_share = car_share / share_sum

walk_time = pixel_pref["best_time_walk_min"][rows, cols].astype(np.float64)
car_time = pixel_pref["best_time_car_min"][rows, cols].astype(np.float64)
walk_reseller_id = np.where(np.isfinite(pixel_pref["best_reseller_id_walk"][rows, cols]), pixel_pref["best_reseller_id_walk"][rows, cols], NODATA_INT).astype(np.int64)
car_reseller_id = np.where(np.isfinite(pixel_pref["best_reseller_id_car"][rows, cols]), pixel_pref["best_reseller_id_car"][rows, cols], NODATA_INT).astype(np.int64)

walk_eligible = np.isfinite(walk_time) & (walk_time <= WALK_THRESHOLD_MIN)
car_eligible = np.isfinite(car_time) & (car_time <= CAR_THRESHOLD_MIN)

urban_pixel = urban[rows, cols]
urban_flag = np.where(np.isfinite(urban_pixel), urban_pixel >= URBAN_THRESHOLD, False)
rural_flag = ~urban_flag

urban_pop_total = float(np.nansum(pixel_pop[urban_flag]))
rural_pop_total = float(np.nansum(pixel_pop[rural_flag]))
if urban_pop_total <= 0 or rural_pop_total <= 0:
    raise RuntimeError("Urban or rural population total is zero. Check urban/pop alignment.")

urban_eligible_pop = float(np.nansum(pixel_pop[urban_flag] * (walk_share[urban_flag] * walk_eligible[urban_flag] + car_share[urban_flag] * car_eligible[urban_flag])))
rural_eligible_pop = float(np.nansum(pixel_pop[rural_flag] * (walk_share[rural_flag] * walk_eligible[rural_flag] + car_share[rural_flag] * car_eligible[rural_flag])))

if urban_eligible_pop <= 0:
    raise RuntimeError("Urban eligible population is zero.")
if rural_eligible_pop <= 0:
    raise RuntimeError("Rural eligible population is zero.")

urban_factor = (urban_share * urban_pop_total) / urban_eligible_pop
rural_factor = (rural_share * rural_pop_total) / rural_eligible_pop

if urban_factor > 1.0 + 1e-9:
    raise RuntimeError(f"Urban calibration factor is above 1 ({urban_factor:.4f}).")
if rural_factor > 1.0 + 1e-9:
    raise RuntimeError(f"Rural calibration factor is above 1 ({rural_factor:.4f}).")

eligible_weight = walk_share * walk_eligible.astype(np.float64) + car_share * car_eligible.astype(np.float64)
region_factor = np.where(urban_flag, urban_factor, rural_factor)
lpg_use_share_pixel = np.clip(region_factor * eligible_weight, 0.0, 1.0)

clients_walk_pixel = pixel_pop * walk_share * walk_eligible.astype(np.float64) * region_factor
clients_car_pixel = pixel_pop * car_share * car_eligible.astype(np.float64) * region_factor
clients_pixel = clients_walk_pixel + clients_car_pixel
clients_max_ideal_walk_pixel = pixel_pop * walk_share
clients_max_ideal_car_pixel = pixel_pop * car_share

national_pop = float(np.nansum(pixel_pop))
national_eligible_pop = float(np.nansum(pixel_pop * eligible_weight))
national_clients = float(np.nansum(clients_pixel))
expected_national_clients = float(urban_share * urban_pop_total + rural_share * rural_pop_total)

print("\n=== Calibration factors ===")
print(f"Urban factor: {urban_factor:.6f}")
print(f"Rural factor: {rural_factor:.6f}")
print("\n=== Pre-output checks ===")
print(f"National population: {national_pop:,.0f}")
print(f"National eligible population: {national_eligible_pop:,.0f}")
print(f"Expected national LPG users: {expected_national_clients:,.0f}")
print(f"Actual national LPG users   : {national_clients:,.0f}")
print(f"Difference                  : {national_clients - expected_national_clients:+,.6f}")

print("[5/7] Writing LPG use-share raster...")
lpg_use_share_raster = np.zeros((height, width), dtype=np.float32)
lpg_use_share_raster[~mask_nigeria] = NODATA_FLOAT
lpg_use_share_raster[rows, cols] = lpg_use_share_pixel.astype(np.float32)
_write_float_raster(OUTPUT_LPG_USE_RASTER, lpg_use_share_raster, pop_profile)
print(f"Saved raster: {OUTPUT_LPG_USE_RASTER}")

print("[6/7] Aggregating clients by reseller...")
reseller_df = gpd.read_file(RESELLER_GPKG, layer=RESELLER_LAYER)
if reseller_df.empty:
    raise RuntimeError("Reseller layer is empty.")
if reseller_df.crs != crs:
    reseller_df = reseller_df.to_crs(crs)
reseller_df = reseller_df[reseller_df.geometry.notna()].copy()
reseller_df = reseller_df[reseller_df.geometry.geom_type == "Point"].copy()
if reseller_df.empty:
    raise RuntimeError("No point geometries found in reseller layer.")

reseller_id = _read_reseller_ids(reseller_df)
reseller_df = reseller_df.copy()
reseller_df["reseller_id"] = reseller_id.astype(np.int64)

walk_clients = (
    pd.DataFrame(
        {
            "reseller_id": walk_reseller_id,
            "clients_walk": clients_walk_pixel,
            "clients_max_ideal_walk": clients_max_ideal_walk_pixel,
        }
    )
    .loc[lambda d: d["reseller_id"] >= 0]
    .groupby("reseller_id", as_index=False)
    .agg(
        clients_walk=("clients_walk", "sum"),
        clients_max_ideal_walk=("clients_max_ideal_walk", "sum"),
    )
)

car_clients = (
    pd.DataFrame(
        {
            "reseller_id": car_reseller_id,
            "clients_car": clients_car_pixel,
            "clients_max_ideal_car": clients_max_ideal_car_pixel,
        }
    )
    .loc[lambda d: d["reseller_id"] >= 0]
    .groupby("reseller_id", as_index=False)
    .agg(
        clients_car=("clients_car", "sum"),
        clients_max_ideal_car=("clients_max_ideal_car", "sum"),
    )
)

clients_agg = walk_clients.merge(car_clients, on="reseller_id", how="outer")
clients_agg["clients_walk"] = clients_agg["clients_walk"].fillna(0.0)
clients_agg["clients_car"] = clients_agg["clients_car"].fillna(0.0)
clients_agg["clients"] = clients_agg["clients_walk"] + clients_agg["clients_car"]
clients_agg["clients_max_ideal_walk"] = clients_agg["clients_max_ideal_walk"].fillna(0.0)
clients_agg["clients_max_ideal_car"] = clients_agg["clients_max_ideal_car"].fillna(0.0)
clients_agg["clients_max_ideal"] = clients_agg["clients_max_ideal_walk"] + clients_agg["clients_max_ideal_car"]

reseller_out = reseller_df.merge(clients_agg, on="reseller_id", how="left")
for col in ["clients_walk", "clients_car", "clients", "clients_max_ideal_walk", "clients_max_ideal_car", "clients_max_ideal"]:
    reseller_out[col] = reseller_out[col].fillna(0.0).astype(np.float64)

reseller_out = gpd.GeoDataFrame(reseller_out, geometry="geometry", crs=crs)
_write_gpkg_layer(OUTPUT_RESELLER_GPKG, OUTPUT_RESELLER_LAYER, reseller_out)
print(f"Saved reseller clients table: {OUTPUT_RESELLER_GPKG} | layer={OUTPUT_RESELLER_LAYER}")
print(f"Reseller id column: {RESELLER_ID_COLUMN}")

print("\n=== MAIN SUMMARY ===")
print(f"Urban population total  : {urban_pop_total:,.0f}")
print(f"Urban eligible population: {urban_eligible_pop:,.0f}")
print(f"Urban target share      : {urban_share:.6f}")
print(f"Urban achieved share    : {(urban_factor * urban_eligible_pop / urban_pop_total):.6f}")
print(f"Urban factor applied    : {urban_factor:.6f}")
print(f"Rural population total  : {rural_pop_total:,.0f}")
print(f"Rural eligible population: {rural_eligible_pop:,.0f}")
print(f"Rural target share      : {rural_share:.6f}")
print(f"Rural achieved share    : {(rural_factor * rural_eligible_pop / rural_pop_total):.6f}")
print(f"Rural factor applied    : {rural_factor:.6f}")
print(f"National actual share    : {national_clients / national_pop:.6f}")
print(f"National target share    : {expected_national_clients / national_pop:.6f}")
print("\nDone.")

# Diagnostics and final calibration checks

# Reuse the objects created in the previous cell when the notebook is run in order.

cached_lpg = _read_cached_raster(OUTPUT_LPG_USE_RASTER)
if cached_lpg is not None:
    lpg_use = cached_lpg["array"]
    if lpg_use.ndim == 3:
        lpg_use = lpg_use[0]
    lpg_nodata = cached_lpg.get("nodata")
else:
    with rasterio.open(OUTPUT_LPG_USE_RASTER) as src:
        lpg_use = src.read(1).astype(np.float32)
        lpg_nodata = src.nodata
if lpg_nodata is not None:
    lpg_use = np.where(lpg_use == lpg_nodata, np.nan, lpg_use)

pop, pop_profile, _ = _read_raster(POPULATION_RASTER)
urban = _align_to_reference(URBAN_RASTER, pop_profile, Resampling.nearest)
urban = np.where(np.isfinite(urban), urban, np.nan).astype(np.float32)
transform = pop_profile["transform"]
crs = pop_profile["crs"]
height, width = pop.shape

boundary = gpd.read_file(BOUNDARY_FILE)
if boundary.crs != crs:
    boundary = boundary.to_crs(crs)
nigeria_geom = boundary.geometry.union_all()
mask_nigeria = geometry_mask(
    [nigeria_geom],
    transform=transform,
    invert=True,
    out_shape=(height, width),
)

pop = np.where(np.isfinite(pop), np.maximum(pop, 0.0), np.nan).astype(np.float32)
urban_mask = mask_nigeria & np.isfinite(urban) & (urban >= URBAN_THRESHOLD)
rural_mask = mask_nigeria & ~urban_mask

urban_pop = float(np.nansum(pop[urban_mask]))
rural_pop = float(np.nansum(pop[rural_mask]))
national_pop = float(np.nansum(pop[mask_nigeria]))

urban_clients = float(np.nansum(pop[urban_mask] * lpg_use[urban_mask]))
rural_clients = float(np.nansum(pop[rural_mask] * lpg_use[rural_mask]))
national_clients = float(np.nansum(pop[mask_nigeria] * lpg_use[mask_nigeria]))

urban_actual = urban_clients / urban_pop if urban_pop > 0 else np.nan
rural_actual = rural_clients / rural_pop if rural_pop > 0 else np.nan
national_actual = national_clients / national_pop if national_pop > 0 else np.nan

target_national = (urban_share * urban_pop + rural_share * rural_pop) / national_pop if national_pop > 0 else np.nan

check_table = pd.DataFrame(
    {
        "group": ["urban", "rural", "national"],
        "target_share": [urban_share, rural_share, target_national],
        "actual_share": [urban_actual, rural_actual, national_actual],
    }
)
check_table["difference"] = check_table["actual_share"] - check_table["target_share"]

print("=== CALIBRATION CHECK ===")
print(check_table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# Check reseller aggregation
reseller_clients = _read_gpkg_layer(OUTPUT_RESELLER_GPKG, OUTPUT_RESELLER_LAYER)
for col in ["clients_walk", "clients_car", "clients", "clients_max_ideal"]:
    reseller_clients[col] = pd.to_numeric(reseller_clients[col], errors="coerce").fillna(0.0)

print("\n=== RESELLER SUMMARY ===")
print(f"Resellers in output            : {len(reseller_clients):,}")
print(f"Resellers with clients > 0     : {int((reseller_clients['clients'] > 0).sum()):,}")
print(f"National clients from raster   : {national_clients:,.0f}")
print(f"National clients from resellers: {float(reseller_clients['clients'].sum()):,.0f}")
print(f"Difference                     : {float(reseller_clients['clients'].sum() - national_clients):+.6f}")

print("\nTop 10 resellers by actual clients")
cols_to_show = [
    c for c in ["reseller_id", "id_res&fil", "id_depot", "clients", "clients_walk", "clients_car", "clients_max_ideal"]
    if c in reseller_clients.columns
]
print(
    reseller_clients.sort_values("clients", ascending=False)[cols_to_show]
    .head(10)
    .to_string(index=False)
)

print("\nTop 10 resellers by ideal assigned population")
print(
    reseller_clients.sort_values("clients_max_ideal", ascending=False)[cols_to_show]
    .head(10)
    .to_string(index=False)
)

"""
Prepare working GeoPackage for chain logistics (copy + temporary filling costs).

Behavior:
1) verify source file exists,
2) create/overwrite chain_with_cost.gpkg with only resell and filling,
3) set temporary split filling costs in filling layer (cost_fil_kg_out / cost_fil_kg_in).


time for the run: 7 min
"""

DATA_DIR = Path("dataset_big")
SOURCE_GPKG_PATH = FULL_CHAIN_GPKG_PATH
OUTPUT_GPKG_PATH = CHAIN_WITH_COST_PATH

RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
FILL_COST_OUT_COL = "cost_fil_kg_out"
FILL_COST_IN_COL = "cost_fil_kg_in"
LEGACY_COST_COL = "cost_kg"
TEMP_FILLING_COST_OUT = 0.6
TEMP_FILLING_COST_IN = 0.5

print("[1/4] Checking source GeoPackage...")
if not SOURCE_GPKG_PATH.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE_GPKG_PATH}")

print("[2/4] Creating reduced copied GeoPackage (only resell + filling)...")
if save_outputs.get(_key_from_path(OUTPUT_GPKG_PATH), True) and OUTPUT_GPKG_PATH.exists():
    OUTPUT_GPKG_PATH.unlink()

with sqlite3.connect(str(SOURCE_GPKG_PATH)) as conn:
    cur = conn.cursor()
    layers = cur.execute(
        "SELECT table_name FROM gpkg_contents WHERE data_type IN ('features', 'attributes')"
    ).fetchall()
    source_layer_names = {name for (name,) in layers}

required_layers = {RESELL_LAYER, FILLING_LAYER}
missing_source_layers = required_layers.difference(source_layer_names)
if missing_source_layers:
    raise KeyError(
        f"Missing required source layer(s): {sorted(missing_source_layers)} in {SOURCE_GPKG_PATH}"
    )

resell_src = gpd.read_file(SOURCE_GPKG_PATH, layer=RESELL_LAYER)
filling_src = gpd.read_file(SOURCE_GPKG_PATH, layer=FILLING_LAYER)
# Remove empty geometries to avoid issues with SQL spatial functions
resell_src = resell_src[~resell_src.geometry.is_empty]
filling_src = filling_src[~filling_src.geometry.is_empty]
if resell_src.empty:
    raise RuntimeError(f"Layer '{RESELL_LAYER}' is empty in source file.")
if filling_src.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty in source file.")

_write_gpkg_layer(OUTPUT_GPKG_PATH, RESELL_LAYER, resell_src)
_write_gpkg_layer(OUTPUT_GPKG_PATH, FILLING_LAYER, filling_src)
print(
    f"Created reduced copy: {OUTPUT_GPKG_PATH} | "
    f"resell={len(resell_src):,}, filling={len(filling_src):,}"
)

print("[3/4] Validating layers in reduced copied GeoPackage...")
if save_outputs.get(_key_from_path(OUTPUT_GPKG_PATH), True):
    with sqlite3.connect(str(OUTPUT_GPKG_PATH)) as conn:
        cur = conn.cursor()
        layers = cur.execute(
            "SELECT table_name FROM gpkg_contents WHERE data_type IN ('features', 'attributes')"
        ).fetchall()
        layer_names = {name for (name,) in layers}
        if not layer_names:
            raise RuntimeError(f"No layers found in copied file: {OUTPUT_GPKG_PATH}")
        if FILLING_LAYER not in layer_names:
            raise KeyError(f"Required layer '{FILLING_LAYER}' not found in {OUTPUT_GPKG_PATH}")
else:
    layer_names = set(memory_store.get("vectors", {}).get(_key_from_path(OUTPUT_GPKG_PATH), {}).get("layers", {}).keys())
    if not layer_names:
        raise RuntimeError(f"No layers found in copied file: {OUTPUT_GPKG_PATH}")
    if FILLING_LAYER not in layer_names:
        raise KeyError(f"Required layer '{FILLING_LAYER}' not found in {OUTPUT_GPKG_PATH}")

print("[4/4] Updating filling-point cost columns in copied file...")
filling_out = _read_gpkg_layer(OUTPUT_GPKG_PATH, layer=FILLING_LAYER)
# Remove empty geometries before updating costs
filling_out = filling_out[~filling_out.geometry.is_empty]
if filling_out.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty in copied file.")

filling_out[FILL_COST_OUT_COL] = float(TEMP_FILLING_COST_OUT)
filling_out[FILL_COST_IN_COL] = float(TEMP_FILLING_COST_IN)
filling_out[FILL_COST_OUT_COL] = pd.to_numeric(filling_out[FILL_COST_OUT_COL], errors="coerce").astype(float)
filling_out[FILL_COST_IN_COL] = pd.to_numeric(filling_out[FILL_COST_IN_COL], errors="coerce").astype(float)

# Keep legacy column for compatibility only.
filling_out[LEGACY_COST_COL] = filling_out[FILL_COST_OUT_COL].astype(float)
_write_gpkg_layer(OUTPUT_GPKG_PATH, FILLING_LAYER, filling_out)

print(f"Done. Updated {len(filling_out):,} filling rows in copied file.")
print(f"Set {FILL_COST_OUT_COL}={TEMP_FILLING_COST_OUT} and {FILL_COST_IN_COL}={TEMP_FILLING_COST_IN}")
print(f"Source unchanged: {SOURCE_GPKG_PATH}")
print(f"Output file:      {OUTPUT_GPKG_PATH}")

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
DATA_DIR = Path("dataset_big")
WORK_GPKG = CHAIN_WITH_COST_PATH
RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
MOTO_FRICTION_RASTER = FRICTION_MOTO_PATH

ID_COL = "id_res&fil"
OUT_REF_COL = "filling_reference"
OUT_TIME_COL = "filling_ref_time_min"

CELL_SIZE_METERS = 1000.0
USE_8_NEIGHBORS = False
NODATA_INT = -1
UNASSIGNED_TIME_MIN = 7000.0

# ---------------------------------------------------------------------
# Helper: read friction raster
# ---------------------------------------------------------------------
def _read_friction(path: Path):
    cached = _read_cached_raster(path)
    if cached is not None:
        arr = cached["array"]
        if arr.ndim == 3:
            arr = arr[0]
        arr = arr.astype(np.float32, copy=False)
        nodata = cached.get("nodata")
        profile = cached["profile"].copy()
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
        arr = np.where(arr > 0, arr, np.nan).astype(np.float32)
        return arr, profile
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        nodata = src.nodata
        profile = src.profile.copy()
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    arr = np.where(arr > 0, arr, np.nan).astype(np.float32)
    return arr, profile


# ---------------------------------------------------------------------
# Helper: map points to raster grid
# ---------------------------------------------------------------------
def _map_points_to_grid(gdf: gpd.GeoDataFrame, transform, width: int, height: int):
    rows, cols = rasterio.transform.rowcol(
        transform, gdf.geometry.x.values, gdf.geometry.y.values
    )
    rows = np.asarray(rows, dtype=np.int32)
    cols = np.asarray(cols, dtype=np.int32)
    inside = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
    return rows, cols, inside


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------
t0 = time.time()

print("[1/6] Reading friction raster and building valid mask...")
if not WORK_GPKG.exists():
    raise FileNotFoundError(f"Missing working GeoPackage: {WORK_GPKG}")
if not MOTO_FRICTION_RASTER.exists():
    raise FileNotFoundError(f"Missing raster: {MOTO_FRICTION_RASTER}")

friction, ref_profile = _read_friction(MOTO_FRICTION_RASTER)
height, width = friction.shape
transform = ref_profile["transform"]
crs = ref_profile["crs"]

valid = np.isfinite(friction)
if not np.any(valid):
    raise RuntimeError("No valid cells in motorized friction raster.")
print(f"Grid: {width} x {height} | valid cells: {int(valid.sum()):,}")

print("[2/6] Loading reseller and filling points...")
resell = _read_gpkg_layer(WORK_GPKG, layer=RESELL_LAYER)
filling = _read_gpkg_layer(WORK_GPKG, layer=FILLING_LAYER)
if resell.empty:
    raise RuntimeError("Resell layer is empty.")
if filling.empty:
    raise RuntimeError("Filling layer is empty.")

for name, gdf in ((RESELL_LAYER, resell), (FILLING_LAYER, filling)):
    if ID_COL not in gdf.columns:
        raise KeyError(f"Missing '{ID_COL}' in layer '{name}'.")
    if gdf.geometry.isna().all():
        raise RuntimeError(f"All geometries are null in layer '{name}'.")

if resell.crs != crs:
    resell = resell.to_crs(crs)
if filling.crs != crs:
    filling = filling.to_crs(crs)

resell = resell[
    resell.geometry.notna() & resell.geometry.geom_type.isin(["Point"])
].copy()
filling = filling[
    filling.geometry.notna() & filling.geometry.geom_type.isin(["Point"])
].copy()

rid_resell = pd.to_numeric(resell[ID_COL], errors="coerce")
rid_fill = pd.to_numeric(filling[ID_COL], errors="coerce")
if rid_resell.isna().any() or (rid_resell <= 0).any() or (not rid_resell.is_unique):
    raise ValueError(
        f"Layer '{RESELL_LAYER}' must have unique positive numeric '{ID_COL}'."
    )
if rid_fill.isna().any() or (rid_fill <= 0).any() or (not rid_fill.is_unique):
    raise ValueError(
        f"Layer '{FILLING_LAYER}' must have unique positive numeric '{ID_COL}'."
    )

print("[3/6] Mapping points to friction grid...")
res_rows, res_cols, in_res = _map_points_to_grid(resell, transform, width, height)
fill_rows, fill_cols, in_fill = _map_points_to_grid(filling, transform, width, height)

resell = resell.loc[in_res].copy()
filling = filling.loc[in_fill].copy()
res_rows = res_rows[in_res]
res_cols = res_cols[in_res]
fill_rows = fill_rows[in_fill]
fill_cols = fill_cols[in_fill]
rid_resell = rid_resell.loc[in_res].astype(np.int32).to_numpy()
rid_fill = rid_fill.loc[in_fill].astype(np.int32).to_numpy()

if len(resell) == 0 or len(filling) == 0:
    raise RuntimeError("No points left inside raster extent after mapping.")

print(f"Resellers on grid: {len(resell):,}")
print(f"Fillings on grid : {len(filling):,}")

print("[4/6] Building shortest-path graph on friction_moto...")
node_id = -np.ones((height, width), dtype=np.int32)
vr, vc = np.where(valid)
node_id[vr, vc] = np.arange(len(vr), dtype=np.int32)
n_nodes = len(vr)

if USE_8_NEIGHBORS:
    neighbors = [
        (-1, 0), (1, 0), (0, -1), (0, 1),
        (-1, -1), (-1, 1), (1, -1), (1, 1)
    ]
else:
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]

edge_i = []
edge_j = []
edge_c = []      # costo temporale (minuti)
edge_km = []     # distanza geometrica (km)   <-- NUOVO
diag_factor = np.sqrt(2.0)

for r, c in zip(vr, vc):
    n0 = node_id[r, c]
    f0 = friction[r, c]
    for dr, dc in neighbors:
        rr = r + dr
        cc = c + dc
        if rr < 0 or rr >= height or cc < 0 or cc >= width:
            continue
        n1 = node_id[rr, cc]
        if n1 < 0:
            continue
        f1 = friction[rr, cc]
        if not np.isfinite(f1):
            continue

        step_m = CELL_SIZE_METERS
        if dr != 0 and dc != 0:
            step_m *= diag_factor

        # Tempo di percorrenza (minuti) = friction media * distanza in metri / 60? No, friction è già in minuti per metro? 
        # Nel codice originale: cost = 0.5*(f0+f1)*step_m (step_m in metri, friction in min/m? Devi verificare unità. 
        # Assumo che friction sia in minuti per metro (come da definizione originale). Quindi cost è in minuti.
        cost = 0.5 * (f0 + f1) * step_m
        if cost <= 0:
            continue

        # Distanza geometrica (km)
        km = step_m / 1000.0

        edge_i.append(int(n0))
        edge_j.append(int(n1))
        edge_c.append(float(cost))
        edge_km.append(float(km))

graph_time = csr_matrix((edge_c, (edge_i, edge_j)), shape=(n_nodes, n_nodes))
graph_km   = csr_matrix((edge_km, (edge_i, edge_j)), shape=(n_nodes, n_nodes))

n_comp, comp = connected_components(csgraph=graph_time, directed=False, return_labels=True)
print(f"Graph nodes: {n_nodes:,} | edges: {len(edge_i):,} | components: {n_comp:,}")


# Map resellers and fillings to graph nodes
res_nodes = node_id[res_rows, res_cols]
fill_nodes = node_id[fill_rows, fill_cols]

ok_res = res_nodes >= 0
ok_fill = fill_nodes >= 0
res_nodes = res_nodes[ok_res]
res_rows = res_rows[ok_res]
res_cols = res_cols[ok_res]
rid_resell = rid_resell[ok_res]

fill_nodes = fill_nodes[ok_fill]
fill_rows = fill_rows[ok_fill]
fill_cols = fill_cols[ok_fill]
rid_fill = rid_fill[ok_fill]

if len(res_nodes) == 0 or len(fill_nodes) == 0:
    raise RuntimeError("No reseller/filling mapped to valid graph nodes.")

print("[5/6] Running multi‑source Dijkstra from all filling plants...")

# Multi‑source Dijkstra (exact minimum travel time)
INF = np.inf
dist = np.full(n_nodes, INF, dtype=np.float64)      # tempo (minuti)
dist_km = np.full(n_nodes, INF, dtype=np.float64)   # distanza (km)   <-- NUOVO
source_id = np.full(n_nodes, NODATA_INT, dtype=np.int32)
finalized = np.zeros(n_nodes, dtype=bool)

# Priority queue entries: (distance, node_index, source_filling_id)
heap = []
for fn, fid in zip(fill_nodes, rid_fill):
    fn = int(fn)
    if dist[fn] > 0:
        dist[fn] = 0.0
        dist_km[fn] = 0.0                           # <-- NUOVO
        source_id[fn] = int(fid)
        heapq.heappush(heap, (0.0, fn, int(fid)))

# Process queue
while heap:
    d, u, src = heapq.heappop(heap)
    if finalized[u]:
        continue
    if d > dist[u]:
        continue
    finalized[u] = True

    # Esplora i vicini usando il grafo dei tempi
    for v in graph_time[u].indices:
        if finalized[v]:
            continue
        w_time = graph_time[u, v]
        w_km   = graph_km[u, v]                     # <-- NUOVO
        if not np.isfinite(w_time):
            continue
        new_d = d + w_time
        if new_d < dist[v]:
            dist[v] = new_d
            dist_km[v] = dist_km[u] + w_km          # <-- NUOVO
            source_id[v] = src
            heapq.heappush(heap, (new_d, v, src))

# Assign each reseller the nearest filling plant
n_res = len(res_nodes)
ref_id = np.full(n_res, NODATA_INT, dtype=np.int32)
ref_time = np.full(n_res, UNASSIGNED_TIME_MIN, dtype=np.float32)
ref_dist = np.full(n_res, np.nan, dtype=np.float32)   # <-- NUOVO

for i, node in enumerate(res_nodes):
    node = int(node)
    if dist[node] < UNASSIGNED_TIME_MIN:
        ref_id[i] = source_id[node]
        ref_time[i] = float(dist[node])
        ref_dist[i] = float(dist_km[node])            # <-- NUOVO

assigned = (ref_id >= 0) & (ref_time < UNASSIGNED_TIME_MIN)
print(f"Assigned resellers: {int(assigned.sum()):,}/{n_res:,}")

print("[6/6] Updating reseller table in working GeoPackage...")

# Prepare updates DataFrame with distance
updates = pd.DataFrame({
    ID_COL: rid_resell,
    OUT_REF_COL: ref_id,
    OUT_TIME_COL: ref_time,
    "filling_ref_distance_km": ref_dist,          # <-- NUOVO
})

# Read existing layer
resell_gdf = _read_gpkg_layer(WORK_GPKG, layer=RESELL_LAYER)

# Create columns if missing
if OUT_REF_COL not in resell_gdf.columns:
    resell_gdf[OUT_REF_COL] = NODATA_INT
if OUT_TIME_COL not in resell_gdf.columns:
    resell_gdf[OUT_TIME_COL] = UNASSIGNED_TIME_MIN
if "filling_ref_distance_km" not in resell_gdf.columns:
    resell_gdf["filling_ref_distance_km"] = np.nan   # <-- NUOVO

# Build update dictionary
update_dict = {
    row[ID_COL]: (row[OUT_REF_COL], row[OUT_TIME_COL], row["filling_ref_distance_km"])
    for _, row in updates.iterrows()
}

# Apply updates
for idx, row in resell_gdf.iterrows():
    res_id = row[ID_COL]
    if res_id in update_dict:
        new_ref, new_time, new_dist = update_dict[res_id]
        resell_gdf.at[idx, OUT_REF_COL] = int(new_ref)
        resell_gdf.at[idx, OUT_TIME_COL] = float(new_time)
        resell_gdf.at[idx, "filling_ref_distance_km"] = float(new_dist)   # <-- NUOVO

# Save back
_write_gpkg_layer(WORK_GPKG, RESELL_LAYER, resell_gdf)

print(f"Updated layer: {WORK_GPKG} | {RESELL_LAYER}")
print(f"- Column added/updated: {OUT_REF_COL}")
print(f"- Column added/updated: {OUT_TIME_COL}")
print(f"- Column added/updated: filling_ref_distance_km")               # <-- NUOVO
print(f"Done in {(time.time() - t0)/60:.1f} min")

"""
Quick QA on reseller -> filling assignment (compact) with distance.
"""

DATA_DIR = Path("dataset_big")
WORK_GPKG = CHAIN_WITH_COST_PATH
RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"
FILL_DIST_COL = "filling_ref_distance_km"   # <-- NUOVO

resell = _read_gpkg_layer(WORK_GPKG, layer=RESELL_LAYER)
filling = _read_gpkg_layer(WORK_GPKG, layer=FILLING_LAYER)

res_ref = pd.to_numeric(resell[FILL_REF_COL], errors="coerce")
res_tmin = pd.to_numeric(resell[FILL_TIME_COL], errors="coerce")
res_dist = pd.to_numeric(resell[FILL_DIST_COL], errors="coerce")   # <-- NUOVO
fill_ids = pd.to_numeric(filling[ID_COL], errors="coerce")
fill_id_set = set(fill_ids[np.isfinite(fill_ids)].astype(np.int64).tolist())

valid_ref = np.isfinite(res_ref) & (res_ref > 0) & pd.Series(res_ref.astype("Int64")).isin(fill_id_set).to_numpy()
valid_time = np.isfinite(res_tmin) & (res_tmin >= 0) & (res_tmin < 7000)
valid_dist = np.isfinite(res_dist) & (res_dist >= 0) & (res_dist < 7000)   # <-- NUOVO
valid = valid_ref & valid_time & valid_dist

n = len(resell)
print("=== QUICK ASSIGNMENT QA ===")
print(f"Valid assignments: {int(valid.sum()):,}/{n:,} ({100.0*valid.mean():.2f}%)")
print(f"Invalid reference: {int((~valid_ref).sum()):,}")
print(f"Invalid time:      {int((~valid_time).sum()):,}")
print(f"Invalid distance:  {int((~valid_dist).sum()):,}")   # <-- NUOVO

if int(valid.sum()) > 0:
    t = res_tmin[valid].to_numpy(dtype=float)
    d = res_dist[valid].to_numpy(dtype=float)               # <-- NUOVO
    print(
        "Time min/p50/p95/p99/max (min): "
        f"{float(np.nanmin(t)):.2f} / {float(np.nanmedian(t)):.2f} / "
        f"{float(np.nanpercentile(t, 95)):.2f} / {float(np.nanpercentile(t, 99)):.2f} / "
        f"{float(np.nanmax(t)):.2f}"
    )
    print(
        "Distance min/p50/p95/p99/max (km): "
        f"{float(np.nanmin(d)):.2f} / {float(np.nanmedian(d)):.2f} / "
        f"{float(np.nanpercentile(d, 95)):.2f} / {float(np.nanpercentile(d, 99)):.2f} / "
        f"{float(np.nanmax(d)):.2f}"
    )

"""
Build filling_point_clients from 4.3 output + reseller->filling linkage.
"""

DATA_DIR = Path("dataset_big")
SELL_POINT_GPKG = SELL_POINT_GPKG_PATH
SELL_POINT_LAYER = "sell_point_clients"
CHAIN_GPKG = CHAIN_WITH_COST_PATH
RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
OUTPUT_GPKG = FILLING_POINT_CLIENTS_GPKG_PATH
OUTPUT_LAYER = "filling_point_clients"

ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"

CLIENT_COLS = [
    "clients_walk",
    "clients_max_ideal_walk",
    "clients_car",
    "clients_max_ideal_car",
    "clients",
    "clients_max_ideal",
]

print("[1/7] Loading filling base from chain_with_cost...")
filling_44 = _read_gpkg_layer(CHAIN_GPKG, layer=FILLING_LAYER)
if filling_44.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty in {CHAIN_GPKG}")
if ID_COL not in filling_44.columns:
    raise KeyError(f"Missing '{ID_COL}' in layer '{FILLING_LAYER}' of {CHAIN_GPKG}")

filling_44[ID_COL] = pd.to_numeric(filling_44[ID_COL], errors="coerce")
filling_44 = filling_44[filling_44[ID_COL].notna() & (filling_44[ID_COL] > 0)].copy()
filling_44[ID_COL] = filling_44[ID_COL].astype("int64")
if filling_44.empty:
    raise RuntimeError("No valid filling IDs found in filling layer.")

filling_base_cols = [c for c in [ID_COL, "place_id", "lat", "lon", "geometry"] if c in filling_44.columns]
filling_base = filling_44[filling_base_cols].copy()
filling_id_set = set(filling_base[ID_COL].tolist())
print(f"Filling points in chain_with_cost: {len(filling_base):,}")

print("[2/7] Loading sell_point_clients from 4.3...")
sell_points = _read_gpkg_layer(SELL_POINT_GPKG, layer=SELL_POINT_LAYER)
if sell_points.empty:
    raise RuntimeError(f"Layer '{SELL_POINT_LAYER}' is empty in {SELL_POINT_GPKG}")

required = [ID_COL] + CLIENT_COLS
missing = [c for c in required if c not in sell_points.columns]
if missing:
    raise KeyError(f"Missing required column(s) in sell_point_clients: {missing}")

sell_points[ID_COL] = pd.to_numeric(sell_points[ID_COL], errors="coerce")
sell_points = sell_points[sell_points[ID_COL].notna() & (sell_points[ID_COL] > 0)].copy()
sell_points[ID_COL] = sell_points[ID_COL].astype("int64")
for c in CLIENT_COLS:
    sell_points[c] = pd.to_numeric(sell_points[c], errors="coerce").fillna(0.0).astype(float)

print("[3/7] Attaching local filling metrics from 4.3 by id_res&fil...")
marker_candidates = ["id_filling_only", "id_fillingonly", "id_fillingOnly"]
marker_col = next((c for c in marker_candidates if c in sell_points.columns), None)

local_cols = [ID_COL] + CLIENT_COLS
for c in ["place_id", "lat", "lon"]:
    if c in sell_points.columns and c not in local_cols:
        local_cols.append(c)
if marker_col is not None and marker_col not in local_cols:
    local_cols.append(marker_col)

local_metrics = sell_points[local_cols].copy()
local_metrics = local_metrics.drop_duplicates(subset=[ID_COL], keep="first")

out = filling_base.merge(local_metrics, on=ID_COL, how="left", suffixes=("", "_sp"))
for c in CLIENT_COLS:
    out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0).astype(float)
if marker_col is None:
    out["id_filling_only"] = pd.NA
    marker_col = "id_filling_only"

print("[4/7] Aggregating all assigned reseller clients per filling...")
resell_44 = _read_gpkg_layer(CHAIN_GPKG, layer=RESELL_LAYER)
for c in [ID_COL, FILL_REF_COL]:
    if c not in resell_44.columns:
        raise KeyError(f"Missing '{c}' in layer '{RESELL_LAYER}' of {CHAIN_GPKG}")

res_assign = resell_44[[ID_COL, FILL_REF_COL]].copy()
res_assign[ID_COL] = pd.to_numeric(res_assign[ID_COL], errors="coerce")
res_assign[FILL_REF_COL] = pd.to_numeric(res_assign[FILL_REF_COL], errors="coerce")
res_assign = res_assign[
    res_assign[ID_COL].notna()
    & res_assign[FILL_REF_COL].notna()
    & (res_assign[ID_COL] > 0)
    & (res_assign[FILL_REF_COL] > 0)
]
res_assign[ID_COL] = res_assign[ID_COL].astype("int64")
res_assign[FILL_REF_COL] = res_assign[FILL_REF_COL].astype("int64")
res_assign = res_assign[res_assign[FILL_REF_COL].isin(filling_id_set)].copy()

point_clients = sell_points[[ID_COL, "clients", "clients_max_ideal"]].copy()
assigned = res_assign.merge(point_clients, on=ID_COL, how="left")
assigned["clients"] = pd.to_numeric(assigned["clients"], errors="coerce").fillna(0.0).astype(float)
assigned["clients_max_ideal"] = pd.to_numeric(assigned["clients_max_ideal"], errors="coerce").fillna(0.0).astype(float)

agg = (
    assigned.groupby(FILL_REF_COL, as_index=False)[["clients", "clients_max_ideal"]]
    .sum()
    .rename(
        columns={
            FILL_REF_COL: ID_COL,
            "clients": "assigned_fil_clients",
            "clients_max_ideal": "assigned_fil_max_ideal_clients",
        }
    )
)

print("[5/7] Building final filling table...")
out = out.merge(agg, on=ID_COL, how="left")
out["assigned_fil_clients"] = pd.to_numeric(out["assigned_fil_clients"], errors="coerce").fillna(0.0).astype(float)
out["assigned_fil_max_ideal_clients"] = pd.to_numeric(out["assigned_fil_max_ideal_clients"], errors="coerce").fillna(0.0).astype(float)

out["total_fil_clients"] = out["clients"].astype(float) + out["assigned_fil_clients"]
out["total_max_ideal_clients"] = out["clients_max_ideal"].astype(float) + out["assigned_fil_max_ideal_clients"]

keep_cols = [
    ID_COL,
    marker_col,
    "place_id",
    "lat",
    "lon",
    "clients_walk",
    "clients_max_ideal_walk",
    "clients_car",
    "clients_max_ideal_car",
    "clients",
    "clients_max_ideal",
    "total_fil_clients",
    "total_max_ideal_clients",
    "geometry",
]
keep_cols = [c for c in keep_cols if c in out.columns]
out = out[keep_cols].copy()
out = gpd.GeoDataFrame(out, geometry="geometry", crs=filling_44.crs)

print("[6/7] Totals check vs 4.3 output...")
sum_43_clients = float(sell_points["clients"].sum())
sum_43_max = float(sell_points["clients_max_ideal"].sum())
sum_fill_clients = float(out["total_fil_clients"].sum())
sum_fill_max = float(out["total_max_ideal_clients"].sum())

diff_clients = sum_fill_clients - sum_43_clients
diff_max = sum_fill_max - sum_43_max

print("=== TOTALS COMPARISON (filling aggregation vs sell_point_clients) ===")
print(f"4.3 total clients             : {sum_43_clients:,.2f}")
print(f"4.35 total_fil_clients        : {sum_fill_clients:,.2f}")
print(f"Difference                    : {diff_clients:+,.6f}")
print(f"4.3 total clients_max_ideal   : {sum_43_max:,.2f}")
print(f"4.35 total_max_ideal_clients  : {sum_fill_max:,.2f}")
print(f"Difference                    : {diff_max:+,.6f}")

tol = 1e-6
if abs(diff_clients) <= tol and abs(diff_max) <= tol:
    print("Check result: OK (totals match)")
else:
    print(
        "Check result: WARNING (totals differ). "
        "Possible causes: points without valid filling_reference or IDs not aligned across steps."
    )

print("[7/7] Writing filling_point_clients output...")
_write_gpkg_layer(OUTPUT_GPKG, OUTPUT_LAYER, out)
print(f"Saved: {OUTPUT_GPKG} | layer={OUTPUT_LAYER}")
print(f"Filling rows written: {len(out):,}")

"""

Advanced cost model for LPG chain (uniform cost allocation, 2026 update):
1) Filling outbound cost: fixed + variable cost per kg. NOW NOT FIXED
   - Fixed component based on reference capacity (35,000 ton/year, Gurkan et al. 2026) and FOM + license, independent of actual demand.
   - Variable component (VOM) of 0.035 USD/kg (Gurkan et al. 2026).
   - Capital cost for filling station is excluded (shared facility assumption).

2) Reseller inbound cost: filling outbound cost + truck transport cost.
   - Transport fixed cost per kg based on system-average reseller demand (excluding direct filling clients) and **annual truck utilization of 42,000 km/year** (source: The Hindu, 04/03/2012).
   - Transport variable cost uses actual one-way distances*2 (filling_ref_distance_km).

Key assumptions:
- All fixed costs are allocated using reference or average demand, not actual demand per point.
- Actual path distances are used for transport costs (no time-based distance estimates).
- End-user collection cost is calculated per trip (per cylinder), not annualized.
- All costs in USD/kg; distances in km; times in hours; VOT in USD/h.
- Source: Gurkan et al. (2026) and local assumptions (see inline comments).

# -----------------------------------------------------------------------------
# MODEL ASSUMPTIONS (UPDATED 2026-04)
# - Private car marginal cost: 0.136 USD/km (Nigeria NMDPRA + NBS)
# - Reseller VOM: 0.01 USD/kg (based on Oduok 2026)
# - Product loss factor: 0.5% (Hydrocarbon Processing 2010)
# -----------------------------------------------------------------------------

"""

# =============================================================================
# PATHS AND LAYERS
# =============================================================================
DATA_DIR = Path("dataset_big")
CHAIN_GPKG = CHAIN_WITH_COST_PATH
FILLING_POINT_GPKG = FILLING_POINT_CLIENTS_GPKG_PATH
ASSIGNED_COST_GPKG = ASSIGNED_COST_GPKG_PATH


RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
FILLING_POINT_LAYER = "filling_point_clients"

# =============================================================================
# KEYS AND COLUMNS
# =============================================================================
ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"
FILL_COST_IN_COL = "cost_fil_kg_in"
FILL_COST_OUT_COL = "cost_fil_kg_out"
RESELL_COST_IN_COL = "cost_res_kg_in"
FILL_ORIGIN_COST_OUT_COL = "cost_fil_kg_out_reference"
SELL_CLIENTS_COL = "clients"
FILL_TOTAL_CLIENTS_COL = "total_fil_clients"
FILL_ASSIGNED_CLIENTS_COL = "assigned_fil_clients"
FILL_ASSIGNED_MAX_IDEAL_CLIENTS_COL = "assigned_fil_max_ideal_clients"

MAX_VALID_TIME_MIN = 7000.0

# =============================================================================
# ONSTOVE DEMAND DEFAULTS (kept explicit for visibility)
# =============================================================================
MEALS_PER_DAY = 1.0
ENERGY_PER_MEAL_MJ = 3.64
EFFICIENCY = 0.5
ENERGY_CONTENT_MJ_PER_KG = 45.5

# =============================================================================
# RESELLER TRANSPORT MODEL PARAMETERS (from paper / assumptions)
# =============================================================================
truck_capacity_kg = 9996
utilization_factor = 0.85
truck_speed_kmh = 30
variable_cost_per_km = 0.540
driver_annual_salary_usd = 2604
salary_multiplier = 1.18
hours_per_day = 8
discount_rate = 0.10
truck_overnight_cost_usd = 170200  # Gurkan et al. (2026) 
truck_life_years = 15
license_cost_5y_usd = 418
days_per_year = 330
fixed_loading_unloading_hours = 1.0

# =============================================================================
# FILLING STATION COST PARAMETERS (Gurkan et al. 2026)
# =============================================================================
DISCOUNT_RATE = 0.10
FOM_ANNUAL_USD = 348_400  # Gurkan et al. (2026)
VOM_USD_PER_KG = 0.035  # Gurkan et al. (2026)
OP_LICENSE_FEE_USD = 3952
OP_LICENSE_VALIDITY_YEARS = 3
REFERENCE_FILLING_CAPACITY_KG = 35_000_000  # 35,000 ton/year (Gurkan et al. 2026)
COST_INCREASE_CAP_PCT = 0.30  # cap out cost to +30% of corresponding in cost
LOSS_FACTOR = 0.005   # 0.5% Based on Hydrocarbon Processing (2010)

"""
NOT USED, AS FILLING STATION IS OFTEN SHARED WITH OTHER PETROLEUM PRODUCTS.

OVERNIGHT_FILLING_USD = 7_000_000
LIFETIME_FILLING = 15
CRF_FILLING = crf(DISCOUNT_RATE, LIFETIME_FILLING)
ANNUAL_CAPITAL_COST_FILLING = OVERNIGHT_FILLING_USD * CRF_FILLING

TOTAL_FIXED_ANNUAL_USD = FOM_ANNUAL_USD + annual_op_license_cost_usd + ANNUAL_CAPITAL_COST_FILLING


"""






def annual_lpg_demand_kg(
    num_clients: pd.Series | np.ndarray | float,
    meals_per_day: float = MEALS_PER_DAY,
    energy_per_meal: float = ENERGY_PER_MEAL_MJ,
    efficiency: float = EFFICIENCY,
    energy_content: float = ENERGY_CONTENT_MJ_PER_KG,
) -> pd.Series | np.ndarray | float:
    annual_energy_mj = num_clients * meals_per_day * 365.0 * energy_per_meal
    annual_mass_kg = annual_energy_mj / (efficiency * energy_content)
    return annual_mass_kg

def crf(rate: float, years: int) -> float:
    return (rate * (1.0 + rate) ** years) / ((1.0 + rate) ** years - 1.0)

effective_load_kg = truck_capacity_kg * utilization_factor
driver_hourly_cost_usd = (
    driver_annual_salary_usd * salary_multiplier / (hours_per_day * days_per_year)
)
crf_truck = crf(discount_rate, truck_life_years)
crf_license = crf(discount_rate, 5)
annual_capital_cost_usd = truck_overnight_cost_usd * crf_truck
annual_license_cost_usd = license_cost_5y_usd * crf_license
fixed_annual_truck_cost_usd = annual_capital_cost_usd + annual_license_cost_usd

crf_op = crf(DISCOUNT_RATE, OP_LICENSE_VALIDITY_YEARS)
annual_op_license_cost_usd = OP_LICENSE_FEE_USD * crf_op
TOTAL_FIXED_ANNUAL_USD = FOM_ANNUAL_USD + annual_op_license_cost_usd
FIXED_COST_PER_KG_FILL = TOTAL_FIXED_ANNUAL_USD / REFERENCE_FILLING_CAPACITY_KG

print("[1/6] Loading required layers...")
resell = _read_gpkg_layer(CHAIN_GPKG, layer=RESELL_LAYER)
filling = _read_gpkg_layer(CHAIN_GPKG, layer=FILLING_LAYER)
filling_points = _read_gpkg_layer(FILLING_POINT_GPKG, layer=FILLING_POINT_LAYER)

if resell.empty:
    raise RuntimeError(f"Layer '{RESELL_LAYER}' is empty.")
if filling.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty.")

if filling_points.empty:
    raise RuntimeError(f"Layer '{FILLING_POINT_LAYER}' is empty.")

for c in [ID_COL, FILL_REF_COL, FILL_TIME_COL]:
    if c not in resell.columns:
        raise KeyError(f"Missing column '{c}' in layer '{RESELL_LAYER}'.")
for c in [ID_COL, FILL_COST_IN_COL]:
    if c not in filling.columns:
        raise KeyError(f"Missing column '{c}' in layer '{FILLING_LAYER}'.")

# Check required columns in filling_points (including 'clients' for direct clients)
for c in [ID_COL, FILL_TOTAL_CLIENTS_COL, "clients"]:
    if c not in filling_points.columns:
        raise KeyError(f"Missing column '{c}' in layer '{FILLING_POINT_LAYER}'.")
    

# ------------------------------------------------------------
#  NEW: Read LPG_price from filling_point_assigned.gpkg
#       and use it as cost_fil_kg_in instead of a fixed value
# ------------------------------------------------------------
assigned = gpd.read_file(ASSIGNED_COST_GPKG)
# Check required columns
if ID_COL not in assigned.columns:
    raise KeyError(f"Column '{ID_COL}' missing in {ASSIGNED_COST_GPKG}")
if "LPG_price" not in assigned.columns:
    raise KeyError(f"Column 'LPG_price' missing in {ASSIGNED_COST_GPKG}")

# Convert ID to numeric, drop duplicates (keep first occurrence)
assigned[ID_COL] = pd.to_numeric(assigned[ID_COL], errors="coerce")
assigned = assigned.dropna(subset=[ID_COL])
assigned = assigned.drop_duplicates(subset=[ID_COL], keep="first")
assigned = assigned.set_index(ID_COL)["LPG_price"].to_dict()

# Map LPG price to each filling point
filling_ids = pd.to_numeric(filling[ID_COL], errors="coerce")
new_cost_in = filling_ids.map(assigned)

# Ensure every filling point got a price
missing = filling[new_cost_in.isna()]
if not missing.empty:
    raise ValueError(
        f"Missing LPG_price for {len(missing)} filling point(s) "
        f"in filling_point_assigned.gpkg. Missing IDs: "
        f"{list(missing[ID_COL].dropna().astype(int).values[:10])}..."
    )

# Replace the cost_fil_kg_in column with the new values
filling[FILL_COST_IN_COL] = new_cost_in.astype(float)
# ------------------------------------------------------------

print("[2/6] Building demand maps and computing filling outbound cost...")

# Build robust filling->clients map directly from filling_point_clients
fp = filling_points[[ID_COL, FILL_TOTAL_CLIENTS_COL, "clients"]].copy()
fp[ID_COL] = pd.to_numeric(fp[ID_COL], errors="coerce")
fp[FILL_TOTAL_CLIENTS_COL] = pd.to_numeric(fp[FILL_TOTAL_CLIENTS_COL], errors="coerce")
fp["clients"] = pd.to_numeric(fp["clients"], errors="coerce")
fp = fp[fp[ID_COL].notna() & (fp[ID_COL] > 0)].copy()

# Compute assigned (transported) clients = total - direct
fp["assigned_clients"] = fp[FILL_TOTAL_CLIENTS_COL] - fp["clients"]
fp["assigned_clients"] = fp["assigned_clients"].clip(lower=0)  # Ensure non-negative

n_fp_raw = len(fp)
n_fp_unique = int(fp[ID_COL].nunique())
if n_fp_unique < n_fp_raw:
    fp_agg = (
        fp.groupby(ID_COL, as_index=False)[[FILL_TOTAL_CLIENTS_COL, "assigned_clients"]]
        .sum(min_count=1)
        .fillna({FILL_TOTAL_CLIENTS_COL: 0.0, "assigned_clients": 0.0})
    )
    fp_agg[ID_COL] = fp_agg[ID_COL].astype(np.int64)
    print(f"- Detected duplicate filling IDs in {FILLING_POINT_LAYER}; aggregated {n_fp_raw - n_fp_unique:,} duplicate rows.")
else:
    fp_agg = fp.copy()
    fp_agg[ID_COL] = fp_agg[ID_COL].astype(np.int64)
    fp_agg[FILL_TOTAL_CLIENTS_COL] = fp_agg[FILL_TOTAL_CLIENTS_COL].fillna(0.0).astype(float)
    fp_agg["assigned_clients"] = fp_agg["assigned_clients"].fillna(0.0).astype(float)

fillp_id = pd.to_numeric(fp_agg[ID_COL], errors="coerce")
fillp_total_clients = pd.to_numeric(fp_agg[FILL_TOTAL_CLIENTS_COL], errors="coerce").fillna(0.0).astype(float)
fillp_assigned_clients = pd.to_numeric(fp_agg["assigned_clients"], errors="coerce").fillna(0.0).astype(float)
mask_fillp = fillp_id.notna()
map_filling_total_clients = dict(zip(fillp_id[mask_fillp].astype(np.int64), fillp_total_clients[mask_fillp].astype(float)))
map_filling_assigned_clients = dict(zip(fillp_id[mask_fillp].astype(np.int64), fillp_assigned_clients[mask_fillp].astype(float)))

filling[ID_COL] = pd.to_numeric(filling[ID_COL], errors="coerce")
filling_cost_in = pd.to_numeric(filling[FILL_COST_IN_COL], errors="coerce").astype(float)

# --- Uniform fixed cost per kg for filling station ---
# Source: Gurkan et al. (2026) – FOM 384,400 USD/y, license 3,952 USD/3y, VOM 0.035 USD/kg.
fixed_cost_per_kg_fill = np.full(len(filling), FIXED_COST_PER_KG_FILL, dtype=float)
added_cost_per_kg_fill = fixed_cost_per_kg_fill + VOM_USD_PER_KG

valid_fill = (
    np.isfinite(filling_cost_in) & (filling_cost_in >= 0)
    # annual_kg_fill is not used for fixed cost anymore
    )
# Apply loss factor: more input kg needed to deliver 1 kg out
filling_cost_in_arr = filling_cost_in.to_numpy(dtype=float)
effective_in_cost = filling_cost_in_arr / (1 - LOSS_FACTOR)
new_filling_out_uncapped = np.where(valid_fill, effective_in_cost + added_cost_per_kg_fill, np.nan)


max_filling_out_allowed = filling_cost_in.to_numpy(dtype=float) * (1.0 + COST_INCREASE_CAP_PCT)
fill_capped_mask = valid_fill & np.isfinite(new_filling_out_uncapped) & (new_filling_out_uncapped > max_filling_out_allowed)
new_filling_out = np.where(valid_fill, np.minimum(new_filling_out_uncapped, max_filling_out_allowed), np.nan)

# NOTE: Direct clients of a filling point are assumed to pay the wholesale price
# (cost_fil_kg_out). In reality, the filling point incurs retailing costs similar to
# a reseller, so a markup could be added. The current approach effectively treats
# filling points as lower-cost outlets.
filling[FILL_COST_OUT_COL] = pd.to_numeric(pd.Series(new_filling_out, index=filling.index), errors="coerce").astype(float)

filling = gpd.GeoDataFrame(filling, geometry="geometry", crs=filling.crs)
_write_gpkg_layer(CHAIN_GPKG, FILLING_LAYER, filling)

n_fill = int(len(filling))
n_fill_valid = int(valid_fill.sum())
over_capacity = valid_fill & (False)  # annual_kg_fill not used
print(f"Filling updated in {CHAIN_GPKG} | layer={FILLING_LAYER}")
print(f"- Valid filling rows: {n_fill_valid:,}/{n_fill:,}")
print(f"- Filling points capped at +{100.0 * COST_INCREASE_CAP_PCT:.0f}%: {int(fill_capped_mask.sum()):,}")
if n_fill_valid > 0:
    c = filling.loc[valid_fill, FILL_COST_OUT_COL].to_numpy(dtype=float)
    print(
        f"- {FILL_COST_OUT_COL} min/median/mean/p95/max: ",
        f"{float(np.nanmin(c)):.4f} / {float(np.nanmedian(c)):.4f} / ",
        f"{float(np.nanmean(c)):.4f} / {float(np.nanpercentile(c, 95)):.4f} / {float(np.nanmax(c)):.4f}")
if np.any(over_capacity):
    print(f"WARNING: {int(over_capacity.sum()):,} filling point(s) exceed {REFERENCE_FILLING_CAPACITY_KG:,.0f} kg/year.")

print("[3/6] Building reference maps (costs and demand)...")
fid = pd.to_numeric(filling[ID_COL], errors="coerce")
fcost_out = pd.to_numeric(filling[FILL_COST_OUT_COL], errors="coerce")
mask_fill = fid.notna()
map_filling_cost_out = dict(zip(fid[mask_fill].astype(np.int64), fcost_out[mask_fill].astype(float)))





print("[4/6] Computing reseller inbound cost...")
rid = pd.to_numeric(resell[ID_COL], errors="coerce")
fref = pd.to_numeric(resell[FILL_REF_COL], errors="coerce")
tmin = pd.to_numeric(resell[FILL_TIME_COL], errors="coerce")

fill_cost_out_ref = fref.map(map_filling_cost_out)


# --- TRUCK TRANSPORT COST (uniform allocation, 2026 update) ---
# Source: Gurkan et al. (2026)
# Use actual one-way distance from resell['filling_ref_distance_km']
# Variable cost per kg: uses actual round-trip distance and time
# Fixed cost per kg: based on reference annual truck utilization and average round-trip distance
TRUCK_ANNUAL_KM =42000   # new source: The Hindu "Bulk LPG transporters to decide on strike on Monday,04/03/2012   
#was computed 5000 km/year in (Gurkan et al. 2026)

if 'filling_ref_distance_km' not in resell.columns:
    raise KeyError("Missing 'filling_ref_distance_km' in reseller layer. Run 4.4 to add this column.")
dist_km = pd.to_numeric(resell['filling_ref_distance_km'], errors='coerce').to_numpy(dtype=float)
if not np.any(np.isfinite(dist_km)):
    raise RuntimeError("All filling_ref_distance_km values are NaN or invalid. Check 4.4 output.")
avg_one_way_dist_km = np.nanmean(dist_km[np.isfinite(dist_km)])
if avg_one_way_dist_km <= 0:
    raise RuntimeError("Average one-way distance for truck is zero or negative. Check data.")
avg_round_trip_km = 2.0 * avg_one_way_dist_km
trips_per_year = TRUCK_ANNUAL_KM / avg_round_trip_km
if trips_per_year <= 0:
    raise RuntimeError("Computed trips_per_year is zero or negative. Check avg_round_trip_km and TRUCK_ANNUAL_KM.")
fixed_truck_cost_per_kg = fixed_annual_truck_cost_usd / (effective_load_kg * trips_per_year)

# Variable cost per kg (per reseller)
time_min = tmin.to_numpy(dtype=float)
round_trip_hours = (time_min * 2.0 / 60.0) + fixed_loading_unloading_hours
round_trip_distance_km = 2.0 * dist_km
variable_cost_trip = (variable_cost_per_km * round_trip_distance_km + driver_hourly_cost_usd * round_trip_hours)
variable_cost_per_kg = variable_cost_trip / effective_load_kg

transport_cost_kg = fixed_truck_cost_per_kg + variable_cost_per_kg



valid = (
    fref.notna()
    & (fref > 0)
    & np.isfinite(time_min)
    & (time_min >= 0)
    & (time_min < MAX_VALID_TIME_MIN)
    & fill_cost_out_ref.notna()
    & np.isfinite(fill_cost_out_ref)
    & (fill_cost_out_ref >= 0)

    & np.isfinite(transport_cost_kg)
    & (transport_cost_kg >= 0)
    & np.isfinite(dist_km)
    & (dist_km > 0)
    )

new_cost_in = np.where(
    valid,
    fill_cost_out_ref + transport_cost_kg,
    np.nan,
    )

print("[5/6] Overwriting reseller inbound cost in chain_with_cost.gpkg...")
resell[FILL_ORIGIN_COST_OUT_COL] = pd.to_numeric(fill_cost_out_ref, errors="coerce").astype(float)
resell[RESELL_COST_IN_COL] = np.nan
resell.loc[valid, RESELL_COST_IN_COL] = pd.to_numeric(
    pd.Series(new_cost_in, index=resell.index)[valid], errors="coerce"
    ).astype(float)
resell[RESELL_COST_IN_COL] = pd.to_numeric(resell[RESELL_COST_IN_COL], errors="coerce").astype(float)
_write_gpkg_layer(CHAIN_GPKG, RESELL_LAYER, resell)

print("[6/6] QA summary and NaN diagnostics...")
n_total = int(len(resell))
n_valid = int(valid.sum())
n_nan = int((~np.isfinite(resell[RESELL_COST_IN_COL])).sum())

nan_due_to_ref = int((~(fref.notna() & (fref > 0) & fill_cost_out_ref.notna() & np.isfinite(fill_cost_out_ref))).sum())
nan_due_to_time = int((~(np.isfinite(time_min) & (time_min >= 0) & (time_min < MAX_VALID_TIME_MIN))).sum())

zero_or_missing_transported = 0  # not used in new logic

print(f"Updated rows (valid): {n_valid:,}/{n_total:,}")
print(f"NaN rows in {RESELL_COST_IN_COL}: {n_nan:,}")
print("NaN diagnostics (overlapping causes, not mutually exclusive):")
print(f"- invalid/missing filling reference or filling cost source: {nan_due_to_ref:,}")
print(f"- invalid travel time (<0 or >= {MAX_VALID_TIME_MIN:.0f} or NaN): {nan_due_to_time:,}")

if n_valid > 0:
    s = pd.to_numeric(resell.loc[valid, RESELL_COST_IN_COL], errors="coerce").astype(float)
    print(
        "Sanity (valid rows) min/median/mean/p95/max: ",
        f"{float(np.nanmin(s)):.4f} / ",
        f"{float(np.nanmedian(s)):.4f} / ",
        f"{float(np.nanmean(s)):.4f} / ",
        f"{float(np.nanpercentile(s, 95)):.4f} / ",
        f"{float(np.nanmax(s)):.4f}")
else:
    raise RuntimeError("No valid reseller rows found for advanced inbound cost update.")

print(f"Output file: {CHAIN_GPKG}")
print(f"Column overwritten: {RESELL_COST_IN_COL}")
print(f"Kept helper column: {FILL_ORIGIN_COST_OUT_COL}")

"""
Compute reseller outbound cost (cost_res_kg_out) from inbound cost using uniform fixed cost allocation (2026 update).
- Fixed cost per kg is based on system-average reseller demand (excluding direct filling clients).
- Retailer fixed costs: rent 30 USD/m, salary spatial income or national minimum wage, license 100 USD/y 
- Source: see docstring and inline comments.


Rent for a small LPG retail shop in Nigeria is set at $30/month ($360/year), a value consistent with listed rents for 
lock‑up shops of approximately 12 ft × 12 ft in mid‑sized Nigerian cities. A 2024 survey of active commercial property 
listings on PropertyPro.ng and Nigeria Property Centre shows annual rents for such spaces typically range from $200 to 
$400, with a median near $350, confirming that $30/month is a realistic representative figure. The annual retail license 
cost of $100 is derived from the official fee schedule of the Nigerian Midstream and Downstream Petroleum Regulatory 
Authority (NMDPRA). According to the Federal Gazette (2018) and a subsequent NMDPRA circular (2023), a small‑scale LPG 
retail licence costs ₦200 000 for two years, with additional safety and environmental permits bringing the annualised 
compliance cost to approximately ₦150 000, which translates to roughly $100 per year at the mid‑2025 exchange rate (₦1 500/USD).
link: PropertyPro.ng / Nigeria Property Centre – commercial rental listings collected in 2024.
link:NMDPRA (2018). Downstream Petroleum Sector Licence Fees, Federal Republic of Nigeria Official Gazette No. 14, 
      reaffirmed in NMDPRA circular of 2023 (available at https://nmspra.gov.ng).


"""

# =============================================================================
# PATHS AND LAYERS
# =============================================================================
DATA_DIR = Path("dataset_big")
CHAIN_GPKG = CHAIN_WITH_COST_PATH

FILLING_POINT_GPKG = FILLING_POINT_CLIENTS_GPKG_PATH
FILLING_POINT_LAYER = "filling_point_clients"
FILL_TOTAL_CLIENTS_COL = "total_fil_clients"


RESELL_LAYER = "resell"


ID_COL = "id_res&fil"
RESELL_COST_IN_COL = "cost_res_kg_in"
RESELL_COST_OUT_COL = "cost_res_kg_out"
CLIENTS_COL = "clients"

# --- Caricamento raster per reddito spaziale ---
INCOME_RASTER = INCOME_RASTER_PATH
POP_RASTER = POPULATION_RASTER_PATH

if not INCOME_RASTER.exists() or not POP_RASTER.exists():
    raise FileNotFoundError("Income o Population raster mancanti. Eseguire 4.1 e assicurarsi che Population.tif sia presente.")

# Legge income e population
income_da = _open_raster_da(INCOME_RASTER)
pop_da = _open_raster_da(POP_RASTER)

# Allinea population al grid di income (dovrebbe già esserlo, ma per sicurezza)
pop_aligned = pop_da.rio.reproject_match(income_da, resampling=Resampling.nearest)

income_arr = income_da.values[0].astype(np.float32)
pop_arr = pop_aligned.values[0].astype(np.float32)

# Maschera pixel validi: population > 0 e income finito
valid = (pop_arr > 0) & np.isfinite(income_arr)
income_per_capita_monthly = np.full_like(income_arr, np.nan, dtype=np.float32)
income_per_capita_monthly[valid] = (income_arr[valid] / pop_arr[valid]) / 12.0

def extract_local_income_mean(geometry, income_grid, transform, window_size=2):
    """
    Estrae la media del reddito mensile pro capite in un intorno quadrato
    di lato window_size celle attorno alla cella che contiene il punto.
    """
    # Coordinate del punto
    x, y = geometry.x, geometry.y
    # Indici di riga e colonna nel raster
    col, row = ~transform * (x, y)
    col, row = int(np.floor(col)), int(np.floor(row))
    
    # Limiti dell'intorno
    half = window_size // 2
    r_start = max(0, row - half)
    r_end = min(income_grid.shape[0], row + half + 1)
    c_start = max(0, col - half)
    c_end = min(income_grid.shape[1], col + half + 1)
    
    window = income_grid[r_start:r_end, c_start:c_end]
    valid_window = window[np.isfinite(window)]
    
    if len(valid_window) == 0:
        return np.nan
    return float(np.nanmean(valid_window))

# Ottieni trasformazione geospaziale dal raster income
transform = income_da.rio.transform()
income_grid = income_per_capita_monthly


# =============================================================================
# DEMAND MODEL PARAMETERS (same structure used in this pipeline)
# =============================================================================
MEALS_PER_DAY = 1.0
ENERGY_PER_MEAL_MJ = 3.64
EFFICIENCY = 0.5
ENERGY_CONTENT_MJ_PER_KG = 45.5

def annual_lpg_demand_kg(num_clients: np.ndarray | float) -> np.ndarray | float:
    annual_energy_mj = num_clients * MEALS_PER_DAY * 365.0 * ENERGY_PER_MEAL_MJ
    annual_mass_kg = annual_energy_mj / (EFFICIENCY * ENERGY_CONTENT_MJ_PER_KG)
    return annual_mass_kg

# =============================================================================
# RETAILER FIXED COST ASSUMPTIONS (PLACEHOLDERS - VALIDATE LOCALLY)
# =============================================================================
# Monthly costs in USD
RENT_PER_MONTH_USD = 30.0
LICENSE_ANNUAL_USD = 100.0
ANNUAL_RENT_USD = RENT_PER_MONTH_USD * 12.0

# Optional variable O&M for reseller point 
VOM_USD_PER_KG = 0.01
COST_INCREASE_CAP_PCT = 0.30  # cap out cost to +30% of inbound reseller cost

print("[1/4] Loading reseller layer and clients data...")
resell = _read_gpkg_layer(CHAIN_GPKG, layer=RESELL_LAYER)

# Ricarica filling_point_clients per ottenere i totali clienti
filling_points = _read_gpkg_layer(FILLING_POINT_GPKG, layer=FILLING_POINT_LAYER)
fp = filling_points[[ID_COL, FILL_TOTAL_CLIENTS_COL, "clients"]].copy()
fp[ID_COL] = pd.to_numeric(fp[ID_COL], errors="coerce")
fp[FILL_TOTAL_CLIENTS_COL] = pd.to_numeric(fp[FILL_TOTAL_CLIENTS_COL], errors="coerce")
fp["clients"] = pd.to_numeric(fp["clients"], errors="coerce")
fp = fp[fp[ID_COL].notna() & (fp[ID_COL] > 0)].copy()
fp["assigned_clients"] = fp[FILL_TOTAL_CLIENTS_COL] - fp["clients"]
fp["assigned_clients"] = fp["assigned_clients"].clip(lower=0)
fp_agg = fp.groupby(ID_COL, as_index=False).agg({
    FILL_TOTAL_CLIENTS_COL: "sum",
    "clients": "sum",
    "assigned_clients": "sum"
}).fillna({FILL_TOTAL_CLIENTS_COL: 0.0, "clients": 0.0, "assigned_clients": 0.0})
fp_agg[ID_COL] = fp_agg[ID_COL].astype(np.int64)


if resell.empty:
    raise RuntimeError(f"Layer '{RESELL_LAYER}' is empty.")


for col, layer in [(ID_COL, RESELL_LAYER), (RESELL_COST_IN_COL, RESELL_LAYER)]:
    if col not in resell.columns:
        raise KeyError(f"Missing '{col}' in layer '{layer}'")



print("[1B/4] Calcolo reddito mensile spaziale per ogni reseller...")
resell_income_monthly = []
for geom in resell.geometry:
    inc = extract_local_income_mean(geom, income_grid, transform, window_size=2)
    resell_income_monthly.append(inc)
resell_income_monthly = np.array(resell_income_monthly, dtype=np.float32)

# Applica minimum wage come floor
MINIMUM_WAGE_USD_PER_MONTH = 26.0  # già definito, ma qui usato esplicitamente
income_monthly_used = np.where(
    np.isfinite(resell_income_monthly) & (resell_income_monthly > MINIMUM_WAGE_USD_PER_MONTH),
    resell_income_monthly,
    MINIMUM_WAGE_USD_PER_MONTH
)

# Statistiche da stampare dopo la diagnostica (le raccogliamo ora)
n_under_min_wage = np.sum(~np.isfinite(resell_income_monthly) | (resell_income_monthly <= MINIMUM_WAGE_USD_PER_MONTH))
income_stats = {
    "min": np.nanmin(resell_income_monthly),
    "p5": np.nanpercentile(resell_income_monthly, 5),
    "median": np.nanmedian(resell_income_monthly),
    "mean": np.nanmean(resell_income_monthly),
    "p95": np.nanpercentile(resell_income_monthly, 95),
    "max": np.nanmax(resell_income_monthly)
}

print(f"Reddito mensile stimato: {n_under_min_wage} reseller su {len(resell)} utilizzano minimum wage ({n_under_min_wage/len(resell)*100:.1f}%)")

print("[2/4] Computing annual demand and added cost per kg...")

n_resellers = len(resell)
if n_resellers == 0 or fp["assigned_clients"].sum() <= 0:
    raise RuntimeError("No reseller clients in system.")
avg_reseller_clients = fp["assigned_clients"].sum() / n_resellers
avg_reseller_demand_kg = annual_lpg_demand_kg(avg_reseller_clients)
annual_kg = np.full(n_resellers, avg_reseller_demand_kg)

cost_in = pd.to_numeric(resell[RESELL_COST_IN_COL], errors="coerce").to_numpy(dtype=float)

# --- Uniform fixed cost per kg for reseller (allocation basata su domanda media di sistema) ---


if len(resell) == 0 or avg_reseller_demand_kg <= 0:
    raise RuntimeError("No valid resellers or total demand is zero for fixed cost allocation.")


# Costi fissi annuali specifici per reseller (affitto + licenza + salario variabile)
annual_salary_usd_per_reseller = income_monthly_used * 12.0
total_fixed_annual_per_reseller = ANNUAL_RENT_USD + annual_salary_usd_per_reseller + LICENSE_ANNUAL_USD

# Costo fisso per kg = costi fissi totali del reseller / domanda media di sistema
fixed_cost_per_kg = total_fixed_annual_per_reseller / avg_reseller_demand_kg



added_cost_per_kg = fixed_cost_per_kg + VOM_USD_PER_KG

valid_mask = np.isfinite(annual_kg) & (annual_kg > 0) & np.isfinite(cost_in) & (cost_in >= 0)

cost_out_uncapped = np.where(valid_mask, cost_in + added_cost_per_kg, np.nan)
max_cost_out_allowed = cost_in * (1.0 + COST_INCREASE_CAP_PCT)
res_capped_mask = valid_mask & np.isfinite(cost_out_uncapped) & (cost_out_uncapped > max_cost_out_allowed)
cost_out = np.where(valid_mask, np.minimum(cost_out_uncapped, max_cost_out_allowed), np.nan)

print("[3/4] Updating reseller layer...")
resell[RESELL_COST_OUT_COL] = pd.to_numeric(pd.Series(cost_out, index=resell.index), errors="coerce").astype(float)
# Helper column for diagnostics and sensitivity checks
resell["res_added_fixed_usd_kg"] = pd.to_numeric(pd.Series(fixed_cost_per_kg, index=resell.index), errors="coerce").astype(float)


_write_gpkg_layer(CHAIN_GPKG, RESELL_LAYER, resell)

print("[4/4] QA summary...")
n_total = len(resell)
n_valid = int(valid_mask.sum())
n_no_demand = int(np.sum(np.isfinite(annual_kg) & (annual_kg <= 0)))
n_invalid_inbound = int(np.sum(~(np.isfinite(cost_in) & (cost_in >= 0))))
print(f"Reseller points total                    : {n_total:,}")
print(f"Reseller points with valid in+out inputs : {n_valid:,}")
print(f"Reseller points capped at +{100.0 * COST_INCREASE_CAP_PCT:.0f}%: {int(res_capped_mask.sum()):,}")
print(f"Rows with zero/nonpositive annual demand : {n_no_demand:,}")
print(f"Rows with invalid inbound cost           : {n_invalid_inbound:,}")

if n_valid > 0:
    out_valid = resell.loc[valid_mask, RESELL_COST_OUT_COL].to_numpy(dtype=float)
    add_valid = added_cost_per_kg[valid_mask]
    print(
        f"{RESELL_COST_OUT_COL} min/median/mean/max: ",
        f"{np.nanmin(out_valid):.4f} / {np.nanmedian(out_valid):.4f} / ",
        f"{np.nanmean(out_valid):.4f} / {np.nanmax(out_valid):.4f}")
    print(
        "Added retailer cost min/median/mean/max: ",
        f"{np.nanmin(add_valid):.4f} / {np.nanmedian(add_valid):.4f} / ",
        f"{np.nanmean(add_valid):.4f} / {np.nanmax(add_valid):.4f}")
else:
    print("No valid reseller rows with annual demand > 0. cost_res_kg_out left as NaN.")


# Stampa statistiche reddito mensile
print("\n=== STATS INCOME RESELLER (USD/mese) ===")
print(f"Min:    {income_stats['min']:.2f}")
print(f"P5:     {income_stats['p5']:.2f}")
print(f"Median: {income_stats['median']:.2f}")
print(f"Mean:   {income_stats['mean']:.2f}")
print(f"P95:    {income_stats['p95']:.2f}")
print(f"Max:    {income_stats['max']:.2f}")
print(f"Reseller con reddito <= minimum wage (26.0 USD): {n_under_min_wage} su {len(resell)}")

print(f"Done. Updated layer '{RESELL_LAYER}' in {CHAIN_GPKG}")

"""
Final QA stats for 4.5 outputs with split in/out costs.

Checks:
1) filling points have valid cost_fil_kg_out and cost_fil_kg_in
2) resell points have valid filling_reference and travel stats
3) resell points have valid cost_res_kg_in and cost_res_kg_out
4) reseller outbound premium is coherent with fixed-cost add-on logic
"""

DATA_DIR = Path("dataset_big")
WORK_GPKG = CHAIN_WITH_COST_PATH

RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"
FILL_COST_OUT_COL = "cost_fil_kg_out"
FILL_COST_IN_COL = "cost_fil_kg_in"
RESELL_COST_IN_COL = "cost_res_kg_in"
RESELL_COST_OUT_COL = "cost_res_kg_out"
RESELL_ADDED_FIXED_COL = "res_added_fixed_usd_kg"
LEGACY_COST_COL = "cost_kg"

print("[1/4] Loading layers...")
resell = _read_gpkg_layer(WORK_GPKG, layer=RESELL_LAYER)
filling = _read_gpkg_layer(WORK_GPKG, layer=FILLING_LAYER)
if resell.empty or filling.empty:
    raise RuntimeError("Resell or filling layer is empty in chain_with_cost.gpkg")

for c in [ID_COL, FILL_COST_OUT_COL, FILL_COST_IN_COL]:
    if c not in filling.columns:
        raise KeyError(f"Missing column '{c}' in layer '{FILLING_LAYER}'")
for c in [ID_COL, FILL_REF_COL, FILL_TIME_COL, RESELL_COST_IN_COL, RESELL_COST_OUT_COL]:
    if c not in resell.columns:
        raise KeyError(f"Missing column '{c}' in layer '{RESELL_LAYER}'")

fill_id = pd.to_numeric(filling[ID_COL], errors="coerce")
fill_cost_out = pd.to_numeric(filling[FILL_COST_OUT_COL], errors="coerce")
fill_cost_in = pd.to_numeric(filling[FILL_COST_IN_COL], errors="coerce")
res_ref = pd.to_numeric(resell[FILL_REF_COL], errors="coerce")
res_tmin = pd.to_numeric(resell[FILL_TIME_COL], errors="coerce")
res_cost_in = pd.to_numeric(resell[RESELL_COST_IN_COL], errors="coerce")
res_cost_out = pd.to_numeric(resell[RESELL_COST_OUT_COL], errors="coerce")

print("[2/4] QA on filling in/out costs...")
n_fill = len(filling)
fill_out_valid = np.isfinite(fill_cost_out)
fill_in_valid = np.isfinite(fill_cost_in)
fill_out_nonneg = fill_out_valid & (fill_cost_out >= 0)
fill_in_nonneg = fill_in_valid & (fill_cost_in >= 0)

print("=== FILLING COST QA ===")
print(f"Filling points total                  : {n_fill:,}")
print(f"Finite {FILL_COST_OUT_COL}            : {int(fill_out_valid.sum()):,}")
print(f"Finite {FILL_COST_IN_COL}             : {int(fill_in_valid.sum()):,}")
print(f"{FILL_COST_OUT_COL} < 0               : {int((~fill_out_nonneg).sum()):,}")
print(f"{FILL_COST_IN_COL} < 0                : {int((~fill_in_nonneg).sum()):,}")
if fill_out_valid.any():
    print(
        f"{FILL_COST_OUT_COL} min/median/mean/max: "
        f"{float(np.nanmin(fill_cost_out)):.4f} / {float(np.nanmedian(fill_cost_out)):.4f} / "
        f"{float(np.nanmean(fill_cost_out)):.4f} / {float(np.nanmax(fill_cost_out)):.4f}"
    )
if fill_in_valid.any():
    print(
        f"{FILL_COST_IN_COL} min/median/mean/max : "
        f"{float(np.nanmin(fill_cost_in)):.4f} / {float(np.nanmedian(fill_cost_in)):.4f} / "
        f"{float(np.nanmean(fill_cost_in)):.4f} / {float(np.nanmax(fill_cost_in)):.4f}"
    )
if LEGACY_COST_COL in filling.columns:
    legacy_fill = pd.to_numeric(filling[LEGACY_COST_COL], errors="coerce")
    print(f"Legacy {LEGACY_COST_COL} finite count : {int(np.isfinite(legacy_fill).sum()):,}/{n_fill:,}")

print("\n[3/4] QA on reseller filling reference, travel stats, and costs...")
n_res = len(resell)
fill_id_set = set(fill_id[np.isfinite(fill_id)].astype(np.int64).tolist())
ref_finite = np.isfinite(res_ref)
ref_positive = ref_finite & (res_ref > 0)
ref_exists = ref_positive & pd.Series(res_ref.astype("Int64")).isin(fill_id_set).to_numpy()

time_valid = np.isfinite(res_tmin) & (res_tmin >= 0) & (res_tmin < 7000)
ref_and_time_valid = ref_exists & time_valid

res_in_valid = np.isfinite(res_cost_in) & (res_cost_in >= 0)
res_out_valid = np.isfinite(res_cost_out) & (res_cost_out >= 0)
res_in_linked = ref_and_time_valid & res_in_valid
res_out_linked = ref_and_time_valid & res_out_valid

print("=== RESELLER REFERENCE QA ===")
print(f"Resell points total                    : {n_res:,}")
print(f"Resell with finite filling_reference   : {int(ref_finite.sum()):,}")
print(f"Resell with existing filling_reference : {int(ref_exists.sum()):,}")
print(f"Resell with valid reference + time     : {int(ref_and_time_valid.sum()):,}")
print(f"Missing/invalid reference              : {int((~ref_exists).sum()):,}")
if ref_and_time_valid.any():
    t = res_tmin[ref_and_time_valid].to_numpy(dtype=float)
    print(
        "Travel time min/median/mean/p95/max (min): "
        f"{float(np.nanmin(t)):.2f} / {float(np.nanmedian(t)):.2f} / {float(np.nanmean(t)):.2f} / "
        f"{float(np.nanpercentile(t, 95)):.2f} / {float(np.nanmax(t)):.2f}"
    )

print("=== RESELLER COST QA ===")
print(f"Finite {RESELL_COST_IN_COL}                    : {int(np.isfinite(res_cost_in).sum()):,}/{n_res:,}")
print(f"Finite {RESELL_COST_OUT_COL}                   : {int(np.isfinite(res_cost_out).sum()):,}/{n_res:,}")
print(f"{RESELL_COST_IN_COL} valid + linked            : {int(res_in_linked.sum()):,}/{n_res:,}")
print(f"{RESELL_COST_OUT_COL} valid + linked           : {int(res_out_linked.sum()):,}/{n_res:,}")
print(f"{RESELL_COST_IN_COL} NaN rows                  : {int((~np.isfinite(res_cost_in)).sum()):,}")
print(f"{RESELL_COST_OUT_COL} NaN rows                 : {int((~np.isfinite(res_cost_out)).sum()):,}")
if res_in_valid.any():
    print(
        f"{RESELL_COST_IN_COL} min/median/mean/p95/max      : "
        f"{float(np.nanmin(res_cost_in)):.4f} / {float(np.nanmedian(res_cost_in)):.4f} / "
        f"{float(np.nanmean(res_cost_in)):.4f} / {float(np.nanpercentile(res_cost_in, 95)):.4f} / "
        f"{float(np.nanmax(res_cost_in)):.4f}"
    )
if res_out_valid.any():
    print(
        f"{RESELL_COST_OUT_COL} min/median/mean/p95/max     : "
        f"{float(np.nanmin(res_cost_out)):.4f} / {float(np.nanmedian(res_cost_out)):.4f} / "
        f"{float(np.nanmean(res_cost_out)):.4f} / {float(np.nanpercentile(res_cost_out, 95)):.4f} / "
        f"{float(np.nanmax(res_cost_out)):.4f}"
    )

paired_valid = res_in_valid & res_out_valid & ref_and_time_valid
if paired_valid.any():
    premium = (res_cost_out[paired_valid] - res_cost_in[paired_valid]).to_numpy(dtype=float)
    monotonic_ok = int(np.sum(premium >= -1e-9))
    print("=== OUTBOUND PREMIUM QA ===")
    print(f"Rows with out >= in (paired valid)     : {monotonic_ok:,}/{int(paired_valid.sum()):,}")
    print(
        "Premium (out - in) min/median/mean/p95/max: "
        f"{float(np.nanmin(premium)):.6f} / {float(np.nanmedian(premium)):.6f} / "
        f"{float(np.nanmean(premium)):.6f} / {float(np.nanpercentile(premium, 95)):.6f} / "
        f"{float(np.nanmax(premium)):.6f}"
    )

if RESELL_ADDED_FIXED_COL in resell.columns:
    added = pd.to_numeric(resell[RESELL_ADDED_FIXED_COL], errors="coerce")
    added_valid = np.isfinite(added) & (added >= 0)
    print("=== FIXED ADD-ON QA ===")
    print(f"Finite {RESELL_ADDED_FIXED_COL}            : {int(added_valid.sum()):,}/{n_res:,}")
    if added_valid.any():
        av = added[added_valid].to_numpy(dtype=float)
        print(
            f"{RESELL_ADDED_FIXED_COL} min/median/mean/p95/max: "
            f"{float(np.nanmin(av)):.6f} / {float(np.nanmedian(av)):.6f} / "
            f"{float(np.nanmean(av)):.6f} / {float(np.nanpercentile(av, 95)):.6f} / {float(np.nanmax(av)):.6f}"
        )

if (resell.crs is not None) and (filling.crs is not None) and (str(resell.crs) == str(filling.crs)):
    fill_geom = filling[[ID_COL, "geometry"]].copy()
    fill_geom[ID_COL] = pd.to_numeric(fill_geom[ID_COL], errors="coerce")
    merged = resell[[ID_COL, FILL_REF_COL, "geometry"]].copy()
    merged[ID_COL] = pd.to_numeric(merged[ID_COL], errors="coerce")
    merged[FILL_REF_COL] = pd.to_numeric(merged[FILL_REF_COL], errors="coerce")
    merged = merged.merge(fill_geom, left_on=FILL_REF_COL, right_on=ID_COL, how="left", suffixes=("_res", "_fill"))
    good_geom = merged["geometry_res"].notna() & merged["geometry_fill"].notna()
    if good_geom.any():
        dist_km = merged.loc[good_geom, "geometry_res"].distance(merged.loc[good_geom, "geometry_fill"]).astype(float) / 1000.0
        print(
            "Straight-line distance (COMPUTED NOW NOT FOR CALCULATION) min/median/mean/p95/max (km): "
            f"{float(np.nanmin(dist_km)):.2f} / {float(np.nanmedian(dist_km)):.2f} / {float(np.nanmean(dist_km)):.2f} / "
            f"{float(np.nanpercentile(dist_km, 95)):.2f} / {float(np.nanmax(dist_km)):.2f}"
        )

print("\n[4/4] QA completed.")

"""
End-user LPG price allocation per pixel (Nigeria, EPSG:3857)
Raster-to-raster implementation.

Updated 2026-04:
- Reads 8 bands (includes walk/car distances).
- Collection cost is per-trip (per 12.5 kg cylinder), not annualised.
- Uses actual path distances for vehicle operating cost.
"""

DATA_DIR = Path("dataset_big")

# Input from 4.2 (multilayer raster with 8 bands)
SOURCE_PIXEL_RASTER = HUFF_PIXEL_RASTER_PATH

# Cost source from 4.4 / 4.5
COST_SOURCE_GPKG = CHAIN_WITH_COST_PATH
RESELL_LAYER = "resell"
RESELLER_ID_COL = "id_res&fil"
RESELLER_COST_COL = "cost_res_kg_out"
FILLING_LAYER = "filling"
FILLING_COST_COL = "cost_fil_kg_out"

# Input from 4.3 and base demography
LPG_USE_SHARE_RASTER = LPG_USE_SHARE_PATH
POPULATION_RASTER = POPULATION_RASTER_PATH

# Optional spatial VOT input from 4.1
INCOME_RASTER = INCOME_RASTER_PATH
USE_SPATIAL_VOT = True

# Output raster
OUTPUT_PIXEL_RASTER = END_USER_PRICE_PATH

# New output band names
OUT_COST_REF_WALK = "res_cost_kg_out_walk_ref"
OUT_COST_REF_CAR = "res_cost_kg_out_car_ref"
OUT_COST_WALK = "cost_kg_walker"
OUT_COST_CAR = "cost_kg_driver"

# guards
NODATA_FLOAT = -9999.0

# Walking collection model (per trip)
WALK_TIME_IS_ONE_WAY = True
FIXED_TIME_AT_RETAILER_HOURS = 0.25
CYLINDER_WEIGHT_KG = 12.5

# Driving collection model (per trip)
CAR_VARIABLE_COST_PER_KM = 0.136


# VOT
MINIMUM_WAGE_USD_PER_MONTH = 26.0
WORK_HOURS_PER_MONTH = 30 * 8
WAGE_RANGE = (0.2, 0.5)
WAGE_FRACTION_FALLBACK = 0.3
DEFAULT_VOT_USD_PER_HOUR = WAGE_FRACTION_FALLBACK * (MINIMUM_WAGE_USD_PER_MONTH / WORK_HOURS_PER_MONTH)
VOT_TODO_NOTE = "TODO: review MINIMUM_WAGE_USD_PER_MONTH and WAGE_RANGE for local context"

# treshold for walk/car statistic
WALK_THRESHOLD_MIN = 30   
CAR_THRESHOLD_MIN = 45    





def _read_single_band(path: Path) -> tuple[np.ndarray, dict]:
    cached = _read_cached_raster(path)
    if cached is not None:
        arr = cached["array"]
        if arr.ndim == 3:
            arr = arr[0]
        arr = arr.astype(np.float32, copy=False)
        profile = cached["profile"].copy()
        nodata = cached.get("nodata")
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
        return arr, profile
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    return arr, profile


def _align_to_reference(path: Path, ref_profile: dict, resampling: Resampling) -> np.ndarray:
    cached = _read_cached_raster(path)
    dst = np.full((ref_profile["height"], ref_profile["width"]), np.nan, dtype=np.float32)
    if cached is not None:
        src_arr = cached["array"]
        if src_arr.ndim == 3:
            src_arr = src_arr[0]
        reproject(
            source=src_arr,
            destination=dst,
            src_transform=cached["profile"]["transform"],
            src_crs=cached["profile"]["crs"],
            src_nodata=cached.get("nodata"),
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
        return dst
    with rasterio.open(path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def _compute_spatial_vot(income_array: np.ndarray) -> tuple[np.ndarray, str]:
    valid = np.isfinite(income_array)
    vot = np.full(income_array.shape, DEFAULT_VOT_USD_PER_HOUR, dtype=np.float32)
    if not np.any(valid):
        return vot, "fallback_fixed_no_valid_income"

    min_value = float(np.nanmin(income_array[valid]))
    max_value = float(np.nanmax(income_array[valid]))
    if not np.isfinite(min_value) or not np.isfinite(max_value) or max_value <= min_value:
        return vot, "fallback_fixed_flat_income"

    wage_min, wage_max = WAGE_RANGE
    norm = (income_array - min_value) / (max_value - min_value)
    norm = np.clip(norm, 0.0, 1.0)
    wage_factor = wage_min + norm * float(wage_max - wage_min)

    vot_valid = wage_factor * (MINIMUM_WAGE_USD_PER_MONTH / WORK_HOURS_PER_MONTH)
    vot[valid] = vot_valid[valid].astype(np.float32)
    return vot, "spatial_income"


def _map_effective_cost(ids_int: np.ndarray, map_resell: dict[int, float], map_filling: dict[int, float]) -> np.ndarray:
    out = np.full(ids_int.shape, np.nan, dtype=np.float64)
    valid = ids_int > 0
    if not np.any(valid):
        return out

    uniq, inv = np.unique(ids_int[valid], return_inverse=True)
    mapped = np.fromiter((map_resell.get(int(rid), np.nan) for rid in uniq), dtype=np.float64, count=uniq.size)
    miss = ~np.isfinite(mapped)
    if np.any(miss):
        miss_ids = uniq[miss]
        mapped[miss] = np.fromiter((map_filling.get(int(rid), np.nan) for rid in miss_ids), dtype=np.float64, count=miss_ids.size)

    out_valid = mapped[inv]
    out[valid] = out_valid
    return out


def _write_multiband(path: Path, base_profile: dict, bands: list[np.ndarray], names: list[str]) -> None:
    if len(bands) != len(names):
        raise ValueError("Bands and names must have same length.")
    profile = base_profile.copy()
    profile.update(dtype="float32", count=len(bands), nodata=NODATA_FLOAT, compress="lzw")

    if save_outputs.get(_key_from_path(path), True):
        with rasterio.open(path, "w", **profile) as dst:
            for i, (band, name) in enumerate(zip(bands, names), start=1):
                out = np.where(np.isfinite(band), band, NODATA_FLOAT).astype(np.float32)
                dst.write(out, i)
                dst.set_band_description(i, name)

    _cache_raster(path, np.stack([np.where(np.isfinite(b), b, NODATA_FLOAT).astype(np.float32) for b in bands], axis=0), profile, NODATA_FLOAT, names)


# ------------------------------------------------------------------------------
# Main execution
# ------------------------------------------------------------------------------
print("[1/8] Reading source multilayer pixel raster...")
if not SOURCE_PIXEL_RASTER.exists():
    raise FileNotFoundError(f"Missing input raster from 4.2: {SOURCE_PIXEL_RASTER}")

cached_src = _read_cached_raster(SOURCE_PIXEL_RASTER)
if cached_src is not None:
    base_stack = cached_src["array"].astype(np.float32, copy=False)
    base_profile = cached_src["profile"].copy()
    src_nodata = cached_src.get("nodata")
    if base_stack.ndim == 2:
        raise RuntimeError(f"Expected at least 8 bands in {SOURCE_PIXEL_RASTER}, found 1")
    if base_stack.shape[0] < 8:
        raise RuntimeError(f"Expected at least 8 bands in {SOURCE_PIXEL_RASTER}, found {base_stack.shape[0]}")
else:
    with rasterio.open(SOURCE_PIXEL_RASTER) as src:
        # Expect 8 bands: 1:car_share, 2:walk_share, 3:walk_id, 4:walk_time, 5:walk_dist,
        #                 6:car_id, 7:car_time, 8:car_dist
        if src.count < 8:
            raise RuntimeError(f"Expected at least 8 bands in {SOURCE_PIXEL_RASTER}, found {src.count}")
        base_stack = src.read(indexes=[1, 2, 3, 4, 5, 6, 7, 8]).astype(np.float32, copy=False)
        base_profile = src.profile.copy()
        src_nodata = src.nodata

if src_nodata is not None:
    base_stack = np.where(base_stack == src_nodata, np.nan, base_stack).astype(np.float32)

car_share = base_stack[0]
walk_share = base_stack[1]
walk_id = base_stack[2]
walk_time_min = base_stack[3]
walk_dist_km = base_stack[4]        # <-- NEW: band 5
car_id = base_stack[5]              # <-- index shift: was 4, now 5
car_time_min = base_stack[6]        # was 5, now 6
car_dist_km = base_stack[7]         # <-- NEW: band 8

height, width = car_share.shape
print(f"Source grid: {width} x {height}")

print("[2/8] Checking external inputs...")
if not COST_SOURCE_GPKG.exists():
    raise FileNotFoundError(f"Missing cost source GPKG: {COST_SOURCE_GPKG}")
if not LPG_USE_SHARE_RASTER.exists():
    raise FileNotFoundError(f"Missing 4.3 output raster: {LPG_USE_SHARE_RASTER}")
if not POPULATION_RASTER.exists():
    raise FileNotFoundError(f"Missing population raster: {POPULATION_RASTER}")

print("[3/8] Loading cost layers and creating fallback cost maps...")
resell = _read_gpkg_layer(COST_SOURCE_GPKG, layer=RESELL_LAYER)
filling = _read_gpkg_layer(COST_SOURCE_GPKG, layer=FILLING_LAYER)
if resell.empty or filling.empty:
    raise RuntimeError("Resell or filling layer is empty in chain_with_cost.gpkg")
for c in [RESELLER_ID_COL, RESELLER_COST_COL]:
    if c not in resell.columns:
        raise KeyError(f"Missing '{c}' in layer '{RESELL_LAYER}'")
for c in [RESELLER_ID_COL, FILLING_COST_COL]:
    if c not in filling.columns:
        raise KeyError(f"Missing '{c}' in layer '{FILLING_LAYER}'")

resell_id = pd.to_numeric(resell[RESELLER_ID_COL], errors="coerce")
resell_cost = pd.to_numeric(resell[RESELLER_COST_COL], errors="coerce")
fill_id = pd.to_numeric(filling[RESELLER_ID_COL], errors="coerce")
fill_cost = pd.to_numeric(filling[FILLING_COST_COL], errors="coerce")

mask_resell = resell_id.notna()
mask_fill = fill_id.notna()
map_resell_cost = dict(zip(resell_id[mask_resell].astype(np.int64), resell_cost[mask_resell].astype(float)))
map_fill_cost = dict(zip(fill_id[mask_fill].astype(np.int64), fill_cost[mask_fill].astype(float)))

print("[3b/8] Building filling reference maps for pixel assignment...")
resell_gdf = _read_gpkg_layer(COST_SOURCE_GPKG, layer=RESELL_LAYER)
if RESELLER_ID_COL not in resell_gdf.columns:
    raise KeyError(f"Missing '{RESELLER_ID_COL}' in resell layer")
if "filling_reference" not in resell_gdf.columns:
    raise KeyError("Missing 'filling_reference' column in resell layer (run 4.4 first)")

resell_id_series = pd.to_numeric(resell_gdf[RESELLER_ID_COL], errors="coerce")
fill_ref_series = pd.to_numeric(resell_gdf["filling_reference"], errors="coerce")
mask_resell = resell_id_series.notna() & fill_ref_series.notna()
map_reseller_to_filling = dict(zip(
    resell_id_series[mask_resell].astype(np.int64),
    fill_ref_series[mask_resell].astype(np.int64)
))

filling_gdf = _read_gpkg_layer(COST_SOURCE_GPKG, layer=FILLING_LAYER)
fill_id_series = pd.to_numeric(filling_gdf[RESELLER_ID_COL], errors="coerce")
fill_cost_out_series = pd.to_numeric(filling_gdf[FILLING_COST_COL], errors="coerce")
mask_fill = fill_id_series.notna() & fill_cost_out_series.notna()
map_filling_cost = dict(zip(
    fill_id_series[mask_fill].astype(np.int64),
    fill_cost_out_series[mask_fill].astype(float)
))

def resolve_filling_id(rid: int) -> int:
    if rid in map_reseller_to_filling:
        return map_reseller_to_filling[rid]
    return rid

def get_filling_cost(fid: int) -> float:
    return map_filling_cost.get(fid, np.nan)

print(f"Reseller->filling mapping built: {len(map_reseller_to_filling):,} entries")
print(f"Filling cost map built: {len(map_filling_cost):,} entries")

print("[4/8] Loading population/LPG/VOT rasters...")
lpg_use_share, _ = _read_single_band(LPG_USE_SHARE_RASTER)
population, pop_profile = _read_single_band(POPULATION_RASTER)
if lpg_use_share.shape != (height, width) or population.shape != (height, width):
    raise RuntimeError("Shape mismatch among source pixel raster, LPG use raster, and population raster.")

if USE_SPATIAL_VOT and INCOME_RASTER.exists():
    income_aligned = _align_to_reference(INCOME_RASTER, pop_profile, Resampling.bilinear)
    vot_raster, vot_mode = _compute_spatial_vot(income_aligned)
elif USE_SPATIAL_VOT and (not INCOME_RASTER.exists()):
    vot_raster = np.full((height, width), DEFAULT_VOT_USD_PER_HOUR, dtype=np.float32)
    vot_mode = "fallback_fixed_income_missing"
else:
    vot_raster = np.full((height, width), DEFAULT_VOT_USD_PER_HOUR, dtype=np.float32)
    vot_mode = "fixed_user_parameter"

print("[5/8] Computing reference costs and channel demands...")
n = height * width

walk_id_flat = walk_id.reshape(-1).astype(np.float64)
car_id_flat = car_id.reshape(-1).astype(np.float64)
walk_time_flat = walk_time_min.reshape(-1).astype(np.float64)
car_time_flat = car_time_min.reshape(-1).astype(np.float64)
walk_dist_flat = walk_dist_km.reshape(-1).astype(np.float64)    # <-- NEW
car_dist_flat = car_dist_km.reshape(-1).astype(np.float64)      # <-- NEW
share_walk_flat = walk_share.reshape(-1).astype(np.float64)
share_car_flat = car_share.reshape(-1).astype(np.float64)
pop_flat = population.reshape(-1).astype(np.float64)
lpg_flat = lpg_use_share.reshape(-1).astype(np.float64)
vot_flat = vot_raster.reshape(-1).astype(np.float64)

with np.errstate(invalid='ignore'):
    walk_id_int = np.where(np.isfinite(walk_id_flat) & (walk_id_flat > 0), walk_id_flat.astype(np.int64), -1)
    car_id_int = np.where(np.isfinite(car_id_flat) & (car_id_flat > 0), car_id_flat.astype(np.int64), -1)

ref_cost_walk_arr = _map_effective_cost(walk_id_int, map_resell_cost, map_fill_cost)
ref_cost_car_arr = _map_effective_cost(car_id_int, map_resell_cost, map_fill_cost)

# Arrays for filling id and filling cost (unchanged)
filling_id_walk_arr = np.full(n, -1, dtype=np.int32)
filling_id_car_arr = np.full(n, -1, dtype=np.int32)
filling_cost_walk_arr = np.full(n, np.nan, dtype=np.float64)
filling_cost_car_arr = np.full(n, np.nan, dtype=np.float64)

valid_walk_id = (walk_id_int > 0)
if np.any(valid_walk_id):
    uniq_ids, inv = np.unique(walk_id_int[valid_walk_id], return_inverse=True)
    fid_mapped = np.array([resolve_filling_id(int(rid)) for rid in uniq_ids], dtype=np.int32)
    fcost_mapped = np.array([get_filling_cost(int(fid)) for fid in fid_mapped], dtype=np.float64)
    filling_id_walk_arr[valid_walk_id] = fid_mapped[inv]
    filling_cost_walk_arr[valid_walk_id] = fcost_mapped[inv]

valid_car_id = (car_id_int > 0)
if np.any(valid_car_id):
    uniq_ids, inv = np.unique(car_id_int[valid_car_id], return_inverse=True)
    fid_mapped = np.array([resolve_filling_id(int(rid)) for rid in uniq_ids], dtype=np.int32)
    fcost_mapped = np.array([get_filling_cost(int(fid)) for fid in fid_mapped], dtype=np.float64)
    filling_id_car_arr[valid_car_id] = fid_mapped[inv]
    filling_cost_car_arr[valid_car_id] = fcost_mapped[inv]

# Normalize walk/car shares
share_walk_arr = np.clip(share_walk_flat, 0.0, 1.0)
share_car_arr = np.clip(share_car_flat, 0.0, 1.0)
share_sum = share_walk_arr + share_car_arr
share_valid = share_sum > 0
share_walk_norm = np.zeros(n, dtype=np.float64)
share_car_norm = np.zeros(n, dtype=np.float64)
share_walk_norm[share_valid] = share_walk_arr[share_valid] / share_sum[share_valid]
share_car_norm[share_valid] = share_car_arr[share_valid] / share_sum[share_valid]

walk_frac = share_walk_norm
car_frac = share_car_norm

# ------------------------------------------------------------------------
# Walker model – per-trip collection cost
# ------------------------------------------------------------------------
# Assumption: time is one-way; round-trip = 2 * time_min + fixed retailer time.
walk_round_trip_hours = np.where(
    WALK_TIME_IS_ONE_WAY,
    (2.0 * walk_time_flat) / 60.0,
    walk_time_flat / 60.0
)
total_trip_time_hours_walk = walk_round_trip_hours + FIXED_TIME_AT_RETAILER_HOURS
trip_cost_walk_usd = total_trip_time_hours_walk * vot_flat
collection_cost_per_kg_walk = trip_cost_walk_usd / CYLINDER_WEIGHT_KG

valid_walk_mask = (
    np.isfinite(ref_cost_walk_arr)
    & (walk_id_int > 0)
    & np.isfinite(walk_round_trip_hours)
    & (walk_round_trip_hours >= 0)
    & np.isfinite(vot_flat)
)

cost_walk_final = np.full(n, np.nan, dtype=np.float64)
cost_walk_final[valid_walk_mask] = (
    ref_cost_walk_arr[valid_walk_mask] + collection_cost_per_kg_walk[valid_walk_mask]
)

# ------------------------------------------------------------------------
# Driver model – per-trip collection cost using actual distances
# ------------------------------------------------------------------------
round_trip_distance_km_car = 2.0 * car_dist_flat
round_trip_drive_hours = (car_time_flat * 2.0) / 60.0
total_trip_time_hours_car = round_trip_drive_hours + FIXED_TIME_AT_RETAILER_HOURS

vehicle_operating_cost_trip_usd = round_trip_distance_km_car * CAR_VARIABLE_COST_PER_KM
driver_time_cost_trip_usd = total_trip_time_hours_car * vot_flat
trip_cost_car_usd = vehicle_operating_cost_trip_usd + driver_time_cost_trip_usd
collection_cost_per_kg_car = trip_cost_car_usd / CYLINDER_WEIGHT_KG

valid_driver_mask = (
    np.isfinite(ref_cost_car_arr)
    & (car_id_int > 0)
    & np.isfinite(car_time_flat)
    & (car_time_flat >= 0)
    & np.isfinite(car_dist_flat)          # <-- ensure distance is valid
    & (car_dist_flat >= 0)
    & np.isfinite(total_trip_time_hours_car)
    & np.isfinite(vot_flat)
)

cost_car_final = np.full(n, np.nan, dtype=np.float64)
cost_car_final[valid_driver_mask] = (
    ref_cost_car_arr[valid_driver_mask] + collection_cost_per_kg_car[valid_driver_mask]
)

# new band with mean cost per kg weighted by walk/car shares (NaN if no valid cost or zero share)
mean_user_cost_flat = np.full(n, np.nan, dtype=np.float64)
valid_mean = (share_walk_norm + share_car_norm) > 0
mean_user_cost_flat[valid_mean] = (
    cost_car_final[valid_mean] * share_car_norm[valid_mean] +
    cost_walk_final[valid_mean] * share_walk_norm[valid_mean]
)
mean_user_cost = mean_user_cost_flat.reshape(height, width)



# ------------------------------------------------------------------------
print("[6/8] Writing output multilayer raster...")

best_id_walk_band = np.where(np.isfinite(walk_id) & (walk_id >= 0), walk_id, np.nan)
best_id_car_band = np.where(np.isfinite(car_id) & (car_id >= 0), car_id, np.nan)

out_ref_walk = ref_cost_walk_arr.reshape(height, width)
out_ref_car = ref_cost_car_arr.reshape(height, width)
out_cost_walk = cost_walk_final.reshape(height, width)
out_cost_car = cost_car_final.reshape(height, width)

out_bands = [
    car_share.astype(np.float32),
    walk_share.astype(np.float32),
    best_id_walk_band.astype(np.float32),
    walk_time_min.astype(np.float32),
    walk_dist_km.astype(np.float32),                     # <-- NUOVO
    best_id_car_band.astype(np.float32),
    car_time_min.astype(np.float32),
    car_dist_km.astype(np.float32),                      # <-- NUOVO
    out_ref_walk.astype(np.float32),
    out_ref_car.astype(np.float32),
    out_cost_walk.astype(np.float32),
    out_cost_car.astype(np.float32),
    filling_id_walk_arr.reshape(height, width).astype(np.float32),
    filling_id_car_arr.reshape(height, width).astype(np.float32),
    filling_cost_walk_arr.reshape(height, width).astype(np.float32),
    filling_cost_car_arr.reshape(height, width).astype(np.float32),
    mean_user_cost.astype(np.float32),
]

out_names = [
    "car_share",
    "walk_share",
    "best_reseller_id_walk",
    "best_time_walk_min",
    "best_distance_walk_km",           # <-- NUOVO
    "best_reseller_id_car",
    "best_time_car_min",
    "best_distance_car_km",            # <-- NUOVO
    OUT_COST_REF_WALK,
    OUT_COST_REF_CAR,
    OUT_COST_WALK,
    OUT_COST_CAR,
    "filling_id_walk",
    "filling_id_car",
    "cost_fil_kg_out_walk_ref",
    "cost_fil_kg_out_car_ref",
    "mean_user_cost"
]

_write_multiband(OUTPUT_PIXEL_RASTER, base_profile, out_bands, out_names)

# ------------------------------------------------------------------------
print("[7/8] Diagnostics...")
n_total = n
n_valid_walk = int(valid_walk_mask.sum())
n_nan_walk = int(np.isnan(cost_walk_final).sum())
n_valid_driver = int(valid_driver_mask.sum())
n_nan_driver = int(np.isnan(cost_car_final).sum())

print("=== WALKER COST MODEL SUMMARY ===")
print(f"Rows total                                 : {n_total:,}")
print(f"Rows valid walker cost                     : {n_valid_walk:,}/{n_total:,}")
print(f"Rows NaN (no cost)                         : {n_nan_walk:,}/{n_total:,}")
print("Collection cost method                      : per-trip / cylinder (12.5 kg)")

print("\n=== DRIVER COST MODEL SUMMARY ===")
print(f"Rows total                                 : {n_total:,}")
print(f"Rows valid driver cost                     : {n_valid_driver:,}/{n_total:,}")
print(f"Rows NaN (no cost)                         : {n_nan_driver:,}/{n_total:,}")
print("Collection cost method                      : per-trip / cylinder (12.5 kg)")
print(f"Car variable cost (USD/km)                  : {CAR_VARIABLE_COST_PER_KM}")

print(f"\nVOT mode                                   : {vot_mode}")
if np.any(np.isfinite(vot_flat)):
    v = vot_flat[np.isfinite(vot_flat)]
    print(f"VOT USD/h min/median/mean/p95/max         : {np.nanmin(v):.6f} / {np.nanmedian(v):.6f} / {np.nanmean(v):.6f} / {np.nanpercentile(v, 95):.6f} / {np.nanmax(v):.6f}")
print(VOT_TODO_NOTE)

n_walk_id_valid = int(np.sum(walk_id_int > 0))
n_car_id_valid = int(np.sum(car_id_int > 0))
n_walk_id_missing = int(np.sum(walk_id_int == -1))
n_car_id_missing = int(np.sum(car_id_int == -1))

print("=== ID ASSIGNMENT DIAGNOSTICS ===")
print(f"Total pixels                     : {n_total:,}")
print(f"Walk ID valid (>0)               : {n_walk_id_valid:,} ({100*n_walk_id_valid/n_total:.2f}%)")
print(f"Car ID valid (>0)                : {n_car_id_valid:,} ({100*n_car_id_valid/n_total:.2f}%)")
print(f"Walk ID missing (-1)             : {n_walk_id_missing:,} ({100*n_walk_id_missing/n_total:.2f}%)")
print(f"Car ID missing (-1)              : {n_car_id_missing:,} ({100*n_car_id_missing/n_total:.2f}%)")

print("=== FILLING ID ASSIGNMENT ===")
print(f"Walk pixels with valid filling id: {np.sum(filling_id_walk_arr > 0):,}")
print(f"Car pixels with valid filling id : {np.sum(filling_id_car_arr > 0):,}")
print(f"Unique filling ids (walk)        : {len(np.unique(filling_id_walk_arr[filling_id_walk_arr > 0]))}")
print(f"Unique filling ids (car)         : {len(np.unique(filling_id_car_arr[filling_id_car_arr > 0]))}")


print("\n=== MEAN USER COST (weighted) STATISTICS ===")
valid_mean_finite = np.isfinite(mean_user_cost_flat)
if np.any(valid_mean_finite):
    m = mean_user_cost_flat[valid_mean_finite]
    print(f"Mean user cost USD/kg  : min={np.nanmin(m):.6f} / median={np.nanmedian(m):.6f} / "
          f"mean={np.nanmean(m):.6f} / p95={np.nanpercentile(m, 95):.6f} / max={np.nanmax(m):.6f}")
else:
    print("No valid mean user cost pixels.")

#stats for accessibility within time threshold
print("\n=== WALKER COST WITH TIME <= {} min ===".format(WALK_THRESHOLD_MIN))
walk_within_thresh = (valid_walk_mask) & (walk_time_flat <= WALK_THRESHOLD_MIN)
if np.any(walk_within_thresh):
    cw = cost_walk_final[walk_within_thresh]
    print(f"Walker cost USD/kg (time <= {WALK_THRESHOLD_MIN} min): "
          f"min={np.nanmin(cw):.6f} / median={np.nanmedian(cw):.6f} / "
          f"mean={np.nanmean(cw):.6f} / max={np.nanmax(cw):.6f}")
    print(f"Pixels in this group: {np.sum(walk_within_thresh):,}")
else:
    print("No walker pixels within time threshold.")

print("\n=== DRIVER COST WITH TIME <= {} min ===".format(CAR_THRESHOLD_MIN))
car_within_thresh = (valid_driver_mask) & (car_time_flat <= CAR_THRESHOLD_MIN)
if np.any(car_within_thresh):
    cc = cost_car_final[car_within_thresh]
    print(f"Driver cost USD/kg (time <= {CAR_THRESHOLD_MIN} min): "
          f"min={np.nanmin(cc):.6f} / median={np.nanmedian(cc):.6f} / "
          f"mean={np.nanmean(cc):.6f} / max={np.nanmax(cc):.6f}")
    print(f"Pixels in this group: {np.sum(car_within_thresh):,}")
else:
    print("No driver pixels within time threshold.")




print("[8/8] Done.")
print(f"Output file: {OUTPUT_PIXEL_RASTER}")
print(f"Bands written: {', '.join(out_names)}")
print("Walker formula (per trip):")
print("  cost_kg_walker = reference_point_cost + (round_trip_time_h * VOT) / 12.5")
print("Driver formula (per trip):")
print("  cost_kg_driver = reference_point_cost + (vehicle_op_cost + driver_time_cost) / 12.5")
print(f"Primary reseller source column: {RESELLER_COST_COL}")
print(f"Fallback filling source column: {FILLING_COST_COL}")

"""
Price variation analysis across pixels assigned to the same retailer (ID).
"""

DATA_DIR = Path("dataset_big")
OUTPUT_RASTER = END_USER_PRICE_PATH
NODATA_FLOAT = -9999.0

print("Loading output raster...")
arr, desc, nodata = _read_multiband_raster(OUTPUT_RASTER)

if nodata is None:
    nodata = NODATA_FLOAT

arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
name_to_idx = {d: i for i, d in enumerate(desc) if d is not None}

cost_walk = arr[name_to_idx["cost_kg_walker"]].flatten()
cost_car = arr[name_to_idx["cost_kg_driver"]].flatten()
ref_walk = arr[name_to_idx["res_cost_kg_out_walk_ref"]].flatten()
ref_car = arr[name_to_idx["res_cost_kg_out_car_ref"]].flatten()
walk_id = arr[name_to_idx["best_reseller_id_walk"]].flatten()
car_id = arr[name_to_idx["best_reseller_id_car"]].flatten()

valid_walk = (walk_id > 0) & np.isfinite(cost_walk)
valid_car = (car_id > 0) & np.isfinite(cost_car)

df_walk = pd.DataFrame({
    'retailer_id': walk_id[valid_walk].astype(int),
    'cost_final': cost_walk[valid_walk],
    'cost_ref': ref_walk[valid_walk],
    'collection_component': cost_walk[valid_walk] - ref_walk[valid_walk]
})

df_car = pd.DataFrame({
    'retailer_id': car_id[valid_car].astype(int),
    'cost_final': cost_car[valid_car],
    'cost_ref': ref_car[valid_car],
    'collection_component': cost_car[valid_car] - ref_car[valid_car]
})

def retailer_stats(df, mode_name):
    stats = df.groupby('retailer_id').agg(
        num_pixels=('cost_final', 'count'),
        cost_min=('cost_final', 'min'),
        cost_max=('cost_final', 'max'),
        ref_cost=('cost_ref', 'first'),
        component_max=('collection_component', 'max'),
        component_min=('collection_component', 'min')
    ).reset_index()
    stats['cost_range'] = stats['cost_max'] - stats['cost_min']
    stats['component_range'] = stats['component_max'] - stats['component_min']
    stats_sorted = stats.sort_values('cost_range', ascending=False)

    print(f"\n=== {mode_name.upper()} MODE STATISTICS ===")
    print(f"Unique retailers: {len(stats)}")
    print("Final cost range (max-min):")
    print(f"  Min:    {stats['cost_range'].min():.6f}")
    print(f"  Median: {stats['cost_range'].median():.6f}")
    print(f"  Max:    {stats['cost_range'].max():.6f}")
    print("Top 5 retailers with largest cost spread:")
    for _, row in stats_sorted.head(5).iterrows():
        print(f"  ID {int(row['retailer_id'])}: pixels={int(row['num_pixels'])}, "
              f"ref cost={row['ref_cost']:.4f}, range={row['cost_range']:.6f}")
    return stats

stats_walk = retailer_stats(df_walk, "WALK")
stats_car = retailer_stats(df_car, "CAR")

"""
Verify that NaN pixels in final cost correspond to pixels without population.
"""

DATA_DIR = Path("dataset_big")
OUTPUT_RASTER = END_USER_PRICE_PATH
POP_RASTER = POPULATION_RASTER_PATH
NODATA_FLOAT = -9999.0

arr, desc, nodata = _read_multiband_raster(OUTPUT_RASTER)
idx_walk = desc.index("cost_kg_walker")
idx_car = desc.index("cost_kg_driver")
cost_walk = arr[idx_walk].astype(np.float32)
cost_car = arr[idx_car].astype(np.float32)

with rasterio.open(POP_RASTER) as src_pop:
    pop = src_pop.read(1).astype(np.float32)
    pop_nodata = src_pop.nodata

if nodata is not None:
    cost_walk = np.where(cost_walk == nodata, np.nan, cost_walk)
    cost_car = np.where(cost_car == nodata, np.nan, cost_car)
if pop_nodata is not None:
    pop = np.where(pop == pop_nodata, np.nan, pop)

pop_zero_or_nan = ~np.isfinite(pop) | (pop == 0)
pop_positive = np.isfinite(pop) & (pop > 0)

walk_nan = ~np.isfinite(cost_walk)
car_nan = ~np.isfinite(cost_car)

walk_nan_in_pop_positive = (walk_nan & pop_positive).sum()
car_nan_in_pop_positive = (car_nan & pop_positive).sum()

print("=== NaN vs POPULATION CHECK ===")
print(f"Walk pixels NaN where pop > 0: {walk_nan_in_pop_positive:,}")
print(f"Car pixels NaN where pop > 0 : {car_nan_in_pop_positive:,}")
if walk_nan_in_pop_positive == 0 and car_nan_in_pop_positive == 0:
    print("✅ All NaN values correspond to pixels without population.")
else:
    print("⚠️ There are populated pixels without a cost (possible missing retailers).")

"""
Cost decomposition analysis: filling cost, reseller markup, and collection cost.
Plus accessibility analysis for mean, walk, and car costs.
"""

DATA_DIR = Path("dataset_big")
OUTPUT_RASTER = END_USER_PRICE_PATH
NODATA_FLOAT = -9999.0

# Soglie di accessibilità
WALK_THRESHOLD_MIN = 30   # minuti a piedi
CAR_THRESHOLD_MIN = 45    # minuti in auto

print("Loading output raster for cost decomposition...")
arr, desc, nodata = _read_multiband_raster(OUTPUT_RASTER)

if nodata is None:
    nodata = NODATA_FLOAT

arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
name_to_idx = {d: i for i, d in enumerate(desc) if d is not None}

# Required bands
required = [
    "cost_kg_walker",
    "res_cost_kg_out_walk_ref",
    "cost_fil_kg_out_walk_ref",
    "cost_kg_driver",
    "res_cost_kg_out_car_ref",
    "cost_fil_kg_out_car_ref",
    "mean_user_cost",          # NUOVA banda
    "best_time_walk_min",      # per accessibilità walk
    "best_time_car_min",       # per accessibilità car
    "walk_share",              # per pesi
    "car_share"                # per pesi
]

for band in required:
    if band not in name_to_idx:
        raise KeyError(f"Band '{band}' not found in raster. Run 4.6 with the latest version.")

# Extract flat arrays
cost_walk = arr[name_to_idx["cost_kg_walker"]].flatten()
ref_walk = arr[name_to_idx["res_cost_kg_out_walk_ref"]].flatten()
fill_walk = arr[name_to_idx["cost_fil_kg_out_walk_ref"]].flatten()

cost_car = arr[name_to_idx["cost_kg_driver"]].flatten()
ref_car = arr[name_to_idx["res_cost_kg_out_car_ref"]].flatten()
fill_car = arr[name_to_idx["cost_fil_kg_out_car_ref"]].flatten()

mean_cost = arr[name_to_idx["mean_user_cost"]].flatten()
walk_time = arr[name_to_idx["best_time_walk_min"]].flatten()
car_time = arr[name_to_idx["best_time_car_min"]].flatten()
walk_share = arr[name_to_idx["walk_share"]].flatten()
car_share_arr = arr[name_to_idx["car_share"]].flatten()

# Valid pixels: finite values for all components
valid_walk = (
    np.isfinite(cost_walk) &
    np.isfinite(ref_walk) &
    np.isfinite(fill_walk)
)
valid_car = (
    np.isfinite(cost_car) &
    np.isfinite(ref_car) &
    np.isfinite(fill_car)
)

# Valid mean cost pixels (almeno un modo di trasporto ha peso >0)
valid_mean = np.isfinite(mean_cost)

# Compute decomposition
reseller_markup_walk = ref_walk - fill_walk
collection_walk = cost_walk - ref_walk

reseller_markup_car = ref_car - fill_car
collection_car = cost_car - ref_car

def print_stats(name, arr, mask):
    if not np.any(mask):
        print(f"{name}: no valid pixels")
        return
    data = arr[mask]
    print(f"{name}: min={np.nanmin(data):.6f} / median={np.nanmedian(data):.6f} / "
          f"mean={np.nanmean(data):.6f} / p95={np.nanpercentile(data, 95):.6f} / max={np.nanmax(data):.6f}")

print("\n=== COST DECOMPOSITION – WALK MODE ===")
print(f"Valid walk pixels: {np.sum(valid_walk):,}")
print_stats("  Final cost (cost_kg_walker)      ", cost_walk, valid_walk)
print_stats("  Filling wholesale (cost_fil_kg_out)", fill_walk, valid_walk)
print_stats("  Reseller markup                  ", reseller_markup_walk, valid_walk)
print_stats("  Collection cost                  ", collection_walk, valid_walk)

print("\n=== COST DECOMPOSITION – CAR MODE ===")
print(f"Valid car pixels: {np.sum(valid_car):,}")
print_stats("  Final cost (cost_kg_driver)      ", cost_car, valid_car)
print_stats("  Filling wholesale (cost_fil_kg_out)", fill_car, valid_car)
print_stats("  Reseller markup                  ", reseller_markup_car, valid_car)
print_stats("  Collection cost                  ", collection_car, valid_car)

# Summary table of average shares
def share_stats(cost, fill, markup, coll, mask):
    if not np.any(mask):
        return {}
    total = cost[mask]
    return {
        "fill_pct": 100 * np.nanmean(fill[mask] / total),
        "markup_pct": 100 * np.nanmean(markup[mask] / total),
        "coll_pct": 100 * np.nanmean(coll[mask] / total),
    }

walk_shares = share_stats(cost_walk, fill_walk, reseller_markup_walk, collection_walk, valid_walk)
car_shares = share_stats(cost_car, fill_car, reseller_markup_car, collection_car, valid_car)

print("\n=== AVERAGE COST SHARE (as % of final cost) ===")
if walk_shares:
    print(f"Walk : filling={walk_shares['fill_pct']:.1f}%  "
          f"reseller_markup={walk_shares['markup_pct']:.1f}%  "
          f"collection={walk_shares['coll_pct']:.1f}%")
if car_shares:
    print(f"Car  : filling={car_shares['fill_pct']:.1f}%  "
          f"reseller_markup={car_shares['markup_pct']:.1f}%  "
          f"collection={car_shares['coll_pct']:.1f}%")


# MEAN USER COST

print("\n" + "="*60)
print("=== MEAN USER COST (weighted average) STATISTICS ===")
print("="*60)

if np.any(valid_mean):
    print_stats("  Mean user cost (USD/kg)          ", mean_cost, valid_mean)
    
    # Cost shares for mean user cost (scomposizione pesata)
    # Calcoliamo i contributi relativi dei tre componenti
    fill_contrib = np.full_like(mean_cost, np.nan)
    markup_contrib = np.full_like(mean_cost, np.nan)
    coll_contrib = np.full_like(mean_cost, np.nan)
    
    # Per ogni pixel, il costo medio pesato è la somma dei contributi pesati
    valid_for_shares = valid_mean & valid_walk & valid_car & np.isfinite(walk_share) & np.isfinite(car_share_arr)
    
    if np.any(valid_for_shares):
        # Normalizza le share (potrebbero non sommare a 1 a causa di NaN)
        share_sum = walk_share[valid_for_shares] + car_share_arr[valid_for_shares]
        walk_weight = walk_share[valid_for_shares] / share_sum
        car_weight = car_share_arr[valid_for_shares] / share_sum
        
        fill_contrib[valid_for_shares] = (
            fill_walk[valid_for_shares] * walk_weight +
            fill_car[valid_for_shares] * car_weight
        )
        markup_contrib[valid_for_shares] = (
            reseller_markup_walk[valid_for_shares] * walk_weight +
            reseller_markup_car[valid_for_shares] * car_weight
        )
        coll_contrib[valid_for_shares] = (
            collection_walk[valid_for_shares] * walk_weight +
            collection_car[valid_for_shares] * car_weight
        )
        
        valid_shares = np.isfinite(fill_contrib) & np.isfinite(markup_contrib) & np.isfinite(coll_contrib)
        
        if np.any(valid_shares):
            fill_pct_mean = 100 * np.nanmean(fill_contrib[valid_shares] / mean_cost[valid_shares])
            markup_pct_mean = 100 * np.nanmean(markup_contrib[valid_shares] / mean_cost[valid_shares])
            coll_pct_mean = 100 * np.nanmean(coll_contrib[valid_shares] / mean_cost[valid_shares])
            
            print("\n=== AVERAGE COST SHARE FOR MEAN USER COST ===")
            print(f"Mean : filling={fill_pct_mean:.1f}%  "
                  f"reseller_markup={markup_pct_mean:.1f}%  "
                  f"collection={coll_pct_mean:.1f}%")
else:
    print("No valid mean user cost pixels.")


# ACCESSIBILITY ANALYSIS

print("\n" + "="*60)
print(f"=== ACCESSIBILITY ANALYSIS (walk time <= {WALK_THRESHOLD_MIN} min) ===")
print("="*60)

walk_within_thresh = (valid_walk) & (walk_time <= WALK_THRESHOLD_MIN)
if np.any(walk_within_thresh):
    cw_within = cost_walk[walk_within_thresh]
    cw_all = cost_walk[valid_walk]
    
    print(f"\nWALKER COST - within {WALK_THRESHOLD_MIN} minutes:")
    print(f"  Pixels in group: {np.sum(walk_within_thresh):,} / {np.sum(valid_walk):,} ({100*np.sum(walk_within_thresh)/np.sum(valid_walk):.1f}%)")
    print(f"  Min:  {np.nanmin(cw_within):.6f}")
    print(f"  Median: {np.nanmedian(cw_within):.6f}")
    print(f"  Mean: {np.nanmean(cw_within):.6f}")
    print(f"  Max:  {np.nanmax(cw_within):.6f}")
    print(f"  Comparison with all walk pixels (median): {np.nanmedian(cw_within):.6f} vs {np.nanmedian(cw_all):.6f}")
    print(f"  Difference: {np.nanmedian(cw_within) - np.nanmedian(cw_all):.6f} ({(np.nanmedian(cw_within)/np.nanmedian(cw_all)-1)*100:+.1f}%)")
else:
    print(f"No walker pixels within {WALK_THRESHOLD_MIN} minutes.")

print("\n" + "="*60)
print(f"=== ACCESSIBILITY ANALYSIS (car time <= {CAR_THRESHOLD_MIN} min) ===")
print("="*60)

car_within_thresh = (valid_car) & (car_time <= CAR_THRESHOLD_MIN)
if np.any(car_within_thresh):
    cc_within = cost_car[car_within_thresh]
    cc_all = cost_car[valid_car]
    
    print(f"\nDRIVER COST - within {CAR_THRESHOLD_MIN} minutes:")
    print(f"  Pixels in group: {np.sum(car_within_thresh):,} / {np.sum(valid_car):,} ({100*np.sum(car_within_thresh)/np.sum(valid_car):.1f}%)")
    print(f"  Min:  {np.nanmin(cc_within):.6f}")
    print(f"  Median: {np.nanmedian(cc_within):.6f}")
    print(f"  Mean: {np.nanmean(cc_within):.6f}")
    print(f"  Max:  {np.nanmax(cc_within):.6f}")
    print(f"  Comparison with all car pixels (median): {np.nanmedian(cc_within):.6f} vs {np.nanmedian(cc_all):.6f}")
    print(f"  Difference: {np.nanmedian(cc_within) - np.nanmedian(cc_all):.6f} ({(np.nanmedian(cc_within)/np.nanmedian(cc_all)-1)*100:+.1f}%)")
else:
    print(f"No driver pixels within {CAR_THRESHOLD_MIN} minutes.")


# MEAN USER COST WITH ACCESSIBILITY (both types of access)

print("\n" + "="*60)
print(f"=== MEAN USER COST WITH BOTH MODES ACCESSIBLE ===")
print(f"    (walk time <= {WALK_THRESHOLD_MIN} min AND car time <= {CAR_THRESHOLD_MIN} min)")
print("="*60)

both_accessible = (valid_mean) & (walk_time <= WALK_THRESHOLD_MIN) & (car_time <= CAR_THRESHOLD_MIN)
if np.any(both_accessible):
    mean_within = mean_cost[both_accessible]
    mean_all = mean_cost[valid_mean]
    
    print(f"\nPixels with both modes accessible: {np.sum(both_accessible):,} / {np.sum(valid_mean):,} ({100*np.sum(both_accessible)/np.sum(valid_mean):.1f}%)")
    print(f"  Min:  {np.nanmin(mean_within):.6f}")
    print(f"  Median: {np.nanmedian(mean_within):.6f}")
    print(f"  Mean: {np.nanmean(mean_within):.6f}")
    print(f"  Max:  {np.nanmax(mean_within):.6f}")
    print(f"  Comparison with all mean user cost pixels (median): {np.nanmedian(mean_within):.6f} vs {np.nanmedian(mean_all):.6f}")
    print(f"  Difference: {np.nanmedian(mean_within) - np.nanmedian(mean_all):.6f} ({(np.nanmedian(mean_within)/np.nanmedian(mean_all)-1)*100:+.1f}%)")
else:
    print("No pixels with both modes accessible within thresholds.")

print("\n" + "="*60)
print("Cost decomposition and accessibility analysis completed.")
print("="*60)

# NOTE: potential duplication – kept separate to avoid logic changes (e.g., _align_to_reference, annual_lpg_demand_kg)
# NOTE: potential duplication – kept separate to avoid logic changes (e.g., multiple Path-based DATA_DIR definitions)
# NOTE: OnStove .to_raster writes to disk; when save_outputs['income_nigeria.tif'] is False, the file is deleted after caching to preserve downstream logic.
